# Level1_Final_v4 — Hint-only Panorama + Lateral Multi-View Triangulated go_to Agent

V3의 8방향 스캔은 방향 힌트 전용으로 제한하고, 실제 삼각측량 관측은 target ray에 수직인 측면 관측 위치에서만 수집하도록 수정한 버전입니다.

## Project 규칙

Level 1에서는 scene_state와 정확한 entity ID를 사용할 수 없습니다. 관찰로 추정한 coordinate만 go_to에 사용할 수 있습니다.

Code cell의 `학생 TODO` 부분을 팀 전략에 맞게 구현하세요. 기본 실행 cell은 10분 scored simulation을 실행합니다.


In [ ]:
# Colab/로컬 setup입니다. 필요하면 한 번 실행하세요.
# 주의: 원래 v2 셀의 menlo-ai/menlo-runner repo는 존재하지 않는다(404).
# menlo_runner 패키지는 공식 워크숍 repo에 들어 있으므로 여기서 설치한다.
%pip install -q "git+https://github.com/asimovinc/hansung-menlo-robotics-workshop.git" python-dotenv opencv-python Pillow matplotlib

# 로컬 clone repo에서 실행 중이면 위 install cell을 건너뛰어도 됩니다.


In [ ]:
# LLM/VLM 모델 선택입니다. Starter code 실행 전에 필요하면 수정하세요.
# 요청 반영: 텍스트 LLM은 qwen, VLM은 minma로 고정합니다.
import os

os.environ["MENLO_LLM_MODEL"] = "qwen/qwen3.6-35b-a3b"
os.environ["MENLO_VLM_MODEL"] = "minimaxai/minimax-m3"
print("LLM:", os.environ["MENLO_LLM_MODEL"])
print("VLM:", os.environ["MENLO_VLM_MODEL"])


In [ ]:
from __future__ import annotations

"""Menlo AI 로봇 분류 챌린지용 Level 1 프로젝트 파일입니다.

Level 1 규칙: scene_state와 정확한 entity ID는 사용할 수 없습니다. Coordinate go_to는
학생 시스템이 관찰로 추정하거나 기록한 좌표에만 사용할 수 있습니다.

전체 전략 — Level 0의 observe-decide-execute-verify-update 흐름을 Level 1 규칙에 맞게 변환:
  - scene_state와 정확한 entity ID는 사용하지 않는다.
  - 부트스트랩: 카메라 관찰로 A/source cube 위치를 추정해 접근하고, 첫 pick 성공
    시점의 로봇 좌표를 A waypoint로 기록한다.
  - 패드 첫 방문: 들고 있는 cube 색상으로 목적지 sign(B/C/D/E)을 결정하고,
    색상 blob은 사용하지 않고 VLM/OCR로 목적지 문자(B/C/D/E)를 직접 찾는다.
    같은 패드를 서로 다른 로봇 pose에서 여러 번 관측하여 bearing ray를 누적하고,
    삼각측량으로 접근 좌표를 추정한 뒤 coordinate go_to로 이동한다.
    place 성공 시점의 로봇 좌표로 waypoint를 확정한다.
  - 정상 루프: 저장된 A/B/C/D/E 좌표로 모든 색상을 반복 운반한다.
  - 가까운 색상 2개만 고르는 전략은 사용하지 않는다. 들고 있는 cube의 색상에
    맞는 패드로 전부 운반하는 것을 기본 정책으로 한다.

의사결정 — 하이브리드: 규칙 stage 머신이 기본이고, 규칙만으로 정답이 없는
지점(탐색 실패 후 재시도/포기, 비정상 상태 복구, 알 수 없는 stage)에서만
텍스트 LLM을 호출한다. 응답은 항상 코드가 검증하고 실패 시 규칙 fallback을 쓴다.
"""

import asyncio
import json
import math
import os
import re
from dataclasses import dataclass, field
from typing import Any, Awaitable, Callable

# v6: VLM Timeout 완화를 위해 VLM에 보내는 카메라 이미지만 OpenCV로 축소/재압축한다.
# OpenCV가 없는 환경에서도 노트북이 죽지 않도록 optional import로 처리한다.
try:
    import cv2
    import numpy as np
except Exception as _cv2_import_error:
    cv2 = None
    np = None
    print(f"[setup] OpenCV import 실패({_cv2_import_error!r}) → VLM 이미지는 원본으로 전송")

from menlo_runner.completion import CompletionConfig, CompletionTracker
from menlo_runner.llm import ask_vlm, call_llm
from menlo_runner.perception import detect_color_blobs
from menlo_runner.scene import delivered_cube_ids, held_cube_info


# ---------------------------------------------------------------------------
# 지원 코드: 공통 과제 정의와 필수 LLM 결정 형식
# ---------------------------------------------------------------------------
# 과제 문장은 고정합니다. 목표는 cube 색상 순서와 시작 위치가 달라져도
# 소스 코드 변경 없이 처리하는 하나의 agent를 만드는 것입니다.
TASK = "Find and sort cubes from the source area into their matching destination pads."
AGENT_VERSION = "Level1_Final_v7_preplace_align_approach_reset_xfence_nosave_qwen_minma"

# 고정 표지판 정보는 사용할 수 있습니다. 단, 이를 정확한 coordinate나 entity ID로
# 바꾸지 말고 관찰을 해석하는 데만 사용하세요.
DESTINATION_SIGN_RULES = {
    "red": "B",
    "green": "C",
    "blue": "D",
    "yellow": "E",
}
SIGNAGE_NOTE = (
    "A는 conveyor/cube source area이며 destination이 아닙니다. "
    "Destination sign은 B red, C green, D blue, E yellow입니다."
)

# 모든 색상을 운반한다. 가까운 2색 선택 전략은 사용하지 않는다.

# pick이 이 횟수만큼 연속 실패하면 잘못된 위치로 판단해 bootstrap을 재시작한다.
MAX_PICK_FAILURES = 3

# 이 횟수만큼 회전 탐색 후 새 지점으로 전진한다 (bootstrap 단방향 회전 sweep).
SEARCH_ROTATION_STEPS = 8

# 하이브리드 LLM / VLM 동작 파라미터.
LLM_TIMEOUT_S = 30            # LLM 상담 1회 최대 대기 — 초과/실패 시 규칙 fallback (연속성 보호)
VLM_FRAME_TIMEOUT_S = 30       # v6: sign frame 1장 VLM 판독 최대 대기. 12초는 너무 짧아 Timeout이 반복되어 30초로 완화
MAX_PAD_SEARCH_FAILURES = 8   # 문자별 탐색 누적 실패 한계 — 일반 운반은 계속 재탐색, 한계 초과 시에만 skip
MAX_VLM_MISMATCHES = 4        # Final_v5: place 전 mismatch는 drop하지 않고 재탐색을 더 허용
MAX_PLACE_FAILURES = 3        # place 연속 실패 한계 — 초과 시 좌표 기반 재접근

# v6 VLM image payload control: Timeout 완화를 위해 VLM 요청용 이미지만 축소/압축한다.
# 표지판 문자 A/B/C/D/E를 읽어야 하므로 800px부터 테스트한다. Timeout이 계속 심하면 640으로 낮춘다.
VLM_IMAGE_MAX_WIDTH = 800
VLM_IMAGE_JPEG_QUALITY = 70
VLM_IMAGE_LOG_BYTES = True

ALLOWED_NEXT_ACTIONS = {
    "search_cube",
    "navigate_to_cube",
    "pick_cube",
    "search_pad",
    "navigate_to_pad",
    "place_cube",
    "recover",
    "skip_target",
    "stop",
}

# stage 전이 다이어그램:
#
#  need_cube ──(bootstrap)──► at_source ──► pick_cube ──► holding_cube
#                                                              │
#             ┌───────────── 패드 좌표 없음 ──────────────────┤
#             ▼                                                │ 패드 좌표 있음
#         search_pad                                           └─ navigate_to_pad
#        /          \
#   (found)        (failed)
#      │               ▼
#      │        search_pad_failed ──(LLM 상담: 재탐색/복구/포기)──► search_pad 재시도
#      │
#      ▼
#  delivering_just_found
#      │
#      ▼
#  navigate_to_pad ──► at_pad ──► place_cube ──► returning_to_source
#                                                   │
#                                                   ▼
#                                      navigate_to_cube(A) ──► at_source
#
#  [첫 방문 패드] place 직전 VLM 표지판 확인.
#  성공 시점의 로봇 좌표로 A/B/C/D/E waypoint를 확정한다.
#  [비정상] holding_cube/delivering_just_found/at_pad + held_color=None
#           ──(LLM 상담: A 복귀/recover)──► at_source 경로로 복귀


@dataclass
class AgentDecision:
    """LLM 또는 규칙이 반환하고 코드가 검증한 상위 단계 결정입니다."""

    next_action: str
    target_color: str | None = None
    reason: str = ""
    recovery_strategy: str | None = None
    target_location: str | None = None


@dataclass
class AgentMemory:
    """observe-decide-act cycle 사이에 agent가 유지하는 상태입니다."""

    delivered_count: int = 0
    held_color: str | None = None
    active_color: str | None = None
    stage: str = "need_cube"
    search_turns: int = 0
    failed_attempts: dict[str, int] = field(default_factory=dict)
    completed_colors: list[str] = field(default_factory=list)
    skipped_colors: list[str] = field(default_factory=list)
    logs: list[dict[str, Any]] = field(default_factory=list)
    # 학습된 좌표: A는 첫 pick 성공 시, 패드는 첫 place 성공 시 로봇 좌표로 확정된다.
    known_locations: dict[str, tuple[float, float]] = field(default_factory=dict)
    bootstrapped_to_a: bool = False
    active_pad_name: str | None = None
    # 첫 배달까지 성공한(=좌표가 로봇 pose로 확정되고 VLM 확인을 통과한) 패드 목록.
    # 이 목록에 있는 패드는 재방문 시 VLM 확인을 생략한다 — "이후엔 빠르게".
    confirmed_pads: list[str] = field(default_factory=list)
    # Level_1_v2_0704: 패드 sign 관측 누적 저장소.
    # 각 관측은 scene_state가 아닌 robot_status pose + VLM sign bearing으로 만든 ray만 저장한다.
    pad_observations: dict[str, list[dict[str, Any]]] = field(default_factory=dict)
    # Level_1_v4_0704: target sign이 실제로 보였던 body yaw를 기억한다.
    # 좌표가 아니라 "다음에 이 패드를 찾을 때 먼저 볼 방향" 힌트이므로 Level 1 규칙을 유지한다.
    pad_direction_hints: dict[str, float] = field(default_factory=dict)
    llm_consult_count: int = 0


@dataclass
class Observation:
    """LLM과 실행 코드에 전달할 간결한 관찰입니다."""

    robot_status: Any
    detections: list[Any]
    note: str = ""
    vlm_summary: str = ""


@dataclass(frozen=True)
class ScannedDetection:
    """해당 camera frame을 얻을 때 사용한 head pose가 함께 기록된 color detection입니다."""

    color: str
    angle_deg: float
    blob_area: int
    centroid: tuple[int, int]
    bbox: tuple[int, int, int, int]
    head_yaw: float
    head_pitch: float

    @property
    def full_bearing_deg(self) -> float:
        """대략적인 body-relative bearing입니다. Image angle에 head yaw를 더합니다."""
        return self.angle_deg + math.degrees(self.head_yaw)


def parse_agent_decision(text: str) -> AgentDecision | None:
    """필수 structured LLM JSON output을 parse하고 validate합니다."""
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = stripped.strip("`")
        if stripped.lower().startswith("json"):
            stripped = stripped[4:].strip()
    if not stripped.startswith("{"):
        start = stripped.find("{")
        end = stripped.rfind("}")
        if start >= 0 and end > start:
            stripped = stripped[start : end + 1]
    try:
        data = json.loads(stripped)
    except json.JSONDecodeError:
        return None

    next_action = data.get("next_action")
    if next_action not in ALLOWED_NEXT_ACTIONS:
        return None

    target_color = data.get("target_color")
    if target_color is not None and not isinstance(target_color, str):
        return None

    return AgentDecision(
        next_action=next_action,
        target_color=target_color,
        reason=str(data.get("reason", "")),
        recovery_strategy=data.get("recovery_strategy"),
        target_location=data.get("target_location"),
    )


def build_decision_context(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Robot state를 LLM에 전달하기 좋은 간결한 text context로 변환합니다."""
    visible = [
        {
            "color": detection.color,
            "angle_deg": detection.angle_deg,
            "full_bearing_deg": round(getattr(detection, "full_bearing_deg", detection.angle_deg), 1),
            "blob_area": detection.blob_area,
            "bbox": detection.bbox,
        }
        for detection in observation.detections
    ]
    return {
        "task": task,
        "visible_targets": visible,
        "held_color": memory.held_color,
        "stage": memory.stage,
        "delivered_count": memory.delivered_count,
        "completed_colors": memory.completed_colors,
        "skipped_colors": memory.skipped_colors,
        "failed_attempts": memory.failed_attempts,
        "last_result": last_result,
        "note": observation.note,
        "signage_note": SIGNAGE_NOTE,
        "known_locations": {name: list(xy) for name, xy in memory.known_locations.items()},
        "confirmed_pads": memory.confirmed_pads,
        "pad_observation_counts": {name: len(obs) for name, obs in memory.pad_observations.items()},
        "bootstrapped_to_a": memory.bootstrapped_to_a,
    }


# ---------------------------------------------------------------------------
# 지원 코드: project 규칙에 맞는 SDK wrapper
# ---------------------------------------------------------------------------
# 이 래퍼들은 프로젝트 규칙에 맞는 input만 노출합니다. 여기에 scene_state,
# ground-truth coordinate, 정확한 cube ID, global asset map을 추가하지 마세요.


async def get_robot_status(ctx: Any) -> Any:
    """Robot pose, motion status, neck state를 읽습니다.

    LiveKit/RPC가 순간적으로 끊기면 전체 셀이 CancelledError로 중단될 수 있어서,
    status 읽기 실패는 None으로 반환하고 상위 로직에서 recover/fallback 하도록 한다.
    """
    try:
        return await ctx.state("robot_status")
    except asyncio.CancelledError as exc:
        print(f"[rpc] robot_status 읽기 취소({exc!r}) → None 처리")
        return None
    except Exception as exc:
        print(f"[rpc] robot_status 읽기 실패({exc!r}) → None 처리")
        return None

async def get_camera_frame(ctx: Any) -> bytes:
    """POV camera frame을 가져옵니다."""
    return await ctx.get_vision("pov")


def compress_vlm_image_cv2(
    image_bytes: bytes,
    *,
    max_width: int = VLM_IMAGE_MAX_WIDTH,
    jpeg_quality: int = VLM_IMAGE_JPEG_QUALITY,
) -> bytes:
    """VLM 요청용 image bytes를 OpenCV로 downsize + JPEG 재압축한다.

    목적: ask_vlm에 보내는 payload를 줄여 전송/처리 지연과 Timeout 가능성을 낮춘다.
    주의: 로봇 시야 자체나 color blob detector 입력은 바꾸지 않고, VLM 호출에만 적용한다.
    """
    if cv2 is None or np is None:
        return image_bytes
    try:
        arr = np.frombuffer(image_bytes, np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if img is None:
            return image_bytes
        h, w = img.shape[:2]
        if w > max_width:
            scale = max_width / float(w)
            new_w = int(max_width)
            new_h = max(1, int(h * scale))
            img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        ok, encoded = cv2.imencode(
            ".jpg",
            img,
            [int(cv2.IMWRITE_JPEG_QUALITY), int(jpeg_quality)],
        )
        if not ok:
            return image_bytes
        return encoded.tobytes()
    except Exception as exc:
        print(f"[vlm-image] OpenCV downsize 실패({exc!r}) → 원본 전송")
        return image_bytes


def prepare_vlm_image(image_bytes: bytes) -> bytes:
    """VLM 호출 직전 공통 image preprocessing. 압축 전/후 용량을 로그로 남긴다."""
    before_kb = len(image_bytes) / 1024.0
    compressed = compress_vlm_image_cv2(image_bytes)
    after_kb = len(compressed) / 1024.0
    if VLM_IMAGE_LOG_BYTES:
        ratio = (after_kb / before_kb) if before_kb > 0 else 1.0
        print(
            f"[vlm-image] before={before_kb:.1f}KB → after={after_kb:.1f}KB "
            f"(max_width={VLM_IMAGE_MAX_WIDTH}, quality={VLM_IMAGE_JPEG_QUALITY}, ratio={ratio:.2f})"
        )
    return compressed



async def get_delivered_count(ctx: Any) -> int:
    """Count delivered cubes using the shared workshop progress helper."""
    try:
        return len(await delivered_cube_ids(ctx))
    except asyncio.CancelledError as exc:
        print(f"[rpc] delivered_count 읽기 취소({exc!r}) → 0 유지")
        return 0
    except Exception as exc:
        print(f"[rpc] delivered_count 읽기 실패({exc!r}) → 0 유지")
        return 0


async def get_held_cube_info(ctx: Any) -> dict[str, str] | None:
    """Return the currently held cube color, if the robot is holding one.

    Level 1에서는 정확한 entity ID 기반 이동이 금지되어 있으므로, 이후 판단에는
    entity_id를 노출하지 않고 color만 사용한다.
    """
    try:
        held = await held_cube_info(ctx)
    except asyncio.CancelledError as exc:
        print(f"[rpc] held_cube_info 읽기 취소({exc!r}) → None 처리")
        return None
    except Exception as exc:
        print(f"[rpc] held_cube_info 읽기 실패({exc!r}) → None 처리")
        return None
    return {"color": held[1]} if held else None

def build_signage_vlm_prompt(held_color: str | None = None) -> str:
    """고정 warehouse signage를 읽기 위한 strategy-neutral prompt를 만듭니다."""
    target = ""
    if held_color in DESTINATION_SIGN_RULES:
        target = f" Robot이 {held_color} cube를 들고 있으므로 target destination sign은 {DESTINATION_SIGN_RULES[held_color]}입니다."
    return (
        "이 robot camera frame에 보이는 warehouse sign을 읽으세요. "
        f"{SIGNAGE_NOTE} "
        "보이는 sign letter, color, 대략적인 left/center/right 위치, confidence를 JSON으로 반환하세요."
        + target
    )


async def ask_vlm_about_frame(ctx: Any, prompt: str, *, api_key: str) -> str:
    """Project에서 허용되는 VLM helper로 현재 POV frame에 대해 질문합니다."""
    jpeg = await get_camera_frame(ctx)
    jpeg = prepare_vlm_image(jpeg)
    return ask_vlm(jpeg, prompt, api_key=api_key)


async def perceive(ctx: Any) -> list[Any]:
    """현재 camera frame에서 Workshop 2 color-blob detector를 실행합니다."""
    jpeg = await get_camera_frame(ctx)
    return detect_color_blobs(jpeg)


async def set_head(ctx: Any, *, yaw: float | None = None, pitch: float | None = None) -> Any:
    """Walking direction을 바꾸지 않고 camera 방향을 조정합니다.

    LiveKit/RPC가 순간적으로 끊기면 asyncio.CancelledError가 바깥까지 전파되어
    전체 run_agent 셀이 중단될 수 있다. Level 1 실험에서는 한 번의 head 제어 실패를
    search 실패로 처리하고 다음 cycle에서 복구하는 편이 낫기 때문에 안전하게 감싼다.
    """
    args: dict[str, float] = {}
    if yaw is not None:
        args["yaw"] = yaw
    if pitch is not None:
        args["pitch"] = pitch
    try:
        return await ctx.invoke("set_head", args, timeout_s=10)
    except asyncio.CancelledError as exc:
        print(f"[rpc] set_head 취소({exc!r}) → 이번 head command 실패로 처리")
        return {"status": "failed", "error": "set_head_cancelled", "args": args}
    except Exception as exc:
        print(f"[rpc] set_head 실패({exc!r}) → 이번 head command 실패로 처리")
        return {"status": "failed", "error": str(exc), "args": args}


async def move_velocity(
    ctx: Any,
    *,
    vx: float = 0.0,
    vy: float = 0.0,
    wz: float = 0.0,
    duration_s: float = 1.0,
) -> Any:
    """짧은 body-frame velocity command.

    v7에서는 관측 baseline 확보용 이동에만 사용한다. 추정 좌표가 계산된 뒤의 운반은
    set_velocity가 아니라 Menlo go_to(pose)를 사용한다. x-fence를 넘을 예측 이동은 차단한다.
    """
    status = await get_robot_status(ctx)
    blocked, fence_info = _x_fence_velocity_would_cross(status, vx=vx, vy=vy, duration_s=duration_s)
    if blocked:
        print(f"[x-fence] velocity 차단: {fence_info}")
        return {
            "status": "failed",
            "error": "x_fence_blocked",
            "args": {"vx": vx, "vy": vy, "wz": wz, "duration_s": duration_s},
            "fence_info": fence_info,
        }

    args = {"vx": vx, "vy": vy, "wz": wz, "duration_s": duration_s}
    try:
        return await ctx.invoke("set_velocity", args, timeout_s=30)
    except asyncio.CancelledError as exc:
        print(f"[rpc] set_velocity 취소({exc!r}) → 이번 motion 실패로 처리")
        return {"status": "failed", "error": "set_velocity_cancelled", "args": args}
    except Exception as exc:
        print(f"[rpc] set_velocity 실패({exc!r}) → 이번 motion 실패로 처리")
        return {"status": "failed", "error": str(exc), "args": args}


async def cancel_action(ctx: Any) -> Any:
    """현재 실행 중인 runtime action을 취소합니다."""
    return await ctx.invoke("cancel", {})


async def pick_nearest_cube(ctx: Any) -> Any:
    """Code가 robot을 시각적으로 충분히 위치시킨 뒤 nearest cube를 집습니다."""
    try:
        return await ctx.invoke(
            "pick_entity",
            {"target": {"kind": "entity", "entity_id": "cube"}},
            timeout_s=300,
        )
    except asyncio.CancelledError as exc:
        print(f"[rpc] pick_entity 취소({exc!r}) → pick 실패로 처리")
        return {"status": "failed", "error": "pick_cancelled"}
    except Exception as exc:
        print(f"[rpc] pick_entity 실패({exc!r}) → pick 실패로 처리")
        return {"status": "failed", "error": str(exc)}


async def place_nearest_zone(ctx: Any) -> Any:
    """Matching pad에 도달한 뒤 nearest zone에 place합니다."""
    try:
        return await ctx.invoke("place_entity", {}, timeout_s=300)
    except asyncio.CancelledError as exc:
        print(f"[rpc] place_entity 취소({exc!r}) → place 실패로 처리")
        return {"status": "failed", "error": "place_cancelled"}
    except Exception as exc:
        print(f"[rpc] place_entity 실패({exc!r}) → place 실패로 처리")
        return {"status": "failed", "error": str(exc)}


def result_summary(result: Any) -> dict[str, Any]:
    """SDK result를 log하기 쉬운 작은 dictionary로 변환합니다."""
    if isinstance(result, dict):
        return {
            "status": result.get("status"),
            "error": result.get("error"),
            "args": result.get("args"),
        }
    error = getattr(result, "error", None)
    status = getattr(result, "status", None)
    return {
        "status": str(status) if status is not None else None,
        "error": getattr(error, "message", None) if error else None,
    }


async def scan_head(
    ctx: Any,
    *,
    yaws: tuple[float, ...] = (-0.8, 0.0, 0.8),
    pitch: float = 0.15,
) -> list[Any]:
    """간단한 scan helper입니다. head yaw 3방향으로 정지 스캔합니다."""
    all_detections: list[Any] = []
    for yaw in yaws:
        await set_head(ctx, yaw=yaw, pitch=pitch)
        await asyncio.sleep(0.4)
        for detection in await perceive(ctx):
            all_detections.append(
                ScannedDetection(
                    color=detection.color,
                    angle_deg=detection.angle_deg,
                    blob_area=detection.blob_area,
                    centroid=detection.centroid,
                    bbox=detection.bbox,
                    head_yaw=yaw,
                    head_pitch=pitch,
                )
            )
    return all_detections


async def visual_search_step(ctx: Any, memory: AgentMemory) -> list[Any]:
    """bootstrap 탐색 한 step: 항상 같은 방향(+wz)으로 회전, 8회마다 전진.

    교번 방향을 쓰면 제자리 진동만 반복되어 A를 못 찾는다.
    """
    memory.search_turns += 1
    await set_head(ctx, yaw=0.0, pitch=0.15)
    if memory.search_turns % SEARCH_ROTATION_STEPS == 0:
        # 한 바퀴 완료 → 새 지점으로 전진해 시야 변경
        await move_velocity(ctx, vx=0.3, wz=0.0, duration_s=2.0)
    else:
        # wz만 주면 제자리 회전 불가, 작은 vx 필수
        await move_velocity(ctx, vx=0.08, wz=0.35, duration_s=1.0)
    return await scan_head(ctx)


async def observe_after_search(ctx: Any, memory: AgentMemory) -> Observation:
    detections = await visual_search_step(ctx, memory)
    robot_status = await get_robot_status(ctx)
    note = "Search step done." if detections else "Search step done; no blob visible."
    return Observation(robot_status=robot_status, detections=detections, note=note)


# ---------------------------------------------------------------------------
# 지원 코드: vision 기반 좌표 추정
# ---------------------------------------------------------------------------

MIN_TARGET_BLOB_AREA = 220
MAX_ESTIMATED_DISTANCE_M = 8.0
DISTANCE_SCALE_CUBE = float(os.getenv("MENLO_DISTANCE_SCALE_CUBE", "55.0"))
DISTANCE_SCALE_PAD = float(os.getenv("MENLO_DISTANCE_SCALE_PAD", "80.0"))


def robot_yaw_deg_from_status(robot_status: Any) -> float | None:
    pose = getattr(getattr(robot_status, "robot", None), "pose", None)
    yaw = getattr(pose, "yaw_deg", None)
    if yaw is None:
        yaw = getattr(pose, "yaw", None)
        if yaw is not None and abs(float(yaw)) <= math.tau:
            yaw = math.degrees(float(yaw))
    return float(yaw) if yaw is not None else None


def robot_yaw_rad_from_status(robot_status: Any) -> float | None:
    """robot_status에서 로봇 yaw를 radian으로 읽는다.

    기존 helper는 degree를 반환하므로 preplace refine에서 radian bearing 계산에
    바로 쓸 수 있도록 얇은 wrapper를 둔다. scene_state/ground-truth는 사용하지 않는다.
    """
    yaw_deg = robot_yaw_deg_from_status(robot_status)
    if yaw_deg is None:
        return None
    return math.radians(float(yaw_deg))


def robot_xy_from_status(robot_status: Any) -> tuple[float, float] | None:
    """robot_status에서 로봇 world x/y를 읽는다. scene_state 미사용."""
    pose = getattr(getattr(robot_status, "robot", None), "pose", None)
    position = getattr(pose, "position", None)
    if position is None or len(position) < 2:
        return None
    return (float(position[0]), float(position[1]))


# ---------------------------------------------------------------------------
# v7 fast-goto safety: 빨간선 x=2.68 기준 금지구역 방지
# ---------------------------------------------------------------------------
# 사용자 overhead 기준에서 노란 장애물 구역은 x가 커지는 방향이다.
# 따라서 안전 영역은 x <= 2.68 - margin 이고, 좌표 go_to / velocity 예측 모두 이 선을 넘지 않는다.
X_FENCE_LINE = float(os.getenv("MENLO_X_FENCE_LINE", "2.68"))
X_FENCE_MARGIN_M = float(os.getenv("MENLO_X_FENCE_MARGIN_M", "0.08"))
X_FENCE_FORBIDDEN_WHEN = os.getenv("MENLO_X_FENCE_FORBIDDEN_WHEN", "gt").strip().lower()


def _x_fence_safe_boundary() -> float:
    return X_FENCE_LINE - X_FENCE_MARGIN_M if X_FENCE_FORBIDDEN_WHEN == "gt" else X_FENCE_LINE + X_FENCE_MARGIN_M


def _x_fence_is_forbidden_x(x: float) -> bool:
    boundary = _x_fence_safe_boundary()
    return float(x) > boundary if X_FENCE_FORBIDDEN_WHEN == "gt" else float(x) < boundary


def _x_fence_clamp_xy(x: float, y: float) -> tuple[float, float]:
    x = float(x)
    y = float(y)
    boundary = _x_fence_safe_boundary()
    if X_FENCE_FORBIDDEN_WHEN == "gt" and x > boundary:
        print(f"[x-fence] target x={x:.3f}가 금지구역(x > {boundary:.3f}) → x={boundary:.3f}로 clamp")
        x = boundary
    elif X_FENCE_FORBIDDEN_WHEN != "gt" and x < boundary:
        print(f"[x-fence] target x={x:.3f}가 금지구역(x < {boundary:.3f}) → x={boundary:.3f}로 clamp")
        x = boundary
    return (x, y)


def _x_fence_segment_crosses(start_xy: tuple[float, float] | None, end_xy: tuple[float, float] | None) -> bool:
    if start_xy is None or end_xy is None:
        return False
    sx, sy = float(start_xy[0]), float(start_xy[1])
    ex, ey = float(end_xy[0]), float(end_xy[1])
    for i in range(1, 13):
        t = i / 12.0
        if _x_fence_is_forbidden_x(sx + (ex - sx) * t):
            return True
    return False


def _x_fence_velocity_would_cross(robot_status: Any, *, vx: float, vy: float, duration_s: float) -> tuple[bool, dict[str, Any]]:
    xy = robot_xy_from_status(robot_status)
    yaw_deg = robot_yaw_deg_from_status(robot_status)
    info: dict[str, Any] = {"safe_boundary": _x_fence_safe_boundary(), "forbidden_when": X_FENCE_FORBIDDEN_WHEN}
    if xy is None or yaw_deg is None:
        info["reason"] = "pose_unknown"
        return False, info
    yaw = math.radians(float(yaw_deg))
    dx = (float(vx) * math.cos(yaw) - float(vy) * math.sin(yaw)) * float(duration_s)
    dy = (float(vx) * math.sin(yaw) + float(vy) * math.cos(yaw)) * float(duration_s)
    predicted_xy = (float(xy[0]) + dx, float(xy[1]) + dy)
    info.update({"xy_now": xy, "predicted_xy": predicted_xy, "vx": vx, "vy": vy, "duration_s": duration_s, "yaw_deg": yaw_deg})
    return _x_fence_segment_crosses(xy, predicted_xy), info


def select_visual_detection(
    observation: Observation,
    target_color: str | None,
    *,
    min_area: int = MIN_TARGET_BLOB_AREA,
) -> ScannedDetection | None:
    candidates = [d for d in observation.detections if getattr(d, "blob_area", 0) >= min_area]
    if target_color is not None:
        candidates = [d for d in candidates if d.color == target_color]
    if not candidates:
        return None
    return max(candidates, key=lambda d: d.blob_area)


def estimate_distance_from_blob(detection: ScannedDetection, *, kind: str = "cube") -> float | None:
    area = max(float(detection.blob_area), 1.0)
    scale = DISTANCE_SCALE_PAD if kind == "pad" else DISTANCE_SCALE_CUBE
    distance_m = scale / math.sqrt(area)
    if distance_m <= 0.0 or distance_m > MAX_ESTIMATED_DISTANCE_M:
        return None
    return distance_m


def estimate_target_xy_from_observation(
    observation: Observation,
    target_color: str | None,
    *,
    kind: str = "cube",
) -> tuple[float, float] | None:
    """Camera observation으로 target world coordinate를 추정합니다.

    로봇 pose + full bearing + blob 면적 기반 거리로 world x/y를 만든다.
    부정확할 수 있으므로 '처음 접근'에만 쓰고, 성공한 위치는 로봇 좌표로 덮어쓴다.
    """
    robot_xy = robot_xy_from_status(observation.robot_status)
    robot_yaw_deg = robot_yaw_deg_from_status(observation.robot_status)
    detection = select_visual_detection(observation, target_color)
    if robot_xy is None or robot_yaw_deg is None or detection is None:
        return None
    distance_m = estimate_distance_from_blob(detection, kind=kind)
    if distance_m is None:
        return None
    world_bearing_rad = math.radians(robot_yaw_deg + detection.full_bearing_deg)
    target_x = robot_xy[0] + distance_m * math.cos(world_bearing_rad)
    target_y = robot_xy[1] + distance_m * math.sin(world_bearing_rad)
    print(
        "[vision] target",
        {
            "color": detection.color,
            "area": detection.blob_area,
            "bearing_deg": round(detection.full_bearing_deg, 1),
            "distance_m": round(distance_m, 2),
            "xy": (round(target_x, 2), round(target_y, 2)),
        },
    )
    return (target_x, target_y)


# ---------------------------------------------------------------------------
# 지원 코드: waypoint 관리
# ---------------------------------------------------------------------------

MANUAL_WAYPOINTS: dict[str, tuple[float, float] | None] = {
    "A": None, "B": None, "C": None, "D": None, "E": None,
}


def _parse_waypoint_env(name: str) -> tuple[float, float] | None:
    raw = os.getenv(f"MENLO_WAYPOINT_{name}")
    if not raw:
        return None
    try:
        x_text, y_text = raw.split(",", 1)
        return (float(x_text.strip()), float(y_text.strip()))
    except ValueError:
        print(f"[waypoint] MENLO_WAYPOINT_{name} 값은 'x,y' 형식이어야 합니다: {raw!r}")
        return None


def load_configured_waypoints() -> dict[str, tuple[float, float]]:
    configured: dict[str, tuple[float, float]] = {}
    for name, xy in MANUAL_WAYPOINTS.items():
        env_xy = _parse_waypoint_env(name)
        chosen = env_xy if env_xy is not None else xy
        if chosen is not None:
            configured[name] = (float(chosen[0]), float(chosen[1]))
    return configured


def set_manual_waypoint(name: str, x: float, y: float) -> tuple[float, float]:
    normalized = name.removeprefix("pad_").upper()
    if normalized not in MANUAL_WAYPOINTS:
        raise ValueError(f"지원하지 않는 waypoint: {name!r}. A/B/C/D/E 중 하나를 사용하세요.")
    MANUAL_WAYPOINTS[normalized] = (float(x), float(y))
    print(f"[waypoint] {normalized} = {MANUAL_WAYPOINTS[normalized]}")
    return MANUAL_WAYPOINTS[normalized]  # type: ignore[return-value]


async def remember_current_location(ctx: Any, name: str) -> tuple[float, float]:
    status = await get_robot_status(ctx)
    xy = robot_xy_from_status(status)
    if xy is None:
        raise RuntimeError("robot_status에서 현재 로봇 좌표를 읽지 못했습니다.")
    return set_manual_waypoint(name, *xy)


async def go_to_xy(ctx: Any, x: float, y: float) -> Any:
    """관측/추정 좌표에 대해 Menlo 내장 go_to(pose)를 바로 호출한다.

    여기서 go_to는 entity ID가 아니라 학생 시스템이 VLM/관측으로 만든 world pose 좌표만 사용한다.
    추정 좌표가 계산되면 운반 단계는 이 함수로 빠르게 이동하고, 폐루프식 조금씩 전진은 사용하지 않는다.
    """
    requested_xy = (float(x), float(y))
    safe_x, safe_y = _x_fence_clamp_xy(float(x), float(y))
    if (safe_x, safe_y) != requested_xy:
        print(f"[x-fence] go_to target 보정: requested={requested_xy} → safe={(safe_x, safe_y)}")
    print(f"[fast-goto] Menlo go_to(pose) target={(round(safe_x, 3), round(safe_y, 3))}")
    try:
        return await ctx.invoke(
            "go_to",
            {
                "target": {
                    "kind": "pose",
                    "pose": {"frame_id": "world", "position": [safe_x, safe_y, 0]},
                }
            },
            timeout_s=300,
        )
    except asyncio.CancelledError as exc:
        print(f"[rpc] go_to 취소({exc!r}) → 이번 go_to 실패로 처리")
        return {"status": "failed", "error": "go_to_cancelled", "target_xy": [safe_x, safe_y]}
    except Exception as exc:
        print(f"[rpc] go_to 실패({exc!r}) → 이번 go_to 실패로 처리")
        return {"status": "failed", "error": str(exc), "target_xy": [safe_x, safe_y]}


async def go_to_known_location(ctx: Any, memory: AgentMemory, name: str) -> dict[str, Any]:
    normalized = name.removeprefix("pad_").upper()
    xy = memory.known_locations.get(normalized)
    if xy is None:
        return {
            "status": "failed",
            "target_location": normalized,
            "reason": f"known waypoint {normalized} 좌표가 없습니다",
        }
    result = await go_to_xy(ctx, *xy)
    return {
        "status": "arrived",
        "target_location": normalized,
        "target_xy": xy,
        "result": result_summary(result),
    }


async def move_to_waypoint(ctx: Any, name: str) -> dict[str, Any]:
    memory = AgentMemory()
    memory.known_locations.update(load_configured_waypoints())
    return await go_to_known_location(ctx, memory, name)


# ---------------------------------------------------------------------------
# 학생 TODO: 큐브 색상 → 목적지 문자 매핑 정책
# ---------------------------------------------------------------------------

def delivery_colors() -> list[str]:
    """가까운 색만 고르지 않고 모든 cube 색상을 목적지 패드로 운반한다."""
    return list(DESTINATION_SIGN_RULES.keys())


# ---------------------------------------------------------------------------
# 학생 TODO: 패드 탐색 — 문자 기반 + 정면 폐루프 접근
# ---------------------------------------------------------------------------

# 제자리 스윕에서 360°를 몇 방향으로 나눠 정지 스캔할지.
# 패드 탐색은 색상 blob을 전혀 사용하지 않고, VLM이 읽은 sign letter만 사용한다.
SEARCH_SWEEP_DIRECTIONS = 4
SEARCH_SWEEP_ROTATE_WZ = 0.5
SEARCH_SWEEP_ROTATE_DURATION_S = 3.0

# VLM 문자 탐색용 head yaw. 각 body 방향마다 좌/중/우를 확인한다.
TEXT_SEARCH_HEAD_YAWS = (-0.8, 0.0, 0.8)
TEXT_SEARCH_PITCH = 0.0
TEXT_SEARCH_FORWARD_STEPS = 4

# 한 search_pad action 안에서 VLM을 너무 많이 호출하면 Timeout이 누적된다.
# 삼각측량은 정렬/기준선 관측이 추가로 필요하므로 너무 낮게 두지는 않는다.
MAX_VLM_CALLS_PER_SEARCH = 16

# 이미지상 target 위치를 카메라 중심선 기준 각도로 환산할 때 쓰는 보수적 값.
# 이 프로젝트 좌표계에서는 기존 관측 로그 기준으로 left를 +, right를 -로 둔다.
# VLM이 target_offset_deg를 숫자로 주면 그 값을 우선 사용하고, 없으면 이 값을 사용한다.
TEXT_SIGN_POSITION_OFFSET_DEG = {
    "left": 12.0,
    "center": 0.0,
    "right": -12.0,
    "unknown": 0.0,
}

# 삼각측량 관련 파라미터.
TRIANGULATION_ALIGN_WZ = 0.45
TRIANGULATION_ALIGN_MAX_TURN_S = 1.8
# v7: yaw feedback을 적용하면서 camera offset 부호를 그대로 사용한다.
# left는 +, right는 -이며, 실제 회전량은 robot_status yaw feedback으로 보정한다.
TRIANGULATION_TURN_SIGN = 1.0
TRIANGULATION_BASELINE_VY = 0.22
TRIANGULATION_BASELINE_DURATION_S = 1.6
TRIANGULATION_MIN_BASELINE_M = 0.18
TRIANGULATION_MIN_CROSS_ANGLE_DEG = 7.0
TRIANGULATION_MAX_CROSS_ANGLE_DEG = 160.0
TRIANGULATION_MAX_SIGN_DISTANCE_M = 6.5
TRIANGULATION_APPROACH_STANDOFF_M = 0.45

# 삼각측량이 실패했을 때만 쓰는 단일 ray fallback 거리.
# 좌표 확정용이 아니라 “한 번 접근해서 다시 관찰하기 위한” 보수적 임시 좌표다.
TEXT_SIGN_FALLBACK_DISTANCE_M = {
    "near": 1.0,
    "medium": 1.75,
    "far": 2.6,
    "unknown": 1.7,
}

# v10: 문자 기반 front-loop에서는 A와의 거리 하드 필터를 사용하지 않는다.
# 과거 색상/rough 좌표 오탐 방지용이었지만, C/D/E/B sign을 직접 읽는 현재 로직에서는
# center/near를 place 시도로 넘기는 것을 막는 부작용이 더 컸다.
MIN_PAD_DISTANCE_FROM_A_M = 0.0

# place 직전 VLM이 timeout/cancelled 되어 확인 불가인 경우, 좌표를 즉시 버리지 않고
# 몇 번 재확인 기회를 준다. mismatch와 unavailable을 분리하기 위한 카운터다.
MAX_VLM_UNAVAILABLE_RETRIES = 2


def _normalize_vlm_position(value: Any) -> str:
    text = str(value or "").strip().lower()
    if text in {"left", "l", "좌", "왼쪽"}:
        return "left"
    if text in {"right", "r", "우", "오른쪽"}:
        return "right"
    if text in {"center", "centre", "middle", "c", "중앙", "가운데"}:
        return "center"
    return "unknown"


def _normalize_vlm_distance(value: Any) -> str:
    text = str(value or "").strip().lower()
    if text in {"near", "close", "short", "가까움", "가까운"}:
        return "near"
    if text in {"far", "distant", "long", "멀리", "먼"}:
        return "far"
    if text in {"medium", "mid", "middle", "보통", "중간"}:
        return "medium"
    return "unknown"


def _coerce_float(value: Any) -> float | None:
    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        match = re.search(r"[-+]?\d+(?:\.\d+)?", str(value))
        if match:
            try:
                return float(match.group(0))
            except ValueError:
                return None
    return None


def _coerce_letters(value: Any) -> list[str]:
    """VLM JSON의 여러 표현을 대문자 sign letter list로 통일한다."""
    if value is None:
        return []
    if isinstance(value, str):
        raw_items = re.findall(r"[A-E]", value.upper())
    elif isinstance(value, list):
        raw_items = []
        for item in value:
            if isinstance(item, str):
                raw_items.extend(re.findall(r"[A-E]", item.upper()))
            elif isinstance(item, dict):
                raw_items.extend(_coerce_letters(item.get("letter") or item.get("sign") or item.get("name")))
    elif isinstance(value, dict):
        raw_items = []
        for key, val in value.items():
            if bool(val):
                raw_items.extend(re.findall(r"[A-E]", str(key).upper()))
    else:
        raw_items = []

    letters: list[str] = []
    for letter in raw_items:
        if letter in {"A", "B", "C", "D", "E"} and letter not in letters:
            letters.append(letter)
    return letters


def _extract_json_object(text: str) -> dict[str, Any] | None:
    """VLM 응답에서 첫 JSON object를 최대한 관대하게 추출한다."""
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = stripped.strip("`")
        if stripped.lower().startswith("json"):
            stripped = stripped[4:].strip()
    start = stripped.find("{")
    end = stripped.rfind("}")
    if start < 0 or end <= start:
        return None
    try:
        parsed = json.loads(stripped[start : end + 1])
    except json.JSONDecodeError:
        return None
    return parsed if isinstance(parsed, dict) else None


def _target_offset_from_position(position: str) -> float:
    return float(TEXT_SIGN_POSITION_OFFSET_DEG.get(position, 0.0))


def _parse_vlm_sign_result(text: str, expected_pad: str | None = None) -> dict[str, Any] | None:
    """VLM 응답을 문자 탐색에 필요한 표준 dict로 변환한다.

    표준 dict:
      visible_letters: 실제 화면에서 보인 것으로 판단한 A/B/C/D/E list
      target_visible: expected_pad가 실제로 보이는지
      target_position: left/center/right/unknown
      target_distance: near/medium/far/unknown
      target_offset_deg: camera 중심선 기준 target 각도. left는 +, right는 -.
    """
    expected = (expected_pad or "").upper()
    data = _extract_json_object(text)
    if data is None:
        letters = []
        for letter in re.findall(r"\b[A-E]\b", text.upper()):
            if letter not in letters:
                letters.append(letter)
        return {
            "visible_letters": letters,
            "target_visible": expected in letters if expected else False,
            "target_position": "unknown",
            "target_distance": "unknown",
            "target_offset_deg": None,
            "raw_text": text[:200],
        }

    letters = _coerce_letters(
        data.get("visible_letters")
        or data.get("letters")
        or data.get("signs")
        or data.get("visible_signs")
    )

    target_position = _normalize_vlm_position(
        data.get("target_position")
        or data.get("position")
        or data.get("target_location_in_image")
    )
    target_distance = _normalize_vlm_distance(
        data.get("target_distance")
        or data.get("distance")
        or data.get("target_distance_estimate")
    )
    target_offset_deg = _coerce_float(
        data.get("target_offset_deg")
        or data.get("target_bearing_offset_deg")
        or data.get("horizontal_offset_deg")
        or data.get("offset_deg")
    )

    sign_items = data.get("signs") or data.get("visible_signs")
    if isinstance(sign_items, list):
        for item in sign_items:
            if not isinstance(item, dict):
                continue
            item_letters = _coerce_letters(item.get("letter") or item.get("sign") or item.get("name"))
            for letter in item_letters:
                if letter not in letters:
                    letters.append(letter)
            if expected and expected in item_letters:
                if target_position == "unknown":
                    target_position = _normalize_vlm_position(item.get("position") or item.get("location"))
                if target_distance == "unknown":
                    target_distance = _normalize_vlm_distance(item.get("distance") or item.get("distance_estimate"))
                if target_offset_deg is None:
                    target_offset_deg = _coerce_float(
                        item.get("target_offset_deg")
                        or item.get("bearing_offset_deg")
                        or item.get("horizontal_offset_deg")
                        or item.get("offset_deg")
                    )

    explicit_visible = data.get("target_visible")
    if isinstance(explicit_visible, bool):
        target_visible = explicit_visible
    else:
        target_visible = expected in letters if expected else False

    return {
        "visible_letters": letters,
        "target_visible": bool(target_visible),
        "target_position": target_position,
        "target_distance": target_distance,
        "target_offset_deg": target_offset_deg,
        "raw_text": text[:200],
    }


def build_text_sign_search_prompt(expected_pad: str) -> str:
    """패드 탐색용 VLM prompt. 색상 판단을 금지하고 실제 보이는 문자만 읽게 한다."""
    return (
        "Robot POV camera frame에서 창고 sign letter를 읽으세요. "
        "색상, 큐브 색, 바닥 색, 패드 색은 절대 판단 근거로 사용하지 마세요. "
        "실제로 화면에 보이는 큰 표지판 문자 A/B/C/D/E만 읽으세요. "
        f"찾아야 하는 target sign은 {expected_pad}입니다. "
        "A는 source이므로 destination이 아닙니다. "
        "응답은 JSON 하나만 반환하세요: "
        '{"visible_letters": ["C"], "target_visible": true, '
        '"target_position": "left|center|right|unknown", '
        '"target_distance": "near|medium|far|unknown", '
        '"target_offset_deg": 0}. '
        "target_position은 target sign이 보일 때 화면 기준 위치입니다. "
        "target_offset_deg는 카메라 화면 중심선에서 target sign 중심까지의 가로 각도 추정값입니다. "
        "중요: target이 화면 왼쪽이면 양수(+), 오른쪽이면 음수(-), 중앙이면 0으로 주세요. "
        "각도 추정이 어렵다면 null로 주세요. "
        "보이는 sign이 없으면 visible_letters는 빈 배열, target_visible은 false입니다."
    )


async def read_signs_with_vlm(ctx: Any, expected_pad: str) -> dict[str, Any] | None:
    """현재 POV frame에서 VLM으로 sign letter만 읽는다. 실패하면 None."""
    api_key = os.environ.get("TOKAMAK_API_KEY", "")
    if not api_key:
        print("[vlm-text] TOKAMAK_API_KEY 없음 → 문자 기반 패드 탐색 불가")
        return None
    try:
        jpeg = await get_camera_frame(ctx)
        jpeg = prepare_vlm_image(jpeg)
        text = await asyncio.wait_for(
            asyncio.to_thread(ask_vlm, jpeg, build_text_sign_search_prompt(expected_pad), api_key=api_key),
            timeout=VLM_FRAME_TIMEOUT_S,
        )
    except asyncio.CancelledError as exc:
        print(f"[vlm-text] 호출 취소({exc!r}) → 이번 frame은 인식 실패로 처리")
        return None
    except Exception as exc:
        print(f"[vlm-text] 호출 실패({exc!r}) → 이번 frame은 인식 실패로 처리 (timeout={VLM_FRAME_TIMEOUT_S}s)")
        return None

    parsed = _parse_vlm_sign_result(text, expected_pad)
    if parsed is None:
        print(f"[vlm-text] 응답 파싱 실패: {text[:120]!r}")
        return None
    return parsed


def xy_distance(a: tuple[float, float] | None, b: tuple[float, float] | None) -> float | None:
    """두 world 좌표 사이 거리. 좌표가 없으면 None."""
    if a is None or b is None:
        return None
    return math.hypot(float(a[0]) - float(b[0]), float(a[1]) - float(b[1]))


def _bearing_from_status_and_image(robot_status: Any, *, head_yaw_rad: float, reading: dict[str, Any]) -> float | None:
    """robot yaw + head yaw + image offset으로 target sign의 world bearing(rad)을 만든다."""
    robot_yaw_deg = robot_yaw_deg_from_status(robot_status)
    if robot_yaw_deg is None:
        return None
    offset_deg = reading.get("target_offset_deg")
    if offset_deg is None:
        offset_deg = _target_offset_from_position(reading.get("target_position", "unknown"))
    offset_deg = float(offset_deg)
    return math.radians(robot_yaw_deg + math.degrees(head_yaw_rad) + offset_deg)


def _make_sign_observation(
    robot_status: Any,
    *,
    head_yaw_rad: float,
    reading: dict[str, Any],
    label: str,
) -> dict[str, Any] | None:
    xy = robot_xy_from_status(robot_status)
    bearing_rad = _bearing_from_status_and_image(robot_status, head_yaw_rad=head_yaw_rad, reading=reading)
    if xy is None or bearing_rad is None:
        return None
    return {
        "label": label,
        "xy": xy,
        "bearing_rad": bearing_rad,
        "bearing_deg": math.degrees(bearing_rad),
        "head_yaw_rad": head_yaw_rad,
        "reading": reading,
    }


def _angle_between_bearings_deg(a_rad: float, b_rad: float) -> float:
    diff = abs(math.atan2(math.sin(a_rad - b_rad), math.cos(a_rad - b_rad)))
    return math.degrees(diff)


def _ray_intersection(
    obs1: dict[str, Any],
    obs2: dict[str, Any],
) -> tuple[float, float] | None:
    """두 관측 ray의 교차점. 두 ray 앞쪽에 있는 교차점만 인정한다."""
    x1, y1 = obs1["xy"]
    x2, y2 = obs2["xy"]
    a1 = float(obs1["bearing_rad"])
    a2 = float(obs2["bearing_rad"])
    d1 = (math.cos(a1), math.sin(a1))
    d2 = (math.cos(a2), math.sin(a2))

    def cross(u: tuple[float, float], v: tuple[float, float]) -> float:
        return u[0] * v[1] - u[1] * v[0]

    denom = cross(d1, d2)
    if abs(denom) < 1e-4:
        return None
    p21 = (x2 - x1, y2 - y1)
    t = cross(p21, d2) / denom
    u = cross(p21, d1) / denom
    if t <= 0.0 or u <= 0.0:
        return None
    if t > TRIANGULATION_MAX_SIGN_DISTANCE_M or u > TRIANGULATION_MAX_SIGN_DISTANCE_M:
        return None
    return (x1 + t * d1[0], y1 + t * d1[1])


def _triangulate_sign_xy(observations: list[dict[str, Any]]) -> tuple[float, float] | None:
    """여러 관측 중 기하적으로 가장 안정적인 두 ray를 골라 sign 좌표를 추정한다."""
    best: tuple[float, tuple[float, float], tuple[str, str], float] | None = None
    for i in range(len(observations)):
        for j in range(i + 1, len(observations)):
            obs1, obs2 = observations[i], observations[j]
            baseline = xy_distance(obs1["xy"], obs2["xy"]) or 0.0
            if baseline < TRIANGULATION_MIN_BASELINE_M:
                continue
            angle_deg = _angle_between_bearings_deg(obs1["bearing_rad"], obs2["bearing_rad"])
            if angle_deg < TRIANGULATION_MIN_CROSS_ANGLE_DEG or angle_deg > TRIANGULATION_MAX_CROSS_ANGLE_DEG:
                continue
            point = _ray_intersection(obs1, obs2)
            if point is None:
                continue
            # 60~90도에 가까울수록, baseline이 클수록 더 안정적이다.
            angle_score = 90.0 - abs(75.0 - angle_deg)
            score = angle_score + baseline * 10.0
            if best is None or score > best[0]:
                best = (score, point, (obs1["label"], obs2["label"]), angle_deg)
    if best is None:
        return None
    _, point, labels, angle_deg = best
    print(f"[triangulate] 사용 관측={labels}, 교차각={angle_deg:.1f}°, sign_xy={point}")
    return point


def _approach_xy_from_sign_xy(
    current_xy: tuple[float, float] | None,
    sign_xy: tuple[float, float] | None,
    *,
    standoff_m: float = TRIANGULATION_APPROACH_STANDOFF_M,
) -> tuple[float, float] | None:
    """추정된 sign/pad 지점까지 깊게 들어가지 않고, 앞에서 멈추는 접근 좌표를 만든다."""
    if current_xy is None or sign_xy is None:
        return None
    dx = float(sign_xy[0]) - float(current_xy[0])
    dy = float(sign_xy[1]) - float(current_xy[1])
    dist = math.hypot(dx, dy)
    if dist < 1e-6:
        return current_xy
    if dist <= standoff_m + 0.15:
        # 이미 충분히 가까우면 현재 위치를 접근점으로 쓴다.
        return current_xy
    ux, uy = dx / dist, dy / dist
    return (float(sign_xy[0]) - standoff_m * ux, float(sign_xy[1]) - standoff_m * uy)


def _single_ray_fallback_xy(
    observation: dict[str, Any],
    current_xy: tuple[float, float] | None,
) -> tuple[float, float] | None:
    """삼각측량 실패 시, 현재 관측 방향으로만 매우 보수적으로 접근하는 fallback."""
    if current_xy is None:
        return None
    reading = observation.get("reading", {})
    distance_m = TEXT_SIGN_FALLBACK_DISTANCE_M.get(
        reading.get("target_distance", "unknown"),
        TEXT_SIGN_FALLBACK_DISTANCE_M["unknown"],
    )
    bearing = float(observation["bearing_rad"])
    sign_xy = (
        float(observation["xy"][0]) + distance_m * math.cos(bearing),
        float(observation["xy"][1]) + distance_m * math.sin(bearing),
    )
    return _approach_xy_from_sign_xy(current_xy, sign_xy, standoff_m=0.45)


def _is_safe_destination_candidate(
    memory: AgentMemory,
    expected_pad: str,
    xy: tuple[float, float] | None,
    *,
    source: str,
) -> bool:
    """B/C/D/E 임시 접근 좌표 유효성 검사.

    v9 수정:
    - A와의 거리 기반 하드 필터는 제거한다.
    - 과거에는 색상/rough 좌표가 A 근처를 목적지로 오인하는 것을 막기 위해 사용했지만,
      현재 front-loop는 VLM이 목적지 문자 자체를 읽고 center/near 상태를 확인하므로
      이 조건이 place 단계 진입을 막는 문제가 더 컸다.
    - 실제 안전성은 place 직전 VLM 확인 + place_entity 후 held 상태 검증으로 판단한다.
    """
    if xy is None:
        return False
    return True



async def _turn_body_toward_observed_sign(ctx: Any, *, head_yaw: float, reading: dict[str, Any]) -> bool:
    """head_yaw/화면 offset으로 본 target 방향만큼 body를 돌려 head=0 정면 관측을 만든다.

    중요 수정:
    - 이전 로그에서 pos=left인데 +회전 후에도 계속 left였으므로 회전 부호를 반전한다.
    - 회전 전후 yaw를 출력해 실제로 몸이 얼마나 돌아갔는지 검증 가능하게 한다.
    - 회전 RPC가 실패하면 False를 반환해 이번 triangulation만 실패 처리한다.
    """
    offset_deg = reading.get("target_offset_deg")
    if offset_deg is None:
        offset_deg = _target_offset_from_position(reading.get("target_position", "unknown"))

    observed_offset_rad = float(head_yaw) + math.radians(float(offset_deg or 0.0))
    # Menlo 환경에서 카메라 좌우 부호와 wz 부호가 반대로 관측되어 보정 부호를 곱한다.
    turn_rad = TRIANGULATION_TURN_SIGN * observed_offset_rad

    if abs(turn_rad) < 0.08:
        await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
        return True

    before_status = await get_robot_status(ctx)
    yaw_before = robot_yaw_deg_from_status(before_status)
    duration = min(abs(turn_rad) / TRIANGULATION_ALIGN_WZ, TRIANGULATION_ALIGN_MAX_TURN_S)
    wz = TRIANGULATION_ALIGN_WZ if turn_rad > 0 else -TRIANGULATION_ALIGN_WZ

    print(
        f"[triangulate] body 정면화 회전: observed_offset={math.degrees(observed_offset_rad):+.1f}°, "
        f"turn={math.degrees(turn_rad):+.1f}°, wz={wz:+.2f}, duration={duration:.2f}s"
    )
    await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
    result = await move_velocity(ctx, vx=0.02, wz=wz, duration_s=duration)
    await asyncio.sleep(0.25)

    after_status = await get_robot_status(ctx)
    yaw_after = robot_yaw_deg_from_status(after_status)
    if yaw_before is not None and yaw_after is not None:
        delta = math.atan2(
            math.sin(math.radians(yaw_after - yaw_before)),
            math.cos(math.radians(yaw_after - yaw_before)),
        )
        print(f"[triangulate] yaw before={yaw_before:.1f}°, after={yaw_after:.1f}°, delta={math.degrees(delta):+.1f}°")

    if isinstance(result, dict) and result.get("status") == "failed":
        print("[triangulate] body 회전 command 실패 → 이번 후보 실패")
        return False
    return True


async def _make_frontal_observation(
    ctx: Any,
    expected_pad: str,
    read_limited: Callable[[], Awaitable[dict[str, Any] | None]],
    *,
    initial_head_yaw: float,
    initial_reading: dict[str, Any],
    label: str,
) -> dict[str, Any] | None:
    """target sign을 body 정면으로 맞춘 뒤 head=0 사진을 다시 읽어 관측값을 만든다.

    이번 버전에서는 left/right가 남아도 무리하게 추가 VLM 호출로 재보정하지 않는다.
    target이 보이기만 하면 image offset을 bearing에 반영한 관측 ray로 사용한다.
    """
    turned = await _turn_body_toward_observed_sign(ctx, head_yaw=initial_head_yaw, reading=initial_reading)
    if not turned:
        return None

    await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
    await asyncio.sleep(0.30)
    reading = await read_limited()
    if reading is None or not reading.get("target_visible"):
        print(f"[triangulate] {label}: 정면화 후 {expected_pad}가 보이지 않음 → 관측 실패")
        return None

    pos = reading.get("target_position")
    if pos in {"left", "right"}:
        print(f"[triangulate] {label}: 정면화 후에도 pos={pos} → 추가 보정 없이 offset bearing 관측으로 사용")

    status = await get_robot_status(ctx)
    obs = _make_sign_observation(status, head_yaw_rad=0.0, reading=reading, label=label)
    if obs is not None:
        print(
            f"[triangulate] {label}: xy={obs['xy']}, bearing={obs['bearing_deg']:.1f}°, "
            f"visible={reading.get('visible_letters')}, pos={reading.get('target_position')}, "
            f"dist={reading.get('target_distance')}, offset={reading.get('target_offset_deg')}"
        )
    return obs


async def _make_baseline_observation(
    ctx: Any,
    expected_pad: str,
    read_limited: Callable[[], Awaitable[dict[str, Any] | None]],
) -> dict[str, Any] | None:
    """조금 옆으로 이동한 뒤 target sign을 다시 읽어 두 번째 관측 ray를 만든다.

    이전 버전은 baseline 위치에서도 다시 정면화 회전 + VLM 확인을 수행해서 호출량이 컸다.
    이번 버전은 baseline에서 target이 보이는 순간의 head_yaw와 image offset을 그대로 bearing에 반영한다.
    """
    before_status = await get_robot_status(ctx)
    before_xy = robot_xy_from_status(before_status)

    chosen_direction = 1.0
    moved = 0.0
    for direction in (1.0, -1.0):
        result = await move_velocity(
            ctx,
            vx=0.0,
            vy=direction * TRIANGULATION_BASELINE_VY,
            wz=0.0,
            duration_s=TRIANGULATION_BASELINE_DURATION_S,
        )
        after_status = await get_robot_status(ctx)
        after_xy = robot_xy_from_status(after_status)
        moved = xy_distance(before_xy, after_xy) or 0.0
        print(f"[triangulate] baseline lateral 이동 direction={direction:+.0f}, moved={moved:.2f}m")
        if not (isinstance(result, dict) and result.get("status") == "failed") and moved >= TRIANGULATION_MIN_BASELINE_M:
            chosen_direction = direction
            break

    # baseline 위치에서 sign을 다시 찾는다. 정면화는 하지 않고 head_yaw를 bearing에 직접 반영한다.
    for head_yaw in (0.0, -0.55, 0.55, -0.9, 0.9):
        head_result = await set_head(ctx, yaw=head_yaw, pitch=TEXT_SEARCH_PITCH)
        if isinstance(head_result, dict) and head_result.get("status") == "failed":
            continue
        await asyncio.sleep(0.25)
        reading = await read_limited()
        if reading is None:
            continue
        print(
            f"[triangulate] baseline scan head={head_yaw:+.2f}, visible={reading.get('visible_letters')}, "
            f"target={expected_pad}, pos={reading.get('target_position')}, dist={reading.get('target_distance')}, "
            f"offset={reading.get('target_offset_deg')}"
        )
        if reading.get("target_visible"):
            status = await get_robot_status(ctx)
            obs = _make_sign_observation(status, head_yaw_rad=head_yaw, reading=reading, label=f"baseline_{chosen_direction:+.0f}")
            if obs is not None:
                print(
                    f"[triangulate] baseline 관측 사용: xy={obs['xy']}, head={head_yaw:+.2f}, "
                    f"bearing={obs['bearing_deg']:.1f}°"
                )
            return obs
    return None


async def _triangulated_pad_approach_from_detection(
    ctx: Any,
    memory: AgentMemory,
    expected_pad: str,
    read_limited: Callable[[], Awaitable[dict[str, Any] | None]],
    *,
    found_head_yaw: float,
    found_reading: dict[str, Any],
) -> tuple[float, float] | None:
    """발견 사진 + 정면화 사진 + baseline 사진을 이용해 패드 접근 좌표를 추정한다.

    실패를 오래 끌지 않고 한 후보당 짧게 판단한다. C를 찾았지만 정면화/VLM이 불안정하면
    다음 cycle에서 다시 시도하는 편이 RPC timeout을 줄인다.
    """
    found_status = await get_robot_status(ctx)
    found_obs = _make_sign_observation(
        found_status,
        head_yaw_rad=found_head_yaw,
        reading=found_reading,
        label="found",
    )
    if found_obs is not None:
        print(
            f"[triangulate] found: xy={found_obs['xy']}, head={found_head_yaw:+.2f}, "
            f"bearing={found_obs['bearing_deg']:.1f}°, pos={found_reading.get('target_position')}, "
            f"dist={found_reading.get('target_distance')}, offset={found_reading.get('target_offset_deg')}"
        )

    frontal_obs = await _make_frontal_observation(
        ctx,
        expected_pad,
        read_limited,
        initial_head_yaw=found_head_yaw,
        initial_reading=found_reading,
        label="frontal",
    )

    baseline_obs = await _make_baseline_observation(ctx, expected_pad, read_limited)

    observations = [obs for obs in (found_obs, frontal_obs, baseline_obs) if obs is not None]
    current_status = await get_robot_status(ctx)
    current_xy = robot_xy_from_status(current_status)

    sign_xy = _triangulate_sign_xy(observations)
    if sign_xy is not None:
        approach_xy = _approach_xy_from_sign_xy(current_xy, sign_xy)
        if _is_safe_destination_candidate(memory, expected_pad, approach_xy, source="triangulation"):
            print(f"[triangulate] {expected_pad} sign_xy={sign_xy} → approach_xy={approach_xy}")
            return approach_xy
        return None

    # 삼각측량 실패 시 fallback은 target이 비교적 정면/중앙에 보인 관측에만 허용한다.
    fallback_source = frontal_obs or found_obs
    if fallback_source is not None:
        reading = fallback_source.get("reading", {})
        if reading.get("target_position") == "center" or abs(float(reading.get("target_offset_deg") or 0.0)) <= 10.0:
            fallback_xy = _single_ray_fallback_xy(fallback_source, current_xy)
            if _is_safe_destination_candidate(memory, expected_pad, fallback_xy, source="single-ray-fallback"):
                print(f"[triangulate] 삼각측량 실패 → center 관측 기반 single-ray 보수 접근좌표 사용: {fallback_xy}")
                return fallback_xy

    print(f"[triangulate] {expected_pad} 관측은 있었지만 안정적인 좌표를 만들지 못함")
    return None

# ---------------------------------------------------------------------------
# v5 수정: VLM 호출을 줄인 정면 폐루프 접근 기반 패드 탐색
# ---------------------------------------------------------------------------
# v4 삼각측량은 이론적으로 좋지만, 실제 로그에서는 VLM Timeout 때문에 관측 2개를 만들기 전에
# 탐색이 막혔다. v5는 head_yaw 3방향 스캔과 반복 삼각측량을 줄이고,
# head_yaw=0 정면 사진 1장씩만 사용해 sign을 중앙에 맞추며 조금씩 접근한다.
FRONT_LOOP_MAX_STEPS = 12
FRONT_LOOP_MAX_VLM_CALLS = 10
FRONT_LOOP_TIMEOUT_ABORTS = 2
FRONT_LOOP_SCAN_TURN_DEG = 45.0   # target이 안 보이면 한 방향으로 360도에 가깝게 훑는다
FRONT_LOOP_POSITION_TURN_DEG = 22.0
FRONT_LOOP_FORWARD_DURATION = {
    "far": 2.00,
    "medium": 1.50,
    "unknown": 1.20,
    "near": 0.70,
}
# v10: v9 로그에서 vx=0.25, duration=1.35에도 실제 이동이 약 8cm에 그쳤다.
# center 상태에서는 더 강하게 접근하고, 이동량이 작으면 unblock arc를 수행한다.
FRONT_LOOP_FORWARD_VX = 0.35
FRONT_LOOP_FINAL_NUDGE_VX = 0.20
FRONT_LOOP_FINAL_NUDGE_DURATION = 0.35
FRONT_LOOP_MIN_MOVE_M = 0.10
FRONT_LOOP_UNBLOCK_WZ = 0.12
FRONT_LOOP_UNBLOCK_DURATION_S = 1.00
# A와의 최소거리 하드 필터는 제거한다. place-ready 여부는 target sign의 center/near와 실제 delivered 증가로만 검증한다.
FRONT_LOOP_MIN_SAFE_FROM_A_M = 0.0

# v8: 튜토리얼 검증 결과 반영.
# Workshop 01/03/04는 공통으로 "cannot spin in place"를 강조하고,
# 회전에는 wz만 쓰지 말고 vx=0.15~0.25를 함께 쓰라고 안내한다.
# v7의 vx=0.03은 사실상 제자리 회전에 가까워 실제 yaw가 9도 근처에서 멈췄다.
# 따라서 v8은 정확한 yaw feedback 회전을 고집하지 않고, 튜토리얼식 "arc turn + 다시 보기"로 변경한다.
ARC_TURN_VX = 0.20
ARC_TURN_WZ = 0.50       # 튜토리얼 예시값. wz max는 0.6이므로 clipping 없이 동작한다.
ARC_SCAN_VX = 0.25
ARC_SCAN_WZ = 0.30       # target 미발견 스캔은 넓은 원호로 부드럽게 돈다.
ARC_ALIGN_VX = 0.20
ARC_ALIGN_WZ = 0.35      # left/right 정렬은 너무 과격하지 않게 짧게 돈다.
ARC_TURN_MIN_S = 0.35
ARC_TURN_MAX_S = 3.20
ARC_SCAN_DURATION_S = 2.60
ARC_ALIGN_MAX_DURATION_S = 1.20
ARC_TURN_SETTLE_S = 0.35
ARC_STALL_RETRY_VX = 0.25
ARC_STALL_RETRY_DURATION_S = 0.80
ARC_MIN_EXPECTED_DELTA_DEG = 3.0
FRONT_LOOP_RETRY_OFFSET_DEG = 45.0


def _closed_loop_turn_deg_from_reading(reading: dict[str, Any]) -> float:
    """VLM의 target 위치를 body arc turn 각도로 변환한다.

    prompt에서 target_offset_deg는 "왼쪽 +, 오른쪽 -"로 요청한다.
    값이 없으면 left/right 기본값을 사용한다. v8에서는 이 각도를 정확히 맞추려 하지 않고,
    arc turn 후 다시 VLM으로 확인한다.
    """
    pos = reading.get("target_position", "unknown")
    offset_deg = reading.get("target_offset_deg")
    if offset_deg is None:
        if pos == "left":
            offset_deg = FRONT_LOOP_POSITION_TURN_DEG
        elif pos == "right":
            offset_deg = -FRONT_LOOP_POSITION_TURN_DEG
        else:
            offset_deg = 0.0
    try:
        offset_deg = float(offset_deg)
    except Exception:
        offset_deg = 0.0
    # 카메라 HFOV 힌트 기준으로 한 번에 너무 큰 회전을 하지 않도록 제한한다.
    return max(-35.0, min(35.0, offset_deg))


def _normalize_deg(angle_deg: float) -> float:
    """각도를 [-180, 180) 범위로 정규화한다."""
    return ((float(angle_deg) + 180.0) % 360.0) - 180.0


def _signed_delta_deg(target_deg: float, current_deg: float) -> float:
    """current에서 target까지 필요한 최단 회전각(deg). 양수면 +wz 방향."""
    return _normalize_deg(float(target_deg) - float(current_deg))


def _duration_for_arc_turn(turn_deg: float, wz: float) -> float:
    """튜토리얼 공식 angle(rad)=wz*duration 기반 회전 시간."""
    duration = math.radians(abs(float(turn_deg))) / max(abs(float(wz)), 1e-6)
    return max(ARC_TURN_MIN_S, min(ARC_TURN_MAX_S, duration))


async def _turn_body_degrees(ctx: Any, turn_deg: float, *, reason: str) -> bool:
    """튜토리얼식 arc turn으로 body 방향을 바꾼다.

    이 로봇은 제자리 회전이 되지 않으므로 반드시 vx와 wz를 함께 준다.
    v7처럼 yaw를 정확히 목표각까지 맞추려고 pulse를 반복하면 같은 자리에서 9도 근처에
    멈추는 현상이 발생했다. v8은 한 번의 arc motion 후 VLM으로 다시 확인하는 방식이다.
    """
    if abs(turn_deg) < 4.0:
        return True

    await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
    await asyncio.sleep(0.10)

    # target 미발견 스캔은 넓은 원호 이동, target 정렬은 짧은 arc turn을 쓴다.
    sign = 1.0 if turn_deg > 0 else -1.0
    if "scan" in reason or "retry-offset" in reason:
        vx = ARC_SCAN_VX
        wz_mag = ARC_SCAN_WZ
        duration = ARC_SCAN_DURATION_S if abs(turn_deg) <= FRONT_LOOP_SCAN_TURN_DEG + 5 else _duration_for_arc_turn(turn_deg, wz_mag)
    else:
        vx = ARC_ALIGN_VX
        wz_mag = ARC_ALIGN_WZ
        duration = min(ARC_ALIGN_MAX_DURATION_S, _duration_for_arc_turn(turn_deg, wz_mag))

    before = await get_robot_status(ctx)
    yaw_before = robot_yaw_deg_from_status(before)
    xy_before = robot_xy_from_status(before)
    print(
        f"[front-loop] body arc-turn({reason}): request={turn_deg:+.1f}°, "
        f"vx={vx:.2f}, wz={sign*wz_mag:+.2f}, duration={duration:.2f}s, "
        f"yaw_before={yaw_before if yaw_before is not None else 'unknown'}, xy_before={xy_before}"
    )

    result = await move_velocity(ctx, vx=vx, wz=sign * wz_mag, duration_s=duration)
    await asyncio.sleep(ARC_TURN_SETTLE_S)

    after = await get_robot_status(ctx)
    yaw_after = robot_yaw_deg_from_status(after)
    xy_after = robot_xy_from_status(after)
    yaw_delta = None
    if yaw_before is not None and yaw_after is not None:
        yaw_delta = _signed_delta_deg(yaw_after, yaw_before)
    print(
        f"[front-loop] arc-turn-done({reason}): yaw_after={yaw_after if yaw_after is not None else 'unknown'}, "
        f"yaw_delta={yaw_delta if yaw_delta is not None else 'unknown'}, xy_after={xy_after}, result={result_summary(result)}"
    )

    failed = isinstance(result, dict) and result.get("status") == "failed"
    # 회전이 거의 안 먹은 경우, 튜토리얼 예시값에 더 가까운 vx=0.25/wz=0.5로 짧게 한 번만 재시도한다.
    # 단, 무한 feedback은 하지 않는다. 이후 판단은 다음 VLM frame에 맡긴다.
    if (not failed) and yaw_delta is not None and abs(yaw_delta) < ARC_MIN_EXPECTED_DELTA_DEG and abs(turn_deg) >= 12.0:
        retry_duration = min(ARC_STALL_RETRY_DURATION_S, max(0.45, duration * 0.65))
        print(
            f"[front-loop] arc-turn delta가 작음({yaw_delta:+.1f}°) → "
            f"tutorial retry vx={ARC_STALL_RETRY_VX:.2f}, wz={sign*ARC_TURN_WZ:+.2f}, duration={retry_duration:.2f}s"
        )
        retry = await move_velocity(ctx, vx=ARC_STALL_RETRY_VX, wz=sign * ARC_TURN_WZ, duration_s=retry_duration)
        await asyncio.sleep(ARC_TURN_SETTLE_S)
        retry_status = await get_robot_status(ctx)
        retry_yaw = robot_yaw_deg_from_status(retry_status)
        retry_delta = _signed_delta_deg(retry_yaw, yaw_after) if (retry_yaw is not None and yaw_after is not None) else None
        print(
            f"[front-loop] arc-turn-retry-done: yaw={retry_yaw if retry_yaw is not None else 'unknown'}, "
            f"retry_delta={retry_delta if retry_delta is not None else 'unknown'}, result={result_summary(retry)}"
        )
        failed = isinstance(retry, dict) and retry.get("status") == "failed"

    return not failed


async def debug_turn_once(ctx: Any, vx: float = 0.20, wz: float = 0.50, duration_s: float = 1.0) -> None:
    """회전 primitive 단독 점검용 helper. 필요할 때 별도 셀에서 호출해 실제 yaw_delta를 확인한다."""
    before = await get_robot_status(ctx)
    y0 = robot_yaw_deg_from_status(before)
    xy0 = robot_xy_from_status(before)
    print(f"[debug-turn] before yaw={y0}, xy={xy0}, command vx={vx}, wz={wz}, duration={duration_s}")
    result = await move_velocity(ctx, vx=vx, wz=wz, duration_s=duration_s)
    await asyncio.sleep(0.5)
    after = await get_robot_status(ctx)
    y1 = robot_yaw_deg_from_status(after)
    xy1 = robot_xy_from_status(after)
    delta = _signed_delta_deg(y1, y0) if (y0 is not None and y1 is not None) else None
    print(f"[debug-turn] after yaw={y1}, xy={xy1}, delta={delta}, result={result_summary(result)}")


async def _forward_nudge(ctx: Any, *, distance_label: str, final: bool = False) -> bool:
    """target sign을 정면에 둔 상태에서 전진한다.

    v10:
      - v9보다 vx를 높여 실제 접근량을 늘린다.
      - 전진 명령 후 실제 xy 이동량을 측정한다.
      - moved < FRONT_LOOP_MIN_MOVE_M이면 물리적으로 막혔거나 방향이 덜 맞은 것으로 보고
        작은 wz를 섞은 unblock arc를 1회 수행한다.
    """
    if final:
        vx = FRONT_LOOP_FINAL_NUDGE_VX
        duration = FRONT_LOOP_FINAL_NUDGE_DURATION
    else:
        vx = FRONT_LOOP_FORWARD_VX
        duration = FRONT_LOOP_FORWARD_DURATION.get(distance_label, FRONT_LOOP_FORWARD_DURATION["unknown"])
    before = await get_robot_status(ctx)
    xy_before = robot_xy_from_status(before)
    yaw_before = robot_yaw_deg_from_status(before)
    print(
        f"[front-loop] target center, dist={distance_label} → 전진 "
        f"vx={vx:.2f}, duration={duration:.2f}s, xy_before={xy_before}, yaw={yaw_before}"
    )
    result = await move_velocity(ctx, vx=vx, wz=0.0, duration_s=duration)
    await asyncio.sleep(0.25)
    after = await get_robot_status(ctx)
    xy_after = robot_xy_from_status(after)
    yaw_after = robot_yaw_deg_from_status(after)
    moved = xy_distance(xy_before, xy_after) if (xy_before is not None and xy_after is not None) else None
    print(
        f"[front-loop] forward-done: xy_after={xy_after}, moved={moved if moved is not None else 'unknown'}, "
        f"yaw_after={yaw_after}, result={result_summary(result)}"
    )

    failed = isinstance(result, dict) and result.get("status") == "failed"
    if failed:
        return False

    # 실제 이동이 너무 작으면, 튜토리얼식으로 약한 arc를 섞어 막힘을 푼다.
    if (not final) and moved is not None and moved < FRONT_LOOP_MIN_MOVE_M:
        unblock_sign = 1.0
        print(
            f"[front-loop] forward 이동량 부족({moved:.3f}m < {FRONT_LOOP_MIN_MOVE_M:.2f}m) "
            f"→ unblock arc vx={FRONT_LOOP_FORWARD_VX:.2f}, wz={unblock_sign*FRONT_LOOP_UNBLOCK_WZ:+.2f}, "
            f"duration={FRONT_LOOP_UNBLOCK_DURATION_S:.2f}s"
        )
        retry = await move_velocity(
            ctx,
            vx=FRONT_LOOP_FORWARD_VX,
            wz=unblock_sign * FRONT_LOOP_UNBLOCK_WZ,
            duration_s=FRONT_LOOP_UNBLOCK_DURATION_S,
        )
        await asyncio.sleep(0.25)
        retry_status = await get_robot_status(ctx)
        retry_xy = robot_xy_from_status(retry_status)
        retry_yaw = robot_yaw_deg_from_status(retry_status)
        retry_moved = xy_distance(xy_after, retry_xy) if (xy_after is not None and retry_xy is not None) else None
        print(
            f"[front-loop] unblock-done: xy_after={retry_xy}, moved={retry_moved if retry_moved is not None else 'unknown'}, "
            f"yaw_after={retry_yaw}, result={result_summary(retry)}"
        )
        if isinstance(retry, dict) and retry.get("status") == "failed":
            return False

    return True


# ---------------------------------------------------------------------------
# Level_1_v4_0704: text-only sign 관측 → 방향 hint/8방향 panorama → 다중 ray 좌표 추정 → go_to
# ---------------------------------------------------------------------------
# v1_0704 로그 문제:
#   - 같은 자리에서 head만 좌/중/우로 돌려 C를 여러 번 봤기 때문에 triangulation baseline이 없었다.
#   - baseline shift가 3~4cm 수준이라 ray 교차가 불안정했고, C를 계속 봐도 좌표가 확정되지 않았다.
# v2 수정:
#   - target sign이 보이면 현재 pose에서 관측 ray를 저장한다.
#   - 그 ray에 수직인 안전한 local observation waypoint를 만들고 coordinate go_to로 다른 pose로 이동한다.
#   - 서로 다른 pose에서 다시 sign을 관찰해 baseline을 확보한다.
#   - 여러 ray는 pair 하나가 아니라 weighted least-squares로 합쳐 VLM offset 노이즈를 완화한다.
#   - 좌표가 확정되면 그때만 known_locations[패드]에 approach_xy를 저장하고 navigate_to_pad에서 go_to를 사용한다.

# v3에서는 v2 multiview를 유지하되, target 미발견/삼각측량 실패 시 8방향 body panorama scan으로 확장한다.
TRIANGULATION_MIN_BASELINE_M = 0.22
TRIANGULATION_MIN_CROSS_ANGLE_DEG = 5.0
TRIANGULATION_MAX_CROSS_ANGLE_DEG = 170.0
TRIANGULATION_MAX_SIGN_DISTANCE_M = 6.8
TRIANGULATION_APPROACH_STANDOFF_M = 0.30

PAD_OBS_MAX_PER_PAD = 18
PAD_OBS_MIN_COUNT_FOR_TRIANGULATION = 2
PAD_TRIANGULATION_SEARCH_STEPS = 5
PAD_TRIANGULATION_HEAD_YAWS = (-0.85, -0.45, 0.0, 0.45, 0.85)
PAD_OBS_CLUSTER_RADIUS_M = 0.16

# v5: 교차각 부족을 줄이기 위해 다음 관측 pose는 go_to waypoint가 아니라
# set_velocity(vy=...) 기반의 "진짜 옆걸음"을 우선 사용한다.
# 거리 label이 멀수록 더 큰 side-step baseline을 둔다.
PAD_VIEWPOINT_SIDE_BY_DISTANCE_M = {
    "near": 0.45,
    "medium": 0.75,
    "far": 1.05,
    "unknown": 0.70,
}
PAD_VIEWPOINT_EXTRA_SIDE_MULTIPLIER = 1.35
PAD_VIEWPOINT_FORWARD_M = 0.00
PAD_VIEWPOINT_MIN_MOVE_M = 0.30
PAD_VIEWPOINT_MAX_DISTANCE_M = 1.45
PAD_VIEWPOINT_MIN_CROSS_TARGET_DEG = 10.0
PAD_APPROACH_MAX_DISTANCE_M = 5.5

# true-strafe baseline: body yaw를 거의 유지한 채 vy로 좌/우 옆걸음한다.
# 목표는 target을 향해 전진하지 않고 관측 위치만 가로로 벌려 교차각을 키우는 것이다.
PAD_TRUE_STRAFE_VY = 0.34
PAD_TRUE_STRAFE_VX = 0.00
PAD_TRUE_STRAFE_WZ = 0.00
PAD_TRUE_STRAFE_MIN_DURATION_S = 0.85
PAD_TRUE_STRAFE_MAX_DURATION_S = 3.40
PAD_TRUE_STRAFE_SETTLE_S = 0.35
PAD_TRUE_STRAFE_MIN_MOVE_M = 0.22
PAD_TRUE_STRAFE_MIN_CROSS_DEG = 8.0
PAD_TRUE_STRAFE_MAX_TRIES = 2

# vy 옆걸음이 막히거나 거의 움직이지 않을 때만 쓰는 마지막 go_to fallback.
PAD_FALLBACK_SHIFT_VY = 0.28
PAD_FALLBACK_SHIFT_VX = 0.03
PAD_FALLBACK_SHIFT_DURATION_S = 1.20
PAD_RAY_FALLBACK_MIN_OBS = 6
PAD_RAY_FALLBACK_MAX_OFFSET_DEG = 12.0

# v4: 8방향 스캔은 좌표 계산에 넣지 않고 "방향 힌트"로만 쓴다.
# 삼각측량 입력은 통제된 lateral observation waypoint에서 얻은 head-scan 관측만 사용한다.
BODY_PANORAMA_STEP_DEG = 45.0
BODY_PANORAMA_MAX_SECTORS = 8
BODY_PANORAMA_HEAD_YAW = 0.0
BODY_PANORAMA_MIN_RECHECK_OBS = 2
BODY_PANORAMA_HINT_MIN_DELTA_DEG = 10.0
BODY_PANORAMA_TURN_VX = 0.07
BODY_PANORAMA_TURN_WZ = 0.35
BODY_PANORAMA_TURN_SETTLE_S = 0.25


def _clamp(value: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, float(value)))


def _normalized_offset_deg_from_reading(reading: dict[str, Any]) -> float:
    """VLM offset 부호/크기 오염을 줄인 camera-center 기준 offset(deg).

    Prompt는 left=+, right=-로 요청하지만 실제 로그에서 `pos=left, offset=-30`처럼
    충돌하는 응답이 나왔다. v2에서는 target_position을 1차 신뢰하고, 숫자 offset은
    magnitude 보정용으로만 사용한다.
    """
    pos = _normalize_vlm_position(reading.get("target_position", "unknown"))
    raw = _coerce_float(reading.get("target_offset_deg"))

    if pos == "center":
        if raw is None:
            return 0.0
        return _clamp(raw, -8.0, 8.0)

    if pos == "left":
        mag = abs(raw) if raw is not None else TEXT_SIGN_POSITION_OFFSET_DEG["left"]
        return _clamp(mag, 8.0, 35.0)

    if pos == "right":
        mag = abs(raw) if raw is not None else abs(TEXT_SIGN_POSITION_OFFSET_DEG["right"])
        return -_clamp(mag, 8.0, 35.0)

    if raw is None:
        return 0.0
    return _clamp(raw, -30.0, 30.0)


# 기존 _bearing_from_status_and_image를 v2용으로 override한다.
def _bearing_from_status_and_image(robot_status: Any, *, head_yaw_rad: float, reading: dict[str, Any]) -> float | None:
    """robot yaw + head yaw + robust image offset으로 target sign의 world bearing(rad)을 만든다."""
    robot_yaw_deg = robot_yaw_deg_from_status(robot_status)
    if robot_yaw_deg is None:
        return None
    offset_deg = _normalized_offset_deg_from_reading(reading)
    return math.radians(robot_yaw_deg + math.degrees(float(head_yaw_rad)) + offset_deg)


def _observation_quality(obs: dict[str, Any]) -> float:
    """center/near에 가까운 관측을 더 신뢰한다."""
    reading = obs.get("reading", {}) or {}
    pos = reading.get("target_position", "unknown")
    dist = reading.get("target_distance", "unknown")
    offset = abs(_normalized_offset_deg_from_reading(reading))
    score = 1.0
    score += {"center": 3.5, "left": 1.5, "right": 1.5, "unknown": 0.0}.get(pos, 0.0)
    score += {"near": 2.0, "medium": 1.3, "far": 0.6, "unknown": 0.2}.get(dist, 0.0)
    score += max(0.0, 2.0 - offset / 12.0)
    return score


def _compact_observation_for_memory(obs: dict[str, Any]) -> dict[str, Any]:
    """삼각측량에 필요한 최소 정보만 memory에 저장한다."""
    reading = obs.get("reading") or {}
    robust_offset = _normalized_offset_deg_from_reading(reading)
    return {
        "label": str(obs.get("label", "obs")),
        "xy": (float(obs["xy"][0]), float(obs["xy"][1])),
        "bearing_rad": float(obs["bearing_rad"]),
        "bearing_deg": float(obs.get("bearing_deg", math.degrees(float(obs["bearing_rad"])))),
        "head_yaw_rad": float(obs.get("head_yaw_rad", 0.0)),
        "quality": float(_observation_quality(obs)),
        "reading": {
            "visible_letters": list(reading.get("visible_letters", []) or []),
            "target_visible": bool(reading.get("target_visible", False)),
            "target_position": str(reading.get("target_position", "unknown")),
            "target_distance": str(reading.get("target_distance", "unknown")),
            "target_offset_deg": robust_offset,
            "raw_target_offset_deg": reading.get("target_offset_deg"),
        },
    }


def _remember_pad_observation(memory: AgentMemory, pad_name: str, obs: dict[str, Any] | None) -> None:
    if obs is None:
        return
    pad = pad_name.upper()
    stored = _compact_observation_for_memory(obs)
    bucket = memory.pad_observations.setdefault(pad, [])

    # 같은 pose cluster 안에서는 품질이 높은 관측만 남긴다. 같은 위치에서 head만 바꾼 5개 ray가
    # triangulation을 오염시키지 않도록 하는 핵심 보정이다.
    for idx, old in enumerate(bucket):
        old_xy = old.get("xy")
        if old_xy is None:
            continue
        if (xy_distance(tuple(old_xy), tuple(stored["xy"])) or 999.0) < PAD_OBS_CLUSTER_RADIUS_M:
            if float(stored.get("quality", 0.0)) >= float(old.get("quality", 0.0)):
                bucket[idx] = stored
                print(
                    f"[pad-obs] {pad} 같은 pose 관측 갱신: xy={stored['xy']}, "
                    f"bearing={stored['bearing_deg']:.1f}°, pos={stored['reading'].get('target_position')}, "
                    f"dist={stored['reading'].get('target_distance')}, quality={stored.get('quality'):.1f}"
                )
            return

    bucket.append(stored)
    if len(bucket) > PAD_OBS_MAX_PER_PAD:
        del bucket[:-PAD_OBS_MAX_PER_PAD]
    print(
        f"[pad-obs] {pad} 관측 저장 #{len(bucket)}: xy={stored['xy']}, "
        f"bearing={stored['bearing_deg']:.1f}°, pos={stored['reading'].get('target_position')}, "
        f"dist={stored['reading'].get('target_distance')}, quality={stored.get('quality'):.1f}"
    )


def _remember_pad_direction_hint(
    memory: AgentMemory,
    pad_name: str,
    robot_status: Any | None,
    *,
    source: str,
    reading: dict[str, Any] | None = None,
) -> None:
    """target sign이 보였던 body yaw를 다음 탐색용 hint로 저장한다."""
    pad = pad_name.upper()
    yaw = robot_yaw_deg_from_status(robot_status) if robot_status is not None else None
    if yaw is None:
        return
    memory.pad_direction_hints[pad] = _normalize_deg(float(yaw))
    pos = (reading or {}).get("target_position") if isinstance(reading, dict) else None
    dist = (reading or {}).get("target_distance") if isinstance(reading, dict) else None
    print(f"[pad-hint] {pad} body_yaw_hint={memory.pad_direction_hints[pad]:+.1f}° 저장(source={source}, pos={pos}, dist={dist})")


async def _turn_body_toward_direction_hint(ctx: Any, memory: AgentMemory, pad_name: str) -> bool:
    """이전에 target을 본 body yaw가 있으면 그 방향부터 먼저 보도록 회전한다."""
    pad = pad_name.upper()
    hint = memory.pad_direction_hints.get(pad)
    if hint is None:
        return False
    status = await get_robot_status(ctx)
    current_yaw = robot_yaw_deg_from_status(status)
    if current_yaw is None:
        return False
    delta = _signed_delta_deg(float(hint), float(current_yaw))
    if abs(delta) < BODY_PANORAMA_HINT_MIN_DELTA_DEG:
        print(f"[pad-hint] {pad} hint yaw 이미 근접(current={current_yaw:+.1f}°, hint={hint:+.1f}°)")
        return True
    print(f"[pad-hint] {pad} 이전 발견 yaw={hint:+.1f}° 우선 확인 → body turn {delta:+.1f}°")
    await _turn_body_degrees(ctx, delta, reason=f"{pad}-direction-hint-scan")
    return True


def _geometry_observations_from_memory(memory: AgentMemory, pad_name: str) -> list[dict[str, Any]]:
    observations: list[dict[str, Any]] = []
    for item in memory.pad_observations.get(pad_name.upper(), []):
        try:
            xy = item.get("xy")
            if xy is None:
                continue
            observations.append({
                "label": str(item.get("label", "mem")),
                "xy": (float(xy[0]), float(xy[1])),
                "bearing_rad": float(item["bearing_rad"]),
                "bearing_deg": float(item.get("bearing_deg", math.degrees(float(item["bearing_rad"])))),
                "head_yaw_rad": float(item.get("head_yaw_rad", 0.0)),
                "quality": float(item.get("quality", 1.0)),
                "reading": item.get("reading", {}),
            })
        except Exception:
            continue
    return observations


def _select_diverse_observations(observations: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """pose가 거의 같은 관측은 최고 품질 1개만 사용한다."""
    clusters: list[list[dict[str, Any]]] = []
    for obs in observations:
        placed = False
        for cluster in clusters:
            if (xy_distance(obs["xy"], cluster[0]["xy"]) or 999.0) < PAD_OBS_CLUSTER_RADIUS_M:
                cluster.append(obs)
                placed = True
                break
        if not placed:
            clusters.append([obs])
    chosen = [max(cluster, key=_observation_quality) for cluster in clusters]
    chosen.sort(key=lambda o: str(o.get("label", "")))
    return chosen


def _max_baseline(observations: list[dict[str, Any]]) -> float:
    best = 0.0
    for i in range(len(observations)):
        for j in range(i + 1, len(observations)):
            best = max(best, xy_distance(observations[i]["xy"], observations[j]["xy"]) or 0.0)
    return best


def _least_squares_ray_intersection(observations: list[dict[str, Any]]) -> tuple[float, float] | None:
    """여러 bearing ray의 수직거리 제곱합을 최소화하는 좌표를 계산한다."""
    if len(observations) < 2:
        return None
    a00 = a01 = a11 = b0 = b1 = 0.0
    for obs in observations:
        theta = float(obs["bearing_rad"])
        # ray 방향 d=(cos,sin)의 법선 n=(-sin,cos). 점 x가 ray line 위면 n·x=n·p.
        nx, ny = -math.sin(theta), math.cos(theta)
        px, py = obs["xy"]
        w = max(0.5, min(4.0, float(obs.get("quality", _observation_quality(obs)))))
        a00 += w * nx * nx
        a01 += w * nx * ny
        a11 += w * ny * ny
        rhs = nx * px + ny * py
        b0 += w * nx * rhs
        b1 += w * ny * rhs
    det = a00 * a11 - a01 * a01
    if abs(det) < 1e-5:
        return None
    x = (b0 * a11 - b1 * a01) / det
    y = (a00 * b1 - a01 * b0) / det
    return (x, y)


# 기존 _triangulate_sign_xy를 v2용 weighted multi-view 함수로 override한다.
def _triangulate_sign_xy(observations: list[dict[str, Any]]) -> tuple[float, float] | None:
    diverse = _select_diverse_observations(observations)
    if len(diverse) < PAD_OBS_MIN_COUNT_FOR_TRIANGULATION:
        print(f"[triangulate-v2] 서로 다른 pose 관측 부족: {len(diverse)}개")
        return None

    baseline = _max_baseline(diverse)
    if baseline < TRIANGULATION_MIN_BASELINE_M:
        print(f"[triangulate-v2] baseline 부족: {baseline:.2f}m < {TRIANGULATION_MIN_BASELINE_M:.2f}m")
        return None

    # ray 교차각이 너무 작으면 좌표가 멀리 튀므로 최소 교차각 확인.
    max_cross = 0.0
    for i in range(len(diverse)):
        for j in range(i + 1, len(diverse)):
            max_cross = max(max_cross, _angle_between_bearings_deg(diverse[i]["bearing_rad"], diverse[j]["bearing_rad"]))
    if max_cross < TRIANGULATION_MIN_CROSS_ANGLE_DEG:
        print(f"[triangulate-v2] 교차각 부족: max_cross={max_cross:.1f}°")
        return None

    point = _least_squares_ray_intersection(diverse)
    if point is None:
        return None

    residuals: list[float] = []
    front_count = 0
    distances: list[float] = []
    for obs in diverse:
        px, py = obs["xy"]
        theta = float(obs["bearing_rad"])
        dx, dy = math.cos(theta), math.sin(theta)
        vx, vy = point[0] - px, point[1] - py
        forward = vx * dx + vy * dy
        lateral = abs(vx * (-dy) + vy * dx)
        distances.append(math.hypot(vx, vy))
        residuals.append(lateral)
        if forward > 0.05:
            front_count += 1

    residuals_sorted = sorted(residuals)
    median_residual = residuals_sorted[len(residuals_sorted)//2]
    max_dist = max(distances) if distances else 999.0
    if front_count < max(1, len(diverse) - 1):
        print(f"[triangulate-v2] 교차점이 ray 뒤쪽에 있음: front={front_count}/{len(diverse)}, point={point}")
        return None
    if max_dist > TRIANGULATION_MAX_SIGN_DISTANCE_M:
        print(f"[triangulate-v2] sign 추정 거리가 너무 멂: max_dist={max_dist:.2f}m, point={point}")
        return None
    if median_residual > 0.75:
        print(f"[triangulate-v2] VLM ray 잔차가 큼: median_residual={median_residual:.2f}m, point={point}")
        return None

    labels = [str(o.get("label", "obs")) for o in diverse]
    print(
        f"[triangulate-v2] pose={len(diverse)}개, baseline={baseline:.2f}m, "
        f"max_cross={max_cross:.1f}°, residual_med={median_residual:.2f}m, sign_xy={point}, labels={labels}"
    )
    return point




def _pad_obs_distance_label(obs: dict[str, Any] | None) -> str:
    reading = (obs or {}).get("reading", {}) or {}
    return _normalize_vlm_distance(reading.get("target_distance", "unknown"))


def _current_label_from_observations(
    observations: list[dict[str, Any]],
    current_xy: tuple[float, float] | None,
) -> str:
    """현재 위치와 가장 가까운 VLM 관측의 distance label을 반환한다."""
    if current_xy is None or not observations:
        return "unknown"
    nearby: list[tuple[float, dict[str, Any]]] = []
    for obs in observations:
        d = xy_distance(current_xy, obs.get("xy"))
        if d is None:
            continue
        nearby.append((float(d), obs))
    if not nearby:
        return "unknown"
    nearby.sort(key=lambda x: x[0])
    if nearby[0][0] > PAD_SIGN_SINGLE_RAY_MAX_OBS_DIST_M:
        return "unknown"
    return _pad_obs_distance_label(nearby[0][1])


def _validate_sign_xy_distance_consistency(
    pad: str,
    sign_xy: tuple[float, float],
    observations: list[dict[str, Any]],
) -> bool:
    """Triangulated sign_xy가 VLM distance label과 물리적으로 맞는지 검사한다.

    예: VLM이 medium이라고 읽은 관측 pose에서 sign_xy가 0.12m 앞에 있다면,
    sign_xy는 실제 C/D sign이 아니라 ray가 잘못 만난 가짜 교차점일 가능성이 높다.
    이 경우 place 단계로 넘기면 같은 좌표에서 무한 실패하므로 즉시 폐기한다.
    """
    conflicts: list[str] = []
    checked = 0
    for obs in observations:
        label = _pad_obs_distance_label(obs)
        if label == "unknown":
            continue
        dist = xy_distance(obs.get("xy"), sign_xy)
        if dist is None:
            continue
        checked += 1
        lo, hi = PAD_SIGN_DISTANCE_COMPAT_BOUNDS.get(label, PAD_SIGN_DISTANCE_COMPAT_BOUNDS["unknown"])
        if dist < lo:
            conflicts.append(f"{obs.get('label','obs')}:{label}인데 sign_dist={dist:.2f}m < {lo:.2f}m")
        elif dist > hi:
            conflicts.append(f"{obs.get('label','obs')}:{label}인데 sign_dist={dist:.2f}m > {hi:.2f}m")
    if conflicts:
        preview = "; ".join(conflicts[:3])
        if len(conflicts) > 3:
            preview += f"; ...(+{len(conflicts)-3})"
        print(f"[fast-triangulate-guard] {pad}: sign_xy와 VLM 거리 label 충돌 → triangulation 폐기: {preview}")
        return False
    if checked > 0:
        print(f"[fast-triangulate-guard] {pad}: sign_xy 거리-label 검증 통과({checked} obs)")
    return True


def _single_ray_progress_xy_from_recent_observation(
    memory: AgentMemory,
    pad: str,
    current_xy: tuple[float, float] | None,
    observations: list[dict[str, Any]],
    *,
    reason: str,
) -> tuple[float, float] | None:
    """현재 위치 근처에서 target sign을 본 최신/고품질 ray 방향으로 짧게 전진한다.

    이 함수는 pad 좌표를 확정하기 위한 함수가 아니다. triangulation이 현재 위치 근처로
    접혔을 때 place를 막고, 다음 VLM 관측이 더 좋은 baseline을 갖도록 만드는 안전 이동이다.
    """
    if current_xy is None or not observations:
        return None

    candidates: list[tuple[float, float, dict[str, Any]]] = []
    for obs in observations:
        d = xy_distance(current_xy, obs.get("xy"))
        if d is None or d > PAD_SIGN_SINGLE_RAY_MAX_OBS_DIST_M:
            continue
        label = _pad_obs_distance_label(obs)
        # 현재 보이는 target sign의 ray만 사용한다. unknown 거리라도 bearing은 보정 이동에 쓸 수 있다.
        q = float(obs.get("quality", _observation_quality(obs)))
        # 가까운 관측일수록, 품질이 높을수록 우선한다.
        score = q - 2.0 * float(d)
        candidates.append((score, float(d), obs))

    if not candidates:
        print(f"[single-ray-progress] {pad}: 현재 위치 근처 target 관측 없음 → 추가 head/body scan 필요(reason={reason})")
        return None

    candidates.sort(key=lambda x: x[0], reverse=True)
    obs = candidates[0][2]
    label = _pad_obs_distance_label(obs)
    move_m = float(PAD_SIGN_SINGLE_RAY_PROGRESS_M.get(label, PAD_SIGN_SINGLE_RAY_PROGRESS_M["unknown"]))
    bearing = float(obs["bearing_rad"])
    target_xy = (
        float(current_xy[0]) + move_m * math.cos(bearing),
        float(current_xy[1]) + move_m * math.sin(bearing),
    )
    target_xy = _x_fence_clamp_xy(float(target_xy[0]), float(target_xy[1]))
    delta = xy_distance(current_xy, target_xy) or 0.0
    if delta < 0.05:
        return None
    if not _is_safe_destination_candidate(memory, pad, target_xy, source=f"single-ray-progress-{reason}"):
        return None
    print(
        f"[single-ray-progress] {pad}: triangulation={reason} → place 금지, "
        f"recent_obs={obs.get('label')}, label={label}, move={move_m:.2f}m, "
        f"target_xy={target_xy}"
    )
    return target_xy

def _estimate_approach_xy_from_memory(
    memory: AgentMemory,
    pad_name: str,
    current_xy: tuple[float, float] | None,
) -> tuple[float, float] | None:
    """누적된 sign bearing ray로 approach_xy를 계산한다."""
    observations = _geometry_observations_from_memory(memory, pad_name)
    if len(observations) < PAD_OBS_MIN_COUNT_FOR_TRIANGULATION:
        return None

    sign_xy = _triangulate_sign_xy(observations)
    if sign_xy is not None:
        approach_xy = _approach_xy_from_sign_xy(current_xy, sign_xy, standoff_m=TRIANGULATION_APPROACH_STANDOFF_M)
        if approach_xy is None:
            return None
        dist_to_approach = xy_distance(current_xy, approach_xy)
        if dist_to_approach is not None and dist_to_approach > PAD_APPROACH_MAX_DISTANCE_M:
            print(f"[triangulate-v2] approach_xy가 너무 멂({dist_to_approach:.2f}m) → 폐기: {approach_xy}")
            return None
        if _is_safe_destination_candidate(memory, pad_name, approach_xy, source="memory-triangulation-v2"):
            print(f"[triangulate-v2] 누적 관측 기반 {pad_name} approach_xy={approach_xy}")
            return approach_xy

    # fallback: 관측 pose가 충분히 쌓였는데 VLM offset 노이즈 때문에 LS가 실패하면,
    # 가장 center에 가까운 single ray + VLM distance로 임시 go_to 좌표를 만든다.
    # 이 좌표는 place 성공 시 확정 좌표로 덮어쓰기 전까지 rough waypoint로 취급한다.
    diverse = _select_diverse_observations(observations)
    if len(observations) >= PAD_RAY_FALLBACK_MIN_OBS and current_xy is not None:
        best = max(observations, key=_observation_quality)
        offset = abs(_normalized_offset_deg_from_reading(best.get("reading", {})))
        if offset <= PAD_RAY_FALLBACK_MAX_OFFSET_DEG:
            fallback_xy = _single_ray_fallback_xy(best, current_xy)
            if fallback_xy is not None and _is_safe_destination_candidate(memory, pad_name, fallback_xy, source="single-ray-distance-v2"):
                print(
                    f"[triangulate-v2] 삼각측량 불안정 → center/거리 기반 fallback approach_xy={fallback_xy} "
                    f"(obs={len(observations)}, diverse={len(diverse)}, offset={offset:.1f}°)"
                )
                return fallback_xy
    return None


def _best_recent_pad_observation(memory: AgentMemory, pad_name: str) -> dict[str, Any] | None:
    obs = _geometry_observations_from_memory(memory, pad_name)
    if not obs:
        return None
    return max(obs, key=_observation_quality)


def _distance_label_to_viewpoint_side_m(reading: dict[str, Any] | None) -> float:
    """VLM distance label에 따라 삼각측량용 측면 baseline 목표를 정한다."""
    dist = _normalize_vlm_distance((reading or {}).get("target_distance", "unknown"))
    return float(PAD_VIEWPOINT_SIDE_BY_DISTANCE_M.get(dist, PAD_VIEWPOINT_SIDE_BY_DISTANCE_M["unknown"]))


def _expected_cross_angle_after_move_deg(obs: dict[str, Any], candidate: tuple[float, float]) -> float | None:
    """후보 위치에서 기존 ray가 얼마나 다른 각도로 보일지 보수적으로 예측한다.

    실제 target 좌표를 모르므로, VLM distance label을 이용한 단일 ray rough point를 target proxy로 쓴다.
    이 값은 필터/정렬용이며 최종 좌표 확정에는 사용하지 않는다.
    """
    reading = obs.get("reading", {}) or {}
    rough_dist = float(TEXT_SIGN_FALLBACK_DISTANCE_M.get(_normalize_vlm_distance(reading.get("target_distance", "unknown")), 1.7))
    ox, oy = obs["xy"]
    theta = float(obs["bearing_rad"])
    proxy = (ox + rough_dist * math.cos(theta), oy + rough_dist * math.sin(theta))
    cand_bearing = math.atan2(proxy[1] - candidate[1], proxy[0] - candidate[0])
    return _angle_between_bearings_deg(theta, cand_bearing)


def _body_left_unit_from_yaw_deg(yaw_deg: float | None) -> tuple[float, float] | None:
    """현재 body yaw에서 robot-frame +vy(왼쪽)가 가리키는 world 단위 벡터."""
    if yaw_deg is None:
        return None
    theta = math.radians(float(yaw_deg))
    return (-math.sin(theta), math.cos(theta))


def _true_strafe_plan_from_observation(
    current_xy: tuple[float, float],
    yaw_deg: float | None,
    obs: dict[str, Any],
    *,
    attempt: int,
) -> list[dict[str, Any]]:
    """set_velocity(vy)로 실행할 true lateral baseline 후보를 만든다.

    +vy/-vy 두 방향 중, 기존 관측 ray와의 예상 교차각이 큰 쪽을 먼저 사용한다.
    go_to와 달리 이동 경로를 planner에게 맡기지 않고, 몸 방향을 유지한 상태에서 camera pose만
    옆으로 벌리는 것이 목적이다.
    """
    left = _body_left_unit_from_yaw_deg(yaw_deg)
    if left is None:
        return []
    base_side = _distance_label_to_viewpoint_side_m(obs.get("reading", {}))
    # attempt가 반복되면 조금 더 큰 baseline을 준다.
    distance = min(PAD_VIEWPOINT_MAX_DISTANCE_M, base_side * (1.0 if attempt <= 1 else PAD_VIEWPOINT_EXTRA_SIDE_MULTIPLIER))
    first_sign = 1.0 if attempt % 2 == 1 else -1.0
    plans: list[dict[str, Any]] = []
    for sign in (first_sign, -first_sign):
        predicted_xy = (current_xy[0] + sign * distance * left[0], current_xy[1] + sign * distance * left[1])
        expected_cross = _expected_cross_angle_after_move_deg(obs, predicted_xy) or 0.0
        duration = _clamp(distance / max(abs(PAD_TRUE_STRAFE_VY), 1e-6), PAD_TRUE_STRAFE_MIN_DURATION_S, PAD_TRUE_STRAFE_MAX_DURATION_S)
        plans.append({
            "vy_sign": sign,
            "distance": distance,
            "duration": duration,
            "predicted_xy": predicted_xy,
            "expected_cross": expected_cross,
        })
    plans.sort(key=lambda p: p["expected_cross"], reverse=True)
    for p in plans:
        print(
            f"[pad-obs] true-strafe 후보: vy_sign={p['vy_sign']:+.0f}, "
            f"distance≈{p['distance']:.2f}m, duration={p['duration']:.2f}s, "
            f"expected_cross≈{p['expected_cross']:.1f}°, predicted_xy={p['predicted_xy']}"
        )
    return plans


def _local_viewpoint_candidates_from_observation(
    current_xy: tuple[float, float],
    obs: dict[str, Any],
    *,
    attempt: int,
) -> list[tuple[float, float]]:
    """현재 pose와 target ray로부터 다른 각도 촬영용 측면 observation waypoint 후보를 만든다.

    v4 핵심:
    - target 방향으로 전진하지 않고 ray에 수직인 방향으로 이동해 교차각을 키운다.
    - near/medium/far에 따라 측면 이동 거리를 다르게 잡는다.
    - 한쪽이 막히거나 교차각 기대값이 작으면 반대쪽/더 먼 측면 후보를 시도한다.
    """
    theta = float(obs["bearing_rad"])
    dx, dy = math.cos(theta), math.sin(theta)
    px, py = -dy, dx  # ray에 수직인 world 방향
    base_side = _distance_label_to_viewpoint_side_m(obs.get("reading", {}))
    first_side = 1.0 if attempt % 2 == 1 else -1.0

    raw_candidates: list[tuple[float, float, float, float]] = []
    for multiplier in (1.0, PAD_VIEWPOINT_EXTRA_SIDE_MULTIPLIER):
        side_distance = base_side * multiplier
        for side in (first_side, -first_side):
            x = current_xy[0] + side * side_distance * px + PAD_VIEWPOINT_FORWARD_M * dx
            y = current_xy[1] + side * side_distance * py + PAD_VIEWPOINT_FORWARD_M * dy
            move_dist = xy_distance(current_xy, (x, y)) or 0.0
            if move_dist > PAD_VIEWPOINT_MAX_DISTANCE_M:
                continue
            expected_cross = _expected_cross_angle_after_move_deg(obs, (x, y)) or 0.0
            raw_candidates.append((expected_cross, move_dist, x, y))

    # 교차각 기대값이 큰 후보를 먼저 사용한다. 모두 작아도 가장 나은 후보부터 시도한다.
    raw_candidates.sort(key=lambda t: (t[0], t[1]), reverse=True)
    candidates: list[tuple[float, float]] = []
    for expected_cross, move_dist, x, y in raw_candidates:
        if (x, y) not in candidates:
            print(
                f"[pad-obs] lateral 후보 예상: move={move_dist:.2f}m, "
                f"expected_cross≈{expected_cross:.1f}°, xy=({x:.3f}, {y:.3f})"
            )
            candidates.append((x, y))
    return candidates


async def _small_observation_baseline_shift(ctx: Any, *, step: int, reason: str) -> bool:
    """go_to observation viewpoint가 실패했을 때만 쓰는 짧은 옆이동 fallback.

    워크숍 튜토리얼의 set_velocity 제약(|vx|, |vy| ≤ 1.5)을 이용한다.
    이 동작은 접근/배달이 아니라 삼각측량 baseline을 만들기 위한 pose 변경이다.
    """
    side = 1.0 if step % 2 == 1 else -1.0
    before = await get_robot_status(ctx)
    xy_before = robot_xy_from_status(before)
    yaw_before = robot_yaw_deg_from_status(before)
    print(
        f"[pad-obs] fallback lateral-shift({reason}): vx={PAD_FALLBACK_SHIFT_VX:.2f}, "
        f"vy={side * PAD_FALLBACK_SHIFT_VY:+.2f}, duration={PAD_FALLBACK_SHIFT_DURATION_S:.2f}s, "
        f"xy_before={xy_before}, yaw_before={yaw_before}"
    )
    result = await move_velocity(
        ctx,
        vx=PAD_FALLBACK_SHIFT_VX,
        vy=side * PAD_FALLBACK_SHIFT_VY,
        wz=0.0,
        duration_s=PAD_FALLBACK_SHIFT_DURATION_S,
    )
    await asyncio.sleep(0.30)
    after = await get_robot_status(ctx)
    xy_after = robot_xy_from_status(after)
    moved = xy_distance(xy_before, xy_after) if (xy_before is not None and xy_after is not None) else None
    print(f"[pad-obs] fallback lateral-shift done: xy_after={xy_after}, moved={moved if moved is not None else 'unknown'}, result={result_summary(result)}")
    return not (isinstance(result, dict) and result.get("status") == "failed") and (moved is None or moved >= 0.10)


async def _move_to_observation_viewpoint(ctx: Any, memory: AgentMemory, pad_name: str, *, attempt: int) -> bool:
    """삼각측량 baseline 확보를 위해 true vy 옆걸음으로 관측 위치를 바꾼다.

    v4의 go_to lateral waypoint는 planner가 경로를 바꾸면서 실제 target 기준 옆방향이 깨질 수 있었다.
    v5는 set_velocity(vy=...)를 1순위로 사용해 body yaw를 거의 유지한 채 카메라 위치만
    좌/우로 벌린다. 이때 실제 이동 후 기존 ray와의 expected cross angle을 다시 확인한다.
    """
    status = await get_robot_status(ctx)
    current_xy = robot_xy_from_status(status)
    yaw_before = robot_yaw_deg_from_status(status)
    if current_xy is None:
        return False
    obs = _best_recent_pad_observation(memory, pad_name)
    if obs is None:
        return False

    plans = _true_strafe_plan_from_observation(current_xy, yaw_before, obs, attempt=attempt)
    tried = 0
    for plan in plans:
        if tried >= PAD_TRUE_STRAFE_MAX_TRIES:
            break
        tried += 1
        vy = float(plan["vy_sign"]) * PAD_TRUE_STRAFE_VY
        print(
            f"[pad-obs] true-strafe baseline 시작: pad={pad_name}, try={tried}, "
            f"vx={PAD_TRUE_STRAFE_VX:.2f}, vy={vy:+.2f}, wz={PAD_TRUE_STRAFE_WZ:+.2f}, "
            f"duration={plan['duration']:.2f}s, xy_before={current_xy}, yaw_before={yaw_before}"
        )
        result = await move_velocity(
            ctx,
            vx=PAD_TRUE_STRAFE_VX,
            vy=vy,
            wz=PAD_TRUE_STRAFE_WZ,
            duration_s=float(plan["duration"]),
        )
        await asyncio.sleep(PAD_TRUE_STRAFE_SETTLE_S)
        after = await get_robot_status(ctx)
        after_xy = robot_xy_from_status(after)
        yaw_after = robot_yaw_deg_from_status(after)
        moved = xy_distance(current_xy, after_xy) if after_xy is not None else None
        actual_cross = _expected_cross_angle_after_move_deg(obs, after_xy) if after_xy is not None else None
        yaw_delta = _signed_delta_deg(yaw_after, yaw_before) if yaw_before is not None and yaw_after is not None else None
        print(
            f"[pad-obs] true-strafe baseline done: xy_after={after_xy}, moved={moved if moved is not None else 'unknown'}, "
            f"actual_cross≈{actual_cross if actual_cross is not None else 'unknown'}°, yaw_delta={yaw_delta}, result={result_summary(result)}"
        )
        if isinstance(result, dict) and result.get("status") == "failed":
            continue
        if moved is not None and moved < PAD_TRUE_STRAFE_MIN_MOVE_M:
            print(f"[pad-obs] true-strafe 이동량 부족({moved:.2f}m) → 반대 방향/다음 후보 시도")
            continue
        if actual_cross is not None and actual_cross < PAD_TRUE_STRAFE_MIN_CROSS_DEG:
            print(f"[pad-obs] true-strafe 교차각 예상 부족({actual_cross:.1f}°) → 반대 방향/다음 후보 시도")
            continue
        return True

    # true-strafe가 막힌 경우에만 go_to lateral waypoint를 마지막 fallback으로 사용한다.
    print(f"[pad-obs] true-strafe 후보 실패 → go_to lateral waypoint fallback 시도")
    status = await get_robot_status(ctx)
    current_xy = robot_xy_from_status(status)
    if current_xy is not None:
        for candidate in _local_viewpoint_candidates_from_observation(current_xy, obs, attempt=attempt):
            print(f"[pad-obs] fallback lateral go_to 후보: {pad_name} candidate={candidate}, from={current_xy}")
            result = await go_to_xy(ctx, *candidate)
            await asyncio.sleep(0.35)
            after = await get_robot_status(ctx)
            after_xy = robot_xy_from_status(after)
            moved = xy_distance(current_xy, after_xy) if after_xy is not None else None
            actual_cross = _expected_cross_angle_after_move_deg(obs, after_xy) if after_xy is not None else None
            print(
                f"[pad-obs] fallback lateral go_to done: xy_after={after_xy}, moved={moved if moved is not None else 'unknown'}, "
                f"actual_cross≈{actual_cross if actual_cross is not None else 'unknown'}°, result={result_summary(result)}"
            )
            if not (isinstance(result, dict) and result.get("status") == "failed") and moved is not None and moved >= PAD_VIEWPOINT_MIN_MOVE_M:
                return True

    print(f"[pad-obs] true-strafe/go_to 모두 실패 → 짧은 vy fallback")
    return await _small_observation_baseline_shift(ctx, step=attempt, reason=f"{pad_name}-true-strafe-last-resort")



async def _turn_body_for_hint_panorama(ctx: Any, turn_deg: float, *, reason: str) -> bool:
    """8방향 힌트 스캔 전용 저속 arc turn.

    v3의 panorama는 vx=0.25라 sector마다 로봇이 크게 이동했고, 그 관측을 좌표 계산에 넣어
    ray 뒤쪽/교차각 부족 문제가 커졌다. v4에서는 panorama를 hint-only로 제한하고,
    이동량도 줄이기 위해 작은 vx로만 몸 방향을 바꾼다.
    """
    if abs(turn_deg) < 4.0:
        return True
    sign = 1.0 if turn_deg > 0 else -1.0
    duration = _duration_for_arc_turn(turn_deg, BODY_PANORAMA_TURN_WZ)
    before = await get_robot_status(ctx)
    yaw_before = robot_yaw_deg_from_status(before)
    xy_before = robot_xy_from_status(before)
    print(
        f"[panorama-v4] hint turn({reason}): request={turn_deg:+.1f}°, "
        f"vx={BODY_PANORAMA_TURN_VX:.2f}, wz={sign * BODY_PANORAMA_TURN_WZ:+.2f}, "
        f"duration={duration:.2f}s, yaw_before={yaw_before}, xy_before={xy_before}"
    )
    result = await move_velocity(ctx, vx=BODY_PANORAMA_TURN_VX, vy=0.0, wz=sign * BODY_PANORAMA_TURN_WZ, duration_s=duration)
    await asyncio.sleep(BODY_PANORAMA_TURN_SETTLE_S)
    after = await get_robot_status(ctx)
    yaw_after = robot_yaw_deg_from_status(after)
    xy_after = robot_xy_from_status(after)
    yaw_delta = _signed_delta_deg(yaw_after, yaw_before) if yaw_before is not None and yaw_after is not None else None
    moved = xy_distance(xy_before, xy_after) if xy_before is not None and xy_after is not None else None
    print(
        f"[panorama-v4] hint turn done: yaw_after={yaw_after}, yaw_delta={yaw_delta}, "
        f"xy_after={xy_after}, moved={moved if moved is not None else 'unknown'}, result={result_summary(result)}"
    )
    return not (isinstance(result, dict) and result.get("status") == "failed")


async def text_forward_search(
    ctx: Any,
    memory: AgentMemory,
    expected_pad: str,
    *,
    scan_directions: int = SEARCH_SWEEP_DIRECTIONS,
    forward_steps: int = TEXT_SEARCH_FORWARD_STEPS,
) -> dict[str, Any]:
    """VLM sign 탐색 + hint-only 8방향 panorama + true-strafe multiview triangulation.

    v5 원칙:
      1. target이 현재 시야/head sweep에서 바로 보이면 8방향 전체 스캔은 하지 않는다.
      2. target이 안 보이거나 삼각측량이 미확정이면 8방향 body panorama로 확장한다.
      3. panorama 중 target을 찾으면 즉시 중단하되, 관측 ray는 저장하지 않고 body yaw hint만 저장한다.
      4. 실제 triangulation 관측은 true vy 옆걸음으로 만든 lateral pose에서 저장한다.
      5. 같은 패드를 다시 찾을 때는 hint yaw 방향부터 먼저 본다.
    """
    expected_pad = expected_pad.upper()
    print(f"[multiview-search] start target={expected_pad}, version={AGENT_VERSION}")

    vlm_calls = 0
    consecutive_unavailable = 0
    target_visible_count = 0
    panorama_runs = 0

    async def current_xy() -> tuple[float, float] | None:
        return robot_xy_from_status(await get_robot_status(ctx))

    async def read_limited(expected_head_yaw: float) -> dict[str, Any] | None:
        nonlocal vlm_calls, consecutive_unavailable
        if vlm_calls >= MAX_VLM_CALLS_PER_SEARCH:
            return None
        head_result = await set_head(ctx, yaw=expected_head_yaw, pitch=TEXT_SEARCH_PITCH)
        if isinstance(head_result, dict) and head_result.get("status") == "failed":
            return None
        await asyncio.sleep(0.20)
        reading = await read_signs_with_vlm(ctx, expected_pad)
        vlm_calls += 1
        if reading is None:
            consecutive_unavailable += 1
        else:
            consecutive_unavailable = 0
        return reading

    async def record_if_target_visible(reading: dict[str, Any] | None, *, head_yaw: float, label: str, source: str) -> tuple[bool, tuple[float, float] | None]:
        """target 관측 저장 후 현재 memory로 approach_xy를 계산한다."""
        nonlocal target_visible_count
        if reading is None or not reading.get("target_visible"):
            return False, None
        target_visible_count += 1
        status = await get_robot_status(ctx)
        obs = _make_sign_observation(status, head_yaw_rad=head_yaw, reading=reading, label=label)
        _remember_pad_observation(memory, expected_pad, obs)
        _remember_pad_direction_hint(memory, expected_pad, status, source=source, reading=reading)
        approach_xy = _estimate_approach_xy_from_memory(memory, expected_pad, await current_xy())
        return True, approach_xy

    async def body_panorama_scan(trigger_reason: str) -> tuple[bool, tuple[float, float] | None]:
        """body yaw를 45도씩 바꿔 최대 8방향을 촬영한다.

        v4에서는 panorama 관측을 삼각측량 memory에 넣지 않는다.
        panorama는 target 재발견과 body_yaw hint 저장 전용이다.
        """
        nonlocal panorama_runs, consecutive_unavailable, target_visible_count
        panorama_runs += 1
        print(f"[panorama-v4] {expected_pad} 8방향 hint-only 스캔 시작(trigger={trigger_reason}, run={panorama_runs})")
        await set_head(ctx, yaw=BODY_PANORAMA_HEAD_YAW, pitch=TEXT_SEARCH_PITCH)
        await asyncio.sleep(0.15)

        for sector in range(1, BODY_PANORAMA_MAX_SECTORS + 1):
            if vlm_calls >= MAX_VLM_CALLS_PER_SEARCH:
                print(f"[panorama-v4] VLM 호출 한계 도달({vlm_calls}/{MAX_VLM_CALLS_PER_SEARCH})")
                break

            reading = await read_limited(BODY_PANORAMA_HEAD_YAW)
            if reading is None:
                print(f"[panorama-v4] sector={sector}/{BODY_PANORAMA_MAX_SECTORS}, target={expected_pad}, VLM unavailable {consecutive_unavailable}/2")
                if consecutive_unavailable >= 2:
                    return False, None
            else:
                status = await get_robot_status(ctx)
                yaw = robot_yaw_deg_from_status(status)
                print(
                    f"[panorama-v4] sector={sector}/{BODY_PANORAMA_MAX_SECTORS}, yaw={yaw if yaw is not None else 'unknown'}, "
                    f"call={vlm_calls}/{MAX_VLM_CALLS_PER_SEARCH}, visible={reading.get('visible_letters')}, "
                    f"target={expected_pad}, pos={reading.get('target_position')}, dist={reading.get('target_distance')}, "
                    f"offset={reading.get('target_offset_deg')}"
                )
                if reading.get("target_visible"):
                    target_visible_count += 1
                    _remember_pad_direction_hint(memory, expected_pad, status, source=f"panorama-hint-only-{trigger_reason}-sec{sector}", reading=reading)
                    print(f"[panorama-v4] {expected_pad} 발견 → hint만 저장하고 8방향 스캔 즉시 중단")
                    return True, None

            if sector < BODY_PANORAMA_MAX_SECTORS:
                await _turn_body_for_hint_panorama(ctx, BODY_PANORAMA_STEP_DEG, reason="v4-body-panorama-hint-scan")

        print(f"[panorama-v4] {expected_pad} 전체/허용 범위 스캔 완료, target 미발견")
        return False, None

    # 이전에 같은 pad를 본 방향이 있으면, 전체 탐색 전에 그 방향부터 먼저 본다.
    await _turn_body_toward_direction_hint(ctx, memory, expected_pad)

    for step in range(1, PAD_TRIANGULATION_SEARCH_STEPS + 1):
        print(f"[multiview-search] step={step}/{PAD_TRIANGULATION_SEARCH_STEPS}, stored_obs={len(memory.pad_observations.get(expected_pad, []))}")
        found_this_step = False
        approach_xy_this_step: tuple[float, float] | None = None

        # 1) 기본 탐색: 현재 body heading에서 5방향 head scan.
        for head_yaw in PAD_TRIANGULATION_HEAD_YAWS:
            if vlm_calls >= MAX_VLM_CALLS_PER_SEARCH:
                break
            reading = await read_limited(head_yaw)
            if reading is None:
                print(f"[multiview-search] head={head_yaw:+.2f}, target={expected_pad}, VLM unavailable {consecutive_unavailable}/2")
                if consecutive_unavailable >= 2:
                    await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
                    return {"status": "continue", "target_xy": await current_xy(), "place_ready": False, "reason": "VLM 일시 실패, 다음 cycle 재시도"}
                continue

            print(
                f"[multiview-search] head={head_yaw:+.2f}, call={vlm_calls}/{MAX_VLM_CALLS_PER_SEARCH}, "
                f"visible={reading.get('visible_letters')}, target={expected_pad}, "
                f"pos={reading.get('target_position')}, dist={reading.get('target_distance')}, "
                f"offset={reading.get('target_offset_deg')}"
            )

            found, approach_xy = await record_if_target_visible(
                reading,
                head_yaw=head_yaw,
                label=f"v4_s{step}_h{head_yaw:+.2f}",
                source=f"head-scan-{step}",
            )
            if found:
                found_this_step = True
                if approach_xy is not None:
                    approach_xy_this_step = approach_xy
                    break

        if approach_xy_this_step is not None:
            await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
            print(f"[multiview-search] {expected_pad} 좌표 추정 성공 → go_to용 approach_xy={approach_xy_this_step}")
            return {"status": "found", "target_xy": approach_xy_this_step, "place_ready": False, "reason": "v4_multiview_triangulated_approach_xy"}

        if vlm_calls >= MAX_VLM_CALLS_PER_SEARCH:
            break

        if found_this_step:
            # target은 보였지만 triangulation이 아직 안 됨. 우선 다른 관측 pose로 이동한다.
            moved = await _move_to_observation_viewpoint(ctx, memory, expected_pad, attempt=step)
            if not moved:
                print(f"[multiview-search] observation viewpoint 이동 실패 → 8방향 확장 스캔으로 전환")
                found, approach_xy = await body_panorama_scan("viewpoint-move-failed")
                if approach_xy is not None:
                    await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
                    return {"status": "found", "target_xy": approach_xy, "place_ready": False, "reason": "v4_panorama_after_viewpoint_failed"}
                break

            # 이동 후에도 누적 ray가 미확정이면, 2개 이상 관측이 있을 때만 8방향으로 빠르게 재획득을 시도한다.
            if len(memory.pad_observations.get(expected_pad, [])) >= BODY_PANORAMA_MIN_RECHECK_OBS:
                maybe_xy = _estimate_approach_xy_from_memory(memory, expected_pad, await current_xy())
                if maybe_xy is not None:
                    await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
                    print(f"[multiview-search] {expected_pad} 이동 후 좌표 추정 성공 → approach_xy={maybe_xy}")
                    return {"status": "found", "target_xy": maybe_xy, "place_ready": False, "reason": "v4_multiview_after_viewpoint_move"}
                # 삼각측량 실패가 반복될 가능성이 있으면 8방향 확장 스캔으로 다른 bearing을 확보한다.
                found, approach_xy = await body_panorama_scan("triangulation-unconfirmed")
                if approach_xy is not None:
                    await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
                    print(f"[multiview-search] {expected_pad} panorama 후 좌표 추정 성공 → approach_xy={approach_xy}")
                    return {"status": "found", "target_xy": approach_xy, "place_ready": False, "reason": "v4_panorama_triangulated"}
            continue

        # 2) target이 현재 기본 head scan에서 안 보이면, 8방향 body panorama로 확장한다.
        found, approach_xy = await body_panorama_scan("target-not-visible")
        if approach_xy is not None:
            await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
            print(f"[multiview-search] {expected_pad} panorama 좌표 추정 성공 → approach_xy={approach_xy}")
            return {"status": "found", "target_xy": approach_xy, "place_ready": False, "reason": "v4_panorama_triangulated"}
        if found:
            # target은 찾았지만 아직 pose/ray가 부족함. 다음 step에서 viewpoint 이동이나 추가 관측을 진행한다.
            continue
        break

    await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)

    current = await current_xy()
    approach_xy = _estimate_approach_xy_from_memory(memory, expected_pad, current)
    if approach_xy is not None:
        return {"status": "found", "target_xy": approach_xy, "place_ready": False, "reason": "v4_triangulated_after_loop"}

    obs_count = len(memory.pad_observations.get(expected_pad, []))
    if target_visible_count > 0 or obs_count > 0:
        print(
            f"[multiview-search] {expected_pad} 관측은 있음({obs_count}개) "
            "하지만 좌표 미확정 → 다음 cycle에서 hint/panorama 기반 추가 관측"
        )
        return {
            "status": "continue",
            "target_xy": current,
            "place_ready": False,
            "reason": f"{expected_pad} v4 multiview/panorama 관측 누적 중, 좌표 미확정",
        }

    print(f"[multiview-search] {expected_pad} sign 미발견 → search 실패")
    return {"status": "failed", "target_xy": None, "place_ready": False, "reason": f"{expected_pad} sign 미발견"}


# ---------------------------------------------------------------------------
# 학생 TODO: VLM 표지판 확인 (첫 방문 패드 무결성 검증)
# ---------------------------------------------------------------------------

async def verify_pad_sign_with_vlm(ctx: Any, expected_pad: str) -> str:
    """첫 방문 패드에서 place 직전, 표지판 문자를 VLM으로 재확인한다.

    반환값: "match" | "mismatch" | "unavailable"
    - match: expected sign이 실제로 보인다.
    - mismatch: 다른 sign(A/B/C/D/E)은 보이는데 expected sign이 아니다.
    - unavailable: Timeout/Cancelled/API 실패 또는 sign 자체가 안 보인다.

    중요한 차이: Timeout/CancelledError는 "틀린 패드"가 아니므로 좌표 폐기 근거로 쓰지 않는다.
    """
    expected_pad = expected_pad.upper()
    if expected_pad not in {"B", "C", "D", "E"}:
        return "unavailable"
    try:
        await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
        await asyncio.sleep(0.25)
        reading = await read_signs_with_vlm(ctx, expected_pad)
    except asyncio.CancelledError as exc:
        print(f"[vlm] 표지판 확인 취소({exc!r}) → unavailable로 처리")
        return "unavailable"
    except Exception as exc:
        print(f"[vlm] 표지판 확인 호출 실패({exc!r}) → unavailable로 처리")
        return "unavailable"

    if reading is None:
        return "unavailable"
    letters = reading.get("visible_letters", []) or []
    print(f"[vlm] visible_letters={letters}, expected={expected_pad}")

    if reading.get("target_visible"):
        return "match"
    # 화면에 아무 sign도 없으면 카메라 각도/거리/API 판단 문제일 수 있으므로 좌표 폐기하지 않는다.
    if not letters:
        return "unavailable"
    # A 또는 다른 destination 문자가 명확히 보이면 실제 불일치로 취급한다.
    return "mismatch"


# ---------------------------------------------------------------------------
# Final_v5: 실제 place 직전 안전 게이트 + 가까이 재접근
# ---------------------------------------------------------------------------

def _vlm_offset_from_reading(reading: dict[str, Any]) -> float | None:
    """VLM reading에서 target bearing offset(deg)을 안전하게 얻는다."""
    raw = reading.get("target_offset_deg")
    try:
        if raw is not None:
            return float(raw)
    except Exception:
        pass
    pos = str(reading.get("target_position") or "").strip().lower()
    if pos in {"center", "centre"}:
        return 0.0
    if pos == "left":
        return 24.0
    if pos == "right":
        return -24.0
    return None


def _is_place_ready_from_reading(expected_pad: str, reading: dict[str, Any]) -> tuple[bool, str]:
    """target sign이 보이는지와 별개로, 실제 place를 해도 안전한 상황인지 판정한다.

    place_nearest_zone은 한 번 실행되면 cube가 손에서 사라질 수 있다. 따라서 최초 방문 rough 좌표에서는
    C/D/B/E sign이 보이는 것만으로 drop하지 않고, near + center 조건을 만족할 때만 place한다.
    """
    expected_pad = expected_pad.upper()
    if not reading or not reading.get("target_visible"):
        return False, "target_not_visible"

    letters = {str(x).upper() for x in (reading.get("visible_letters") or [])}
    if PAD_PLACE_BLOCK_A_VISIBLE and "A" in letters and expected_pad != "A":
        return False, "A_visible_with_target"

    dist_label = _normalize_vlm_distance(reading.get("target_distance", "unknown"))
    pos = str(reading.get("target_position") or "unknown").strip().lower()
    offset = _vlm_offset_from_reading(reading)
    centered = pos in PAD_PLACE_READY_POSITIONS or (offset is not None and abs(offset) <= PAD_PLACE_READY_MAX_OFFSET_DEG)

    if dist_label == "near" and centered:
        return True, f"ready: dist={dist_label}, pos={pos}, offset={offset}"
    return False, f"not_ready: dist={dist_label}, pos={pos}, offset={offset}, centered={centered}"



def _preplace_turn_deg_from_offset(offset: float | None) -> float | None:
    """VLM offset(deg)을 짧은 body 정렬 회전각으로 변환한다.

    left는 +, right는 - 부호를 그대로 쓴다. 한 번에 과하게 돌지 않도록 clamp한다.
    """
    if offset is None:
        return None
    try:
        raw = float(offset) * float(PAD_PLACE_ALIGN_SCALE)
    except Exception:
        return None
    if abs(raw) < PAD_PLACE_ALIGN_MIN_DEG:
        return None
    return max(-PAD_PLACE_ALIGN_MAX_DEG, min(PAD_PLACE_ALIGN_MAX_DEG, raw))


def _preplace_approach_args_from_reading(reading: dict[str, Any]) -> dict[str, float]:
    """medium 거리에서 한 번만 쓰는 짧은 body-frame 접근 motion을 만든다.

    world 좌표 go_to를 반복하면 sign 주변을 맴돌기 쉬우므로, 현재 heading 기준으로 짧게만 전진한다.
    offset이 있으면 아주 약한 wz를 섞어 target sign 쪽으로 호를 그리며 접근한다.
    """
    offset = _vlm_offset_from_reading(reading) or 0.0
    wz = max(-PAD_PLACE_APPROACH_MAX_WZ, min(PAD_PLACE_APPROACH_MAX_WZ, float(offset) * PAD_PLACE_APPROACH_WZ_PER_DEG))
    return {"vx": PAD_PLACE_APPROACH_VX, "vy": 0.0, "wz": wz, "duration_s": PAD_PLACE_APPROACH_DURATION_S}


def _preplace_action_plan(
    memory: AgentMemory,
    expected_pad: str,
    reading: dict[str, Any],
    ready_reason: str,
) -> dict[str, Any]:
    """place 금지 상황에서 어떤 안전 보정만 할지 결정한다.

    Final_v7 핵심:
      - near + left/right: 좌표 go_to 금지, 짧은 body align만 수행
      - medium: body-frame short approach를 최대 1회만 수행
      - far/unknown 또는 반복 실패: 오래된 관측/rough 좌표를 버리고 재탐색
    """
    expected_pad = expected_pad.upper()
    letters = {str(x).upper() for x in (reading.get("visible_letters") or [])}
    dist_label = _normalize_vlm_distance(reading.get("target_distance", "unknown"))
    pos = str(reading.get("target_position") or "unknown").strip().lower()
    offset = _vlm_offset_from_reading(reading)
    centered = pos in PAD_PLACE_READY_POSITIONS or (offset is not None and abs(offset) <= PAD_PLACE_READY_MAX_OFFSET_DEG)
    a_visible = PAD_PLACE_BLOCK_A_VISIBLE and "A" in letters and expected_pad != "A"

    # near인데 center가 아니면 더 이상 world 좌표를 새로 찍지 말고 시야 정렬만 한다.
    if dist_label == "near" and not centered:
        turn_deg = _preplace_turn_deg_from_offset(offset)
        if turn_deg is not None:
            return {
                "status": "align",
                "turn_deg": turn_deg,
                "reading": reading,
                "reason": f"near_but_not_center: pos={pos}, offset={offset}, A_visible={a_visible}; {ready_reason}",
            }
        return {"status": "reset_search", "reading": reading, "reason": f"near_not_center_but_no_offset: {ready_reason}"}

    # near+center라도 A가 같이 보이면 place 금지. 아주 짧게 한 번 더 앞으로 빠진 뒤 재확인한다.
    if dist_label == "near" and centered and a_visible:
        fwd_key = f"preplace_forward_{expected_pad}"
        if memory.failed_attempts.get(fwd_key, 0) < PAD_PLACE_MEDIUM_APPROACH_MAX_PER_PAD:
            args = _preplace_approach_args_from_reading(reading)
            args["vx"] = min(args["vx"], 0.14)
            args["duration_s"] = min(args["duration_s"], 0.75)
            return {
                "status": "approach",
                "motion_args": args,
                "reading": reading,
                "reason": f"near_center_but_A_visible: drop 금지 후 source에서 조금 더 벗어남; {ready_reason}",
            }
        return {"status": "reset_search", "reading": reading, "reason": f"near_center_but_A_visible_repeated: {ready_reason}"}

    # medium은 좌표 go_to를 반복하지 않고, body-frame 짧은 접근을 최대 1회만 수행한다.
    if dist_label == "medium":
        fwd_key = f"preplace_forward_{expected_pad}"
        if memory.failed_attempts.get(fwd_key, 0) < PAD_PLACE_MEDIUM_APPROACH_MAX_PER_PAD:
            return {
                "status": "approach",
                "motion_args": _preplace_approach_args_from_reading(reading),
                "reading": reading,
                "reason": f"medium_short_approach_once: pos={pos}, offset={offset}, A_visible={a_visible}; {ready_reason}",
            }
        return {"status": "reset_search", "reading": reading, "reason": f"medium_approach_already_used: {ready_reason}"}

    # far/unknown에서는 place도, refine go_to도 하지 않는다. rough 좌표/관측을 새로 잡는다.
    return {
        "status": "reset_search",
        "reading": reading,
        "reason": f"not_place_ready_reset: dist={dist_label}, pos={pos}, offset={offset}, A_visible={a_visible}; {ready_reason}",
    }


async def evaluate_preplace_with_vlm(
    ctx: Any,
    memory: AgentMemory,
    expected_pad: str,
) -> dict[str, Any]:
    """place 직전 VLM으로 target sign/거리/위치를 확인하고 place 또는 refine를 결정한다.

    반환 status:
      - ready: 실제 place 수행 가능
      - align: near지만 left/right라 짧은 body 회전만 수행
      - approach: medium이라 body-frame 짧은 접근 1회 수행
      - reset_search: rough 좌표/관측을 버리고 문자 재탐색
      - mismatch: 다른 sign만 보임. rough 좌표 폐기 후 재탐색
      - unavailable: VLM 실패/빈 화면. drop하지 않음
    """
    expected_pad = expected_pad.upper()
    try:
        await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
        await asyncio.sleep(0.18)
        reading = await read_signs_with_vlm(ctx, expected_pad)
    except asyncio.CancelledError as exc:
        print(f"[preplace-check] {expected_pad}: VLM 취소({exc!r}) → drop 금지")
        return {"status": "unavailable", "reason": "vlm_cancelled"}
    except Exception as exc:
        print(f"[preplace-check] {expected_pad}: VLM 실패({exc!r}) → drop 금지")
        return {"status": "unavailable", "reason": "vlm_error"}

    if reading is None:
        print(f"[preplace-check] {expected_pad}: VLM reading 없음 → drop 금지")
        return {"status": "unavailable", "reason": "vlm_none"}

    letters = [str(x).upper() for x in (reading.get("visible_letters") or [])]
    print(
        f"[preplace-check] expected={expected_pad}, visible={letters}, "
        f"target_visible={reading.get('target_visible')}, pos={reading.get('target_position')}, "
        f"dist={reading.get('target_distance')}, offset={reading.get('target_offset_deg')}"
    )

    if not reading.get("target_visible"):
        if letters:
            return {"status": "mismatch", "reading": reading, "reason": f"expected {expected_pad} not visible; visible={letters}"}
        return {"status": "unavailable", "reading": reading, "reason": "no_sign_visible"}

    ready, why = _is_place_ready_from_reading(expected_pad, reading)
    if ready:
        print(f"[preplace-check] {expected_pad}: place 허용 — {why}")
        return {"status": "ready", "reading": reading, "reason": why}

    plan = _preplace_action_plan(memory, expected_pad, reading, why)
    print(f"[preplace-plan] {expected_pad}: status={plan.get('status')}, reason={plan.get('reason')}")
    return plan


# ---------------------------------------------------------------------------
# 학생 TODO: LLM decision 함수 (하이브리드)
# ---------------------------------------------------------------------------

LLM_SYSTEM_PROMPT = (
    "당신은 창고 분류 로봇의 상위 의사결정자입니다. "
    "situation과 로봇 상태를 보고 allowed_next_actions 중 하나를 고르세요. "
    '{"next_action": "...", "target_color": null, "reason": "..."} 형식의 JSON만 반환하세요.'
)


async def consult_llm_decision(
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None,
    *,
    situation: str,
    allowed: set[str],
    fallback: AgentDecision,
) -> AgentDecision:
    """규칙만으로 정답이 없는 상황에서 텍스트 LLM에 다음 행동을 묻는다.

    무결성 원칙:
    - 응답은 parse_agent_decision + allowed 집합으로 이중 검증한다.
    - target_color/target_location은 LLM 값을 버리고 코드(fallback)의 값을
      강제한다 — LLM이 잘못된 패드로 배송을 유도할 수 없다.
    - 키 없음/타임아웃/검증 실패 시 항상 안전한 규칙 fallback을 쓴다(연속성).
    """
    api_key = os.environ.get("TOKAMAK_API_KEY", "")
    if not api_key:
        print("[llm] TOKAMAK_API_KEY 없음 → 규칙 fallback 사용")
        return fallback
    context = build_decision_context(TASK, observation, memory, last_result)
    context["situation"] = situation
    context["allowed_next_actions"] = sorted(allowed)
    messages = [
        {"role": "system", "content": LLM_SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(context, ensure_ascii=False, default=str)},
    ]
    try:
        # call_llm은 동기 HTTP 호출이므로 thread로 보낸다.
        text = await asyncio.to_thread(call_llm, messages, api_key=api_key, timeout_s=LLM_TIMEOUT_S)
    except Exception as exc:
        print(f"[llm] 호출 실패({exc!r}) → 규칙 fallback 사용")
        return fallback
    memory.llm_consult_count += 1
    decision = parse_agent_decision(text)
    if decision is None or decision.next_action not in allowed:
        print(f"[llm] 응답 검증 실패 → 규칙 fallback 사용: {text[:120]!r}")
        return fallback
    decision.target_color = fallback.target_color
    decision.target_location = fallback.target_location
    decision.reason = f"LLM: {decision.reason}"
    print(f"[llm] 결정 채택: {decision.next_action} — {decision.reason}")
    return decision


async def decide_next_action(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> AgentDecision:
    """stage 기반 규칙이 기본, 애매한 상황에서만 LLM에 묻는 하이브리드 정책.

    정상 루프(부트스트랩→pick→이동→place→복귀)는 결정적 규칙으로 처리해
    사이클당 지연을 없앤다(행동 연속성). LLM은 규칙만으로 정답이 없는
    지점에서만 호출한다(§7 LLM 의사결정 루프):
      1. search_pad_failed — 재탐색 vs 포기(skip_target)
      2. 들고 있어야 할 큐브 소실(holding/delivering/at_pad) — A 복귀 vs recover
      3. 알 수 없는 stage — recover / A 복귀 / stop
    핵심 분기 우선순위(큐브 보유 시):
      패드 좌표 없음 → 탐색 / 패드 좌표 있음 → 해당 패드로 이동
    가까운 2색 선택 전략은 제거했으므로 모든 색상을 동일하게 운반한다.
    """

    def _held_lost_fallback(stage_name: str) -> AgentDecision:
        return AgentDecision(
            next_action="navigate_to_cube",
            target_location="A",
            reason=f"규칙 fallback: {stage_name}인데 held_color 없음 → A 복귀",
        )

    # --- Bootstrap: 첫 시작 시 A 탐색 ---
    if not memory.bootstrapped_to_a:
        # estimate_target_xy_from_observation 내부 min_area 필터와 일치시켜야
        # navigate_to_cube → failed 무한 루프를 막는다
        navigable = any(d.blob_area >= MIN_TARGET_BLOB_AREA for d in observation.detections)
        if navigable or "A" in memory.known_locations:
            return AgentDecision(
                next_action="navigate_to_cube",
                target_location="A",
                reason="Bootstrap: 유효 blob 또는 A 좌표 있음, source로 이동합니다.",
            )
        return AgentDecision(
            next_action="search_cube",
            target_location="A",
            reason="Bootstrap: 유효 blob 없음, 탐색합니다.",
        )

    # --- A에서 큐브 pick ---
    if memory.stage == "at_source":
        pick_failures = memory.failed_attempts.get("pick", 0)
        if pick_failures >= MAX_PICK_FAILURES:
            return AgentDecision(
                next_action="recover",
                reason=f"pick {pick_failures}회 연속 실패. 후진 후 bootstrap 재시작.",
            )
        return AgentDecision(next_action="pick_cube", reason="A에서 큐브를 집습니다.")

    # --- 방금 패드를 발견한 경우 — 바로 배달 ---
    # 이유: 탐색으로 얻은 좌표는 첫 접근용이며, 실제 place 성공 위치로 확정해야 한다.
    if memory.stage == "delivering_just_found":
        color = memory.held_color
        if color is None:
            return await consult_llm_decision(
                observation, memory, last_result,
                situation="배달 중이던 큐브가 손에서 사라졌습니다. A로 복귀해 다시 집을지(navigate_to_cube), 자세 복구(recover)를 먼저 할지 결정하세요.",
                allowed={"navigate_to_cube", "recover"},
                fallback=_held_lost_fallback("delivering_just_found"),
            )
        pad_name = DESTINATION_SIGN_RULES.get(color or "")
        if pad_name and pad_name in memory.known_locations:
            return AgentDecision(
                next_action="navigate_to_pad",
                target_color=color,
                target_location=pad_name,
                reason=f"방금 발견한 {color} 패드({pad_name})로 배달합니다.",
            )
        # 이론상 불가 케이스: search_pad 성공 시 저장했으므로 항상 known에 있어야 한다
        print(f"[recovery] delivering_just_found인데 pad 좌표 없음 (color={color})")
        return AgentDecision(next_action="recover", reason=f"pad 좌표 소실 (color={color}).")

    # --- 큐브 보유 중: 메모리 확인이 1순위 ---
    if memory.stage == "holding_cube":
        color = memory.held_color
        if color is None:
            return await consult_llm_decision(
                observation, memory, last_result,
                situation="들고 있어야 할 큐브가 손에 없습니다. A로 복귀해 다시 집을지(navigate_to_cube), 자세 복구(recover)를 먼저 할지 결정하세요.",
                allowed={"navigate_to_cube", "recover"},
                fallback=_held_lost_fallback("holding_cube"),
            )
        pad_name = DESTINATION_SIGN_RULES.get(color)
        if pad_name is None:
            print(f"[recovery] 알 수 없는 큐브 색상: {color} → A 복귀")
            return AgentDecision(
                next_action="navigate_to_cube",
                target_location="A",
                reason=f"알 수 없는 색상 {color} → A 복귀",
            )

        # [메모리에 없음] → 탐색. 단 누적 실패가 한계를
        # 넘은 색은 다시 헤매지 않고 즉시 포기한다 (연속성 — 시간 낭비 방지).
        if pad_name not in memory.known_locations:
            search_failures = memory.failed_attempts.get(f"search_{pad_name}", 0)
            if search_failures >= MAX_PAD_SEARCH_FAILURES:
                return AgentDecision(
                    next_action="skip_target",
                    target_color=color,
                    reason=f"{pad_name} 탐색 누적 {search_failures}회 실패 → 포기하고 A에 drop.",
                )
            return AgentDecision(
                next_action="search_pad",
                target_color=color,
                target_location=pad_name,
                reason=f"{color} 패드({pad_name}) 위치 미확인 → 탐색합니다.",
            )

        # [메모리에 있음] 모든 색상을 동일하게 좌표로 바로 이동 (방향·가시성 무관)
        return AgentDecision(
            next_action="navigate_to_pad",
            target_color=color,
            target_location=pad_name,
            reason=f"{color} 패드({pad_name}) 좌표 알고 있음 → 바로 이동.",
        )

    # --- 패드 탐색 실패 직후: 우선 결정적 재탐색, 한계 근처에서만 LLM 상담 ---
    if memory.stage == "search_pad_failed":
        color = memory.held_color
        if color is None:
            return AgentDecision(
                next_action="navigate_to_cube",
                target_location="A",
                reason="탐색 실패 후 큐브도 소실 → A 복귀",
            )
        pad_name = DESTINATION_SIGN_RULES.get(color, "")
        search_failures = memory.failed_attempts.get(f"search_{pad_name}", 0)
        # VLM Timeout이 많은 상황에서 매 실패마다 LLM까지 호출하면 지연이 커진다.
        # 초기 실패는 바로 재탐색하고, 한계 직전부터만 LLM에 재시도/포기 판단을 맡긴다.
        if search_failures < max(2, MAX_PAD_SEARCH_FAILURES - 1):
            return AgentDecision(
                next_action="search_pad",
                target_color=color,
                target_location=pad_name,
                reason=f"패드 문자 탐색 실패 {search_failures}회 → VLM 호출을 줄인 정면 폐루프 방식으로 재탐색",
            )
        fallback_action = "search_pad" if search_failures < MAX_PAD_SEARCH_FAILURES else "skip_target"
        fallback = AgentDecision(
            next_action=fallback_action,
            target_color=color,
            target_location=pad_name,
            reason=f"규칙 fallback: 탐색 실패 {search_failures}회 → {fallback_action}",
        )
        return await consult_llm_decision(
            observation, memory, last_result,
            situation=(
                f"{color} 패드({pad_name}) 탐색이 지금까지 {search_failures}회 실패했습니다. "
                "기본 원칙은 다시 탐색(search_pad)하여 정상 배달하는 것입니다. "
                "단, 누적 실패가 한계를 넘은 경우에만 skip_target을 선택해 루프 정지를 방지하세요."
            ),
            allowed={"search_pad", "skip_target"},
            fallback=fallback,
        )

    # --- 패드 탐색 포기 후 A로 복귀해서 drop ---
    if memory.stage == "pending_drop_at_a":
        if "A" in memory.known_locations:
            return AgentDecision(
                next_action="navigate_to_cube",
                target_location="A",
                reason="패드 탐색 포기. A로 복귀 중.",
            )
        return AgentDecision(
            next_action="place_cube",
            target_location="A",
            reason="패드 탐색 포기. A 좌표 없어 현 위치에서 drop.",
        )

    # --- A에 도착했는데 큐브 보유 중 (패드 탐색 포기 복구) ---
    if memory.stage == "drop_at_a_ready":
        return AgentDecision(
            next_action="place_cube",
            target_location="A",
            reason="A 도달. 포기한 큐브 drop.",
        )

    # --- 패드 도달 후 place ---
    if memory.stage == "at_pad":
        if memory.held_color is None:
            # 이동 중 큐브를 떨어뜨린 경우 — 빈손 place는 무의미하고 상태만 꼬인다
            return await consult_llm_decision(
                observation, memory, last_result,
                situation="패드 앞에 도착했지만 큐브가 손에 없습니다. A로 복귀해 다시 집을지(navigate_to_cube), 자세 복구(recover)를 먼저 할지 결정하세요.",
                allowed={"navigate_to_cube", "recover"},
                fallback=_held_lost_fallback("at_pad"),
            )
        return AgentDecision(
            next_action="place_cube",
            target_color=memory.held_color,
            target_location=memory.active_pad_name,
            reason="패드 도달. drop합니다.",
        )

    # --- A로 복귀 ---
    if memory.stage == "returning_to_source":
        return AgentDecision(
            next_action="navigate_to_cube",
            target_location="A",
            reason="배달 완료. A로 복귀합니다.",
        )

    # --- 알 수 없는 stage: LLM 상담 ---
    fallback = AgentDecision(next_action="recover", reason=f"규칙 fallback: 알 수 없는 stage {memory.stage}")
    return await consult_llm_decision(
        observation, memory, last_result,
        situation=f"agent가 정의되지 않은 stage '{memory.stage}'에 도달했습니다. 복구 방법을 고르세요.",
        allowed={"recover", "navigate_to_cube", "stop"},
        fallback=fallback,
    )


# ---------------------------------------------------------------------------
# 학생 TODO: observation
# ---------------------------------------------------------------------------

async def observe_world(ctx: Any, memory: AgentMemory) -> Observation:
    """(연속성 최적화) 시각 정보가 필요한 stage에서만 풀 스캔한다.

    사이클 단위 detections를 실제로 소비하는 곳은 bootstrap 경로뿐이다:
      - decide: bootstrap에서 navigable blob 유무 판단
      - execute: bootstrap navigate_to_cube의 vision 좌표 추정
    부트스트랩 이후에는 known_locations 좌표로만 이동하고, 패드 탐색은
    execute의 text_forward_search가 자체 스캔하므로 매 사이클 3방향
    스캔(≈1.2초+)은 순수 시간 낭비다. robot_status는 항상 읽는다 —
    place 성공 시점의 로봇 좌표 기록(waypoint 확정)에 필요하다.
    """
    robot_status = await get_robot_status(ctx)
    if not memory.bootstrapped_to_a:
        detections = await scan_head(ctx)
        note = "bootstrap: 풀 스캔 수행"
    else:
        detections = []
        note = "bootstrap 완료: 사이클 스캔 생략 (좌표 기반 이동)"
    return Observation(robot_status=robot_status, detections=detections, note=note)



# ---------------------------------------------------------------------------
# v8_0705 blob-guided fast-estimate-go_to override
# ---------------------------------------------------------------------------
# 목표: C가 보일 때 폐루프처럼 조금씩 전진하지 않는다.
# 1) VLM sign bearing을 서로 다른 pose에서 최소 2개 확보한다.
# 2) 교차각/잔차 검사를 통과하면 즉시 approach_xy를 확정한다.
# 3) search_pad action 안에서 바로 Menlo go_to(pose)까지 수행한다.
# 4) place 성공 전까지는 rough waypoint로만 취급하고, place 성공 pose로 최종 덮어쓴다.

# 빠른 탐색을 위해 호출 수/스텝을 줄인다. 좌표가 확정되면 즉시 go_to한다.
MAX_VLM_CALLS_PER_SEARCH = int(os.getenv("MENLO_MAX_VLM_CALLS_PER_SEARCH", "10"))
PAD_TRIANGULATION_SEARCH_STEPS = int(os.getenv("MENLO_PAD_TRIANGULATION_SEARCH_STEPS", "3"))
PAD_FAST_MIN_DIVERSE_OBS = 2
PAD_FAST_MIN_CROSS_ANGLE_DEG = float(os.getenv("MENLO_PAD_FAST_MIN_CROSS_DEG", "8.0"))
PAD_FAST_MAX_MEDIAN_RESIDUAL_M = float(os.getenv("MENLO_PAD_FAST_MAX_RESIDUAL", "0.65"))
# v8: sign triangulation fallback도 너무 멀리 서지 않게 standoff를 낮춘다.
# v7 로그에서는 C sign이 보이는데 standoff=0.70m 때문에 현재 위치 근처를
# approach_xy로 잡고 바로 place를 시도해 실패가 반복되었다.
PAD_FAST_APPROACH_STANDOFF = {
    "B": 0.25,
    "C": 0.24,
    "D": 0.28,
    "E": 0.25,
}
PAD_FAST_HEAD_YAWS = (-0.85, -0.45, 0.0, 0.45, 0.85)
PAD_FAST_RECHECK_HEAD_YAWS = (0.0, -0.55, 0.55)

# Final_v4: triangulation sign_xy가 현재 위치 근처로 접히는 문제 방지.
# 로그에서는 VLM이 C를 medium으로 보고 있는데 sign_xy가 현재 위치에서 약 0.12m로 계산되어
# approach_xy가 사실상 현재 위치가 되었고, go_to 후 바로 place 실패가 반복되었다.
PAD_SIGN_DISTANCE_COMPAT_BOUNDS = {
    "near": (0.05, 1.10),
    "medium": (0.40, 2.80),
    "far": (0.85, 4.80),
    "unknown": (0.00, 5.00),
}
PAD_SIGN_APPROACH_MIN_DELTA_BY_LABEL = {
    "near": 0.04,
    "medium": 0.18,
    "far": 0.32,
    "unknown": 0.10,
}
# triangulation이 VLM 거리 label과 충돌하면, place하지 않고 현재 보이는 sign 방향으로
# 짧게 전진해 새 관측 pose를 만든다. 이 값은 pad 좌표 확정이 아니라 재관측용 이동량이다.
PAD_SIGN_SINGLE_RAY_PROGRESS_M = {
    "near": 0.12,
    "medium": 0.42,
    "far": 0.78,
    "unknown": 0.30,
}
PAD_SIGN_SINGLE_RAY_MAX_OBS_DIST_M = float(os.getenv("MENLO_PAD_SIGN_SINGLE_RAY_MAX_OBS_DIST_M", "0.36"))

# Final_v5: place 직전 안전 게이트.
# C/D/B/E sign이 보인다는 사실만으로는 scoring zone 위에 있다는 뜻이 아니다.
# target sign이 center+near일 때만 실제 place를 수행하고, 아니면 sign 방향으로 Menlo go_to(pose) 재접근한다.
PAD_PLACE_READY_MAX_OFFSET_DEG = float(os.getenv("MENLO_PAD_PLACE_READY_MAX_OFFSET_DEG", "12.0"))
PAD_PLACE_READY_POSITIONS = {"center", "centre"}
PAD_PLACE_BLOCK_A_VISIBLE = os.getenv("MENLO_PAD_PLACE_BLOCK_A_VISIBLE", "1").strip().lower() not in {"0", "false", "no"}
# Final_v7: preplace는 더 이상 world 좌표 go_to를 반복하지 않는다.
# near+left/right는 짧은 body align, medium은 짧은 body-frame approach 1회만 허용한다.
PAD_PLACE_REFINE_MOVE_M = {
    "near": float(os.getenv("MENLO_PAD_PLACE_REFINE_NEAR_M", "0.10")),
    "medium": float(os.getenv("MENLO_PAD_PLACE_REFINE_MEDIUM_M", "0.22")),
    "far": float(os.getenv("MENLO_PAD_PLACE_REFINE_FAR_M", "0.00")),
    "unknown": float(os.getenv("MENLO_PAD_PLACE_REFINE_UNKNOWN_M", "0.00")),
}
PAD_PLACE_REFINE_MAX_PER_PAD = int(os.getenv("MENLO_PAD_PLACE_REFINE_MAX_PER_PAD", "3"))
PAD_PLACE_MEDIUM_APPROACH_MAX_PER_PAD = int(os.getenv("MENLO_PAD_PLACE_MEDIUM_APPROACH_MAX_PER_PAD", "1"))
PAD_PLACE_REFINE_MIN_DELTA_M = float(os.getenv("MENLO_PAD_PLACE_REFINE_MIN_DELTA_M", "0.04"))
PAD_PLACE_ALIGN_MIN_DEG = float(os.getenv("MENLO_PAD_PLACE_ALIGN_MIN_DEG", "5.0"))
PAD_PLACE_ALIGN_MAX_DEG = float(os.getenv("MENLO_PAD_PLACE_ALIGN_MAX_DEG", "18.0"))
PAD_PLACE_ALIGN_SCALE = float(os.getenv("MENLO_PAD_PLACE_ALIGN_SCALE", "0.70"))
PAD_PLACE_APPROACH_VX = float(os.getenv("MENLO_PAD_PLACE_APPROACH_VX", "0.18"))
PAD_PLACE_APPROACH_DURATION_S = float(os.getenv("MENLO_PAD_PLACE_APPROACH_DURATION_S", "1.00"))
PAD_PLACE_APPROACH_WZ_PER_DEG = float(os.getenv("MENLO_PAD_PLACE_APPROACH_WZ_PER_DEG", "0.008"))
PAD_PLACE_APPROACH_MAX_WZ = float(os.getenv("MENLO_PAD_PLACE_APPROACH_MAX_WZ", "0.18"))
PAD_TRIANGULATION_CONFLICT_RESET_N = int(os.getenv("MENLO_PAD_TRIANGULATION_CONFLICT_RESET_N", "2"))


# debug frame 저장은 사용하지 않는다. 이전 버전과 달리 디스크 I/O 없음.
VLM_DEBUG_FRAME_SAVE = False


# ---------------------------------------------------------------------------
# Final_v4: VLM sign 확인 + color blob 엄격 필터 + 현재위치-근접 triangulation 방지
# ---------------------------------------------------------------------------
# 목적: VLM으로 "이 프레임에 target sign(C/D/...)이 보인다"를 확인한 뒤,
# 같은 head 방향에서 색상 blob의 bearing/area를 사용해 바로 world 좌표를 만든다.
# 교차각을 만들기 위해 true-strafe를 반복하기 전에 빠른 1차 좌표를 만들기 위한 경로다.
PAD_TO_COLOR = {sign: color for color, sign in DESTINATION_SIGN_RULES.items()}
# Final_v1: held cube / nearby cube 오인을 막기 위해, 현재 들고 있는 큐브 색상과
# 같은 color blob은 기본적으로 pad 좌표 추정에 사용하지 않는다.
# 필요 시 환경변수 MENLO_PAD_BLOB_USE_HELD_COLOR=1 로만 예외 허용.
PAD_BLOB_USE_HELD_COLOR = os.getenv("MENLO_PAD_BLOB_USE_HELD_COLOR", "0").strip().lower() in {"1", "true", "yes", "y"}
PAD_BLOB_MIN_AREA = int(os.getenv("MENLO_PAD_BLOB_MIN_AREA", "450"))
# Final_v2: 너무 큰 blob은 held cube/source cube/foreground 물체일 가능성이 크므로 폐기한다.
# v8 로그에서 270k~400k area blob이 패드로 오인되어 잘못된 go_to를 만들었다.
PAD_BLOB_MAX_AREA = int(os.getenv("MENLO_PAD_BLOB_MAX_AREA", "120000"))
# Final_v2: VLM offset과 blob angle은 같은 대상일 때만 좌표 추정에 사용한다.
# 기본 허용치는 9도. 필요하면 MENLO_PAD_BLOB_MAX_OFFSET_DIFF_DEG=8~10 사이로 조절한다.
PAD_BLOB_MAX_OFFSET_DIFF_DEG = float(os.getenv("MENLO_PAD_BLOB_MAX_OFFSET_DIFF_DEG", "9.0"))
PAD_BLOB_DISTANCE_SCALE = float(os.getenv("MENLO_PAD_BLOB_DISTANCE_SCALE", "115.0"))
PAD_BLOB_MAX_DISTANCE_M = float(os.getenv("MENLO_PAD_BLOB_MAX_DISTANCE_M", "3.6"))
PAD_BLOB_MIN_TRAVEL_BY_DISTANCE = {
    "near": 0.25,
    "medium": 0.55,
    "far": 0.90,
    "unknown": 0.35,
}
PAD_BLOB_DISTANCE_BOUNDS = {
    "near": (0.30, 0.95),
    "medium": (0.80, 1.85),
    "far": (1.35, 3.20),
    "unknown": (0.45, 2.30),
}
# Final_v3: raw_dist(area 기반 거리)와 VLM distance label이 명백히 충돌하면 blob을 버린다.
# 충돌하지 않는 경우에도 VLM label 하한(예: medium=0.80m)으로 강제 보정하지 않고
# raw_dist를 그대로 사용한다. v8/v2의 used_dist=0.80m 강제 보정 문제를 제거한다.
PAD_BLOB_RAW_DIST_COMPAT_BOUNDS = {
    "near": (0.18, 1.10),
    "medium": (0.45, 2.30),
    "far": (1.00, 3.80),
    "unknown": (0.18, 3.80),
}
PAD_BLOB_APPROACH_STANDOFF = {
    "B": 0.12,
    "C": 0.14,
    "D": 0.18,
    "E": 0.12,
}
PAD_BLOB_MIN_GO_TO_DELTA_M = float(os.getenv("MENLO_PAD_BLOB_MIN_GOTO_DELTA_M", "0.22"))


def _clamp_distance_by_vlm_label(distance_m: float, label: str) -> float:
    # legacy helper: Final_v3 blob-guided path에서는 더 이상 사용하지 않는다.
    # 남겨두는 이유는 이전 함수/실험 코드와의 호환성 때문이다.
    lo, hi = PAD_BLOB_DISTANCE_BOUNDS.get(str(label or "unknown"), PAD_BLOB_DISTANCE_BOUNDS["unknown"])
    return _clamp(float(distance_m), lo, hi)


def _resolve_blob_distance_m(raw_distance: float, label: str) -> float | None:
    """Final_v3 distance resolver.

    핵심 원칙:
    1) raw_dist와 VLM distance label이 명백히 충돌하면 None 반환.
    2) 충돌하지 않으면 raw_dist를 그대로 사용한다.
       즉, medium label이라고 해서 0.56m를 0.80m로 끌어올리지 않는다.
    3) 안전을 위해 전역 최대 거리만 제한한다.
    """
    label = str(label or "unknown")
    raw_distance = float(raw_distance)
    lo, hi = PAD_BLOB_RAW_DIST_COMPAT_BOUNDS.get(label, PAD_BLOB_RAW_DIST_COMPAT_BOUNDS["unknown"])
    if not (lo <= raw_distance <= hi):
        return None
    return min(max(raw_distance, 0.05), PAD_BLOB_MAX_DISTANCE_M)


def _angle_diff_deg(a: float, b: float) -> float:
    return abs(_normalize_deg(float(a) - float(b)))


def _pad_blob_expected_offset_deg(reading: dict[str, Any]) -> float:
    # robust offset: target_position과 숫자 offset 충돌 시 position을 1차 신뢰한다.
    return float(_normalized_offset_deg_from_reading(reading))


def _choose_target_color_blob(
    detections: list[Any],
    *,
    target_color: str,
    expected_offset_deg: float,
    distance_label: str,
    allow_loose: bool,
) -> Any | None:
    candidates = []
    reject_reasons: list[str] = []
    for d in detections:
        if getattr(d, "color", None) != target_color:
            continue
        area = int(getattr(d, "blob_area", 0) or 0)
        if area < PAD_BLOB_MIN_AREA:
            reject_reasons.append(f"area<{PAD_BLOB_MIN_AREA}:{area}")
            continue
        if area > PAD_BLOB_MAX_AREA:
            reject_reasons.append(f"area>{PAD_BLOB_MAX_AREA}:{area}")
            continue
        angle = float(getattr(d, "angle_deg", 0.0) or 0.0)
        diff = _angle_diff_deg(angle, expected_offset_deg)
        # Final_v2: target_position이 unknown이어도 같은 대상인지 확인하려면 각도 일치가 필수다.
        # 느슨한 허용은 held/source blob 오인을 키우므로 기본적으로 사용하지 않는다.
        if diff > PAD_BLOB_MAX_OFFSET_DIFF_DEG:
            reject_reasons.append(f"diff>{PAD_BLOB_MAX_OFFSET_DIFF_DEG:.1f}:{diff:.1f}")
            continue
        raw_distance = PAD_BLOB_DISTANCE_SCALE / math.sqrt(max(area, 1))
        label = str(distance_label or "unknown")
        raw_lo, raw_hi = PAD_BLOB_RAW_DIST_COMPAT_BOUNDS.get(label, PAD_BLOB_RAW_DIST_COMPAT_BOUNDS["unknown"])
        if not (raw_lo <= raw_distance <= raw_hi):
            reject_reasons.append(f"raw_dist_conflict:{raw_distance:.2f}m/{label}")
            continue
        # area가 너무 큰 blob을 이미 버렸으므로, offset 일치도와 합리적 크기를 함께 본다.
        score = math.log(max(area, 1)) * 1.2 - diff * 0.45
        candidates.append((score, diff, area, raw_distance, d))
    if not candidates:
        if reject_reasons:
            sample = ", ".join(reject_reasons[:5])
            if len(reject_reasons) > 5:
                sample += f", ...(+{len(reject_reasons)-5})"
            print(f"[blob-filter] {target_color}: 모든 blob 폐기 → {sample}")
        return None
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][4]


async def _estimate_pad_approach_xy_from_vlm_blob(
    ctx: Any,
    memory: AgentMemory,
    expected_pad: str,
    *,
    head_yaw: float,
    reading: dict[str, Any],
) -> tuple[float, float] | None:
    """VLM이 target sign을 확인한 frame 근처에서 색상 blob으로 pad 접근 좌표를 즉시 만든다.

    - VLM: target sign 정체성 확인, left/center/right offset 제공
    - blob: 같은 색 패드의 bearing/area로 거리와 world xy 추정
    - 실패하면 None을 반환하고 기존 bearing-ray triangulation fallback으로 넘어간다.
    """
    expected_pad = expected_pad.upper()
    target_color = PAD_TO_COLOR.get(expected_pad)
    if not target_color or not reading.get("target_visible"):
        return None

    held_color = str(getattr(memory, "held_color", "") or "").strip().lower()
    target_color_norm = str(target_color or "").strip().lower()
    if held_color and target_color_norm and held_color == target_color_norm and not PAD_BLOB_USE_HELD_COLOR:
        print(
            f"[blob-guard] {expected_pad}/{target_color}: held_color={memory.held_color}와 같은 color blob은 "
            "held cube/source cube 오인 위험으로 사용하지 않음 → triangulation fallback"
        )
        return None

    # VLM 호출 직후 같은 head pose에서 color blob을 읽는다. 완전히 같은 frame은 아니지만
    # 로봇이 정지해 있으므로 좌표 추정에는 충분히 가까운 관측이다.
    try:
        detections = await perceive(ctx)
    except Exception as exc:
        print(f"[blob-guided] {expected_pad}: perceive 실패({exc!r}) → triangulation fallback")
        return None

    expected_offset = _pad_blob_expected_offset_deg(reading)
    distance_label = str(reading.get("target_distance", "unknown") or "unknown")
    allow_loose = False
    blob = _choose_target_color_blob(
        detections,
        target_color=target_color,
        expected_offset_deg=expected_offset,
        distance_label=distance_label,
        allow_loose=allow_loose,
    )
    if blob is None:
        print(
            f"[blob-guided] {expected_pad}: target_color={target_color} blob 없음/offset 불일치 "
            f"(expected_offset={expected_offset:+.1f}°) → triangulation fallback"
        )
        return None

    status = await get_robot_status(ctx)
    robot_xy = robot_xy_from_status(status)
    robot_yaw_deg = robot_yaw_deg_from_status(status)
    if robot_xy is None or robot_yaw_deg is None:
        return None

    area = max(float(getattr(blob, "blob_area", 1) or 1), 1.0)
    blob_angle = float(getattr(blob, "angle_deg", 0.0) or 0.0)
    offset_diff = _angle_diff_deg(blob_angle, expected_offset)
    raw_distance = PAD_BLOB_DISTANCE_SCALE / math.sqrt(area)
    raw_lo, raw_hi = PAD_BLOB_RAW_DIST_COMPAT_BOUNDS.get(distance_label, PAD_BLOB_RAW_DIST_COMPAT_BOUNDS["unknown"])
    if not (raw_lo <= raw_distance <= raw_hi):
        print(
            f"[blob-filter] {expected_pad}: raw_dist={raw_distance:.2f}m와 dist_label={distance_label} 충돌 "
            f"(허용 {raw_lo:.2f}~{raw_hi:.2f}m) → triangulation fallback"
        )
        return None
    if area > PAD_BLOB_MAX_AREA:
        print(
            f"[blob-filter] {expected_pad}: blob_area={int(area)}가 상한 {PAD_BLOB_MAX_AREA} 초과 "
            "→ held/source/foreground blob 오인 방지, triangulation fallback"
        )
        return None
    if offset_diff > PAD_BLOB_MAX_OFFSET_DIFF_DEG:
        print(
            f"[blob-filter] {expected_pad}: offset diff={offset_diff:.1f}° > {PAD_BLOB_MAX_OFFSET_DIFF_DEG:.1f}° "
            "→ VLM sign과 다른 blob으로 판단, triangulation fallback"
        )
        return None
    distance_m = _resolve_blob_distance_m(raw_distance, distance_label)
    if distance_m is None:
        print(
            f"[blob-filter] {expected_pad}: raw_dist={raw_distance:.2f}m와 dist_label={distance_label} 충돌 "
            "→ forced used_dist 보정 없이 triangulation fallback"
        )
        return None

    world_bearing_rad = math.radians(float(robot_yaw_deg) + math.degrees(float(head_yaw)) + blob_angle)
    standoff = PAD_BLOB_APPROACH_STANDOFF.get(expected_pad, 0.14)
    # Final_v3: raw_dist를 VLM label 하한으로 강제 보정하지 않는다.
    # used_dist는 raw_dist 기반이고, travel도 distance-standoff 그대로 사용한다.
    # 단, 이동 명령이 0에 가까워지는 것만 막기 위해 아주 작은 하한만 둔다.
    travel_m = max(0.06, distance_m - standoff)

    approach_xy = (
        float(robot_xy[0]) + travel_m * math.cos(world_bearing_rad),
        float(robot_xy[1]) + travel_m * math.sin(world_bearing_rad),
    )
    approach_xy = _x_fence_clamp_xy(float(approach_xy[0]), float(approach_xy[1]))

    if not _is_safe_destination_candidate(memory, expected_pad, approach_xy, source="vlm-color-blob-final-v3"):
        return None

    delta = xy_distance(robot_xy, approach_xy) or 0.0
    if delta < PAD_BLOB_MIN_GO_TO_DELTA_M and distance_label not in {"near"}:
        print(
            f"[blob-guided] {expected_pad}: approach delta={delta:.2f}m가 너무 작음 "
            f"(dist_label={distance_label}) → triangulation fallback"
        )
        return None

    print(
        f"[blob-guided] {expected_pad}/{target_color}: blob_area={int(area)}, "
        f"blob_angle={blob_angle:+.1f}°, vlm_offset={expected_offset:+.1f}°, diff={offset_diff:.1f}°, "
        f"raw_dist={raw_distance:.2f}m, dist_label={distance_label}, used_dist(raw)={distance_m:.2f}m, "
        f"travel={travel_m:.2f}m → approach_xy={approach_xy}"
    )
    return approach_xy


def _fast_ray_metrics(point: tuple[float, float], observations: list[dict[str, Any]]) -> dict[str, Any]:
    residuals: list[float] = []
    distances: list[float] = []
    front_count = 0
    max_cross = 0.0
    for i in range(len(observations)):
        for j in range(i + 1, len(observations)):
            max_cross = max(max_cross, _angle_between_bearings_deg(observations[i]["bearing_rad"], observations[j]["bearing_rad"]))
    for obs in observations:
        px, py = obs["xy"]
        theta = float(obs["bearing_rad"])
        dx, dy = math.cos(theta), math.sin(theta)
        vx_p, vy_p = float(point[0]) - float(px), float(point[1]) - float(py)
        forward = vx_p * dx + vy_p * dy
        lateral = abs(vx_p * (-dy) + vy_p * dx)
        residuals.append(lateral)
        distances.append(math.hypot(vx_p, vy_p))
        if forward > 0.05:
            front_count += 1
    residuals_sorted = sorted(residuals)
    median_residual = residuals_sorted[len(residuals_sorted) // 2] if residuals_sorted else 999.0
    return {
        "count": len(observations),
        "front_count": front_count,
        "max_cross": max_cross,
        "median_residual": median_residual,
        "max_distance": max(distances) if distances else 999.0,
    }




def _pad_obs_distance_label(obs: dict[str, Any] | None) -> str:
    reading = (obs or {}).get("reading", {}) or {}
    return _normalize_vlm_distance(reading.get("target_distance", "unknown"))


def _current_label_from_observations(
    observations: list[dict[str, Any]],
    current_xy: tuple[float, float] | None,
) -> str:
    """현재 위치와 가장 가까운 VLM 관측의 distance label을 반환한다."""
    if current_xy is None or not observations:
        return "unknown"
    nearby: list[tuple[float, dict[str, Any]]] = []
    for obs in observations:
        d = xy_distance(current_xy, obs.get("xy"))
        if d is None:
            continue
        nearby.append((float(d), obs))
    if not nearby:
        return "unknown"
    nearby.sort(key=lambda x: x[0])
    if nearby[0][0] > PAD_SIGN_SINGLE_RAY_MAX_OBS_DIST_M:
        return "unknown"
    return _pad_obs_distance_label(nearby[0][1])


def _validate_sign_xy_distance_consistency(
    pad: str,
    sign_xy: tuple[float, float],
    observations: list[dict[str, Any]],
) -> bool:
    """Triangulated sign_xy가 VLM distance label과 물리적으로 맞는지 검사한다.

    예: VLM이 medium이라고 읽은 관측 pose에서 sign_xy가 0.12m 앞에 있다면,
    sign_xy는 실제 C/D sign이 아니라 ray가 잘못 만난 가짜 교차점일 가능성이 높다.
    이 경우 place 단계로 넘기면 같은 좌표에서 무한 실패하므로 즉시 폐기한다.
    """
    conflicts: list[str] = []
    checked = 0
    for obs in observations:
        label = _pad_obs_distance_label(obs)
        if label == "unknown":
            continue
        dist = xy_distance(obs.get("xy"), sign_xy)
        if dist is None:
            continue
        checked += 1
        lo, hi = PAD_SIGN_DISTANCE_COMPAT_BOUNDS.get(label, PAD_SIGN_DISTANCE_COMPAT_BOUNDS["unknown"])
        if dist < lo:
            conflicts.append(f"{obs.get('label','obs')}:{label}인데 sign_dist={dist:.2f}m < {lo:.2f}m")
        elif dist > hi:
            conflicts.append(f"{obs.get('label','obs')}:{label}인데 sign_dist={dist:.2f}m > {hi:.2f}m")
    if conflicts:
        preview = "; ".join(conflicts[:3])
        if len(conflicts) > 3:
            preview += f"; ...(+{len(conflicts)-3})"
        print(f"[fast-triangulate-guard] {pad}: sign_xy와 VLM 거리 label 충돌 → triangulation 폐기: {preview}")
        return False
    if checked > 0:
        print(f"[fast-triangulate-guard] {pad}: sign_xy 거리-label 검증 통과({checked} obs)")
    return True


def _single_ray_progress_xy_from_recent_observation(
    memory: AgentMemory,
    pad: str,
    current_xy: tuple[float, float] | None,
    observations: list[dict[str, Any]],
    *,
    reason: str,
) -> tuple[float, float] | None:
    """현재 위치 근처에서 target sign을 본 최신/고품질 ray 방향으로 짧게 전진한다.

    이 함수는 pad 좌표를 확정하기 위한 함수가 아니다. triangulation이 현재 위치 근처로
    접혔을 때 place를 막고, 다음 VLM 관측이 더 좋은 baseline을 갖도록 만드는 안전 이동이다.
    """
    if current_xy is None or not observations:
        return None

    candidates: list[tuple[float, float, dict[str, Any]]] = []
    for obs in observations:
        d = xy_distance(current_xy, obs.get("xy"))
        if d is None or d > PAD_SIGN_SINGLE_RAY_MAX_OBS_DIST_M:
            continue
        label = _pad_obs_distance_label(obs)
        # 현재 보이는 target sign의 ray만 사용한다. unknown 거리라도 bearing은 보정 이동에 쓸 수 있다.
        q = float(obs.get("quality", _observation_quality(obs)))
        # 가까운 관측일수록, 품질이 높을수록 우선한다.
        score = q - 2.0 * float(d)
        candidates.append((score, float(d), obs))

    if not candidates:
        print(f"[single-ray-progress] {pad}: 현재 위치 근처 target 관측 없음 → 추가 head/body scan 필요(reason={reason})")
        return None

    candidates.sort(key=lambda x: x[0], reverse=True)
    obs = candidates[0][2]
    label = _pad_obs_distance_label(obs)
    move_m = float(PAD_SIGN_SINGLE_RAY_PROGRESS_M.get(label, PAD_SIGN_SINGLE_RAY_PROGRESS_M["unknown"]))
    bearing = float(obs["bearing_rad"])
    target_xy = (
        float(current_xy[0]) + move_m * math.cos(bearing),
        float(current_xy[1]) + move_m * math.sin(bearing),
    )
    target_xy = _x_fence_clamp_xy(float(target_xy[0]), float(target_xy[1]))
    delta = xy_distance(current_xy, target_xy) or 0.0
    if delta < 0.05:
        return None
    if not _is_safe_destination_candidate(memory, pad, target_xy, source=f"single-ray-progress-{reason}"):
        return None
    print(
        f"[single-ray-progress] {pad}: triangulation={reason} → place 금지, "
        f"recent_obs={obs.get('label')}, label={label}, move={move_m:.2f}m, "
        f"target_xy={target_xy}"
    )
    return target_xy

def _estimate_approach_xy_from_memory(
    memory: AgentMemory,
    pad_name: str,
    current_xy: tuple[float, float] | None,
) -> tuple[float, float] | None:
    """v7 fast 좌표 추정.

    기존 방식의 큰 문제는 C를 보고도 delivery 단계에서 폐루프로 계속 조금씩 전진하는 것이었다.
    이 함수는 서로 다른 pose 2개 이상이 확보되면 바로 sign_xy를 계산하고, 검증을 통과한
    approach_xy를 즉시 Menlo go_to target으로 넘긴다.
    """
    pad = pad_name.upper()
    observations = _geometry_observations_from_memory(memory, pad)
    diverse = _select_diverse_observations(observations)
    if len(diverse) < PAD_FAST_MIN_DIVERSE_OBS:
        print(f"[fast-triangulate] {pad}: pose {len(diverse)}개 < {PAD_FAST_MIN_DIVERSE_OBS}개 → baseline 추가 필요")
        return None

    sign_xy = _triangulate_sign_xy(diverse)
    if sign_xy is None:
        print(f"[fast-triangulate] {pad}: sign_xy 계산 실패 → 관측 추가 필요")
        return None

    metrics = _fast_ray_metrics(sign_xy, diverse)
    if metrics["max_cross"] < PAD_FAST_MIN_CROSS_ANGLE_DEG:
        print(
            f"[fast-triangulate] {pad}: 교차각 부족 {metrics['max_cross']:.1f}° "
            f"< {PAD_FAST_MIN_CROSS_ANGLE_DEG:.1f}° → 관측 pose 추가"
        )
        return None
    if metrics["median_residual"] > PAD_FAST_MAX_MEDIAN_RESIDUAL_M:
        print(
            f"[fast-triangulate] {pad}: ray 잔차 큼 {metrics['median_residual']:.2f}m "
            f"> {PAD_FAST_MAX_MEDIAN_RESIDUAL_M:.2f}m → 관측 추가"
        )
        return None
    if metrics["front_count"] < max(1, len(diverse) - 1):
        print(f"[fast-triangulate] {pad}: 교차점이 ray 뒤쪽 → 폐기")
        return None

    # Final_v4: sign_xy가 현재 위치 근처로 잘못 접히는 경우를 차단한다.
    # VLM distance label과 triangulated sign distance가 충돌하면 place로 넘기지 않고,
    # 최근 sign bearing 방향으로 짧게 이동하여 다음 관측 baseline을 만든다.
    if not _validate_sign_xy_distance_consistency(pad, sign_xy, diverse):
        conflict_key = f"triangulation_conflict_{pad}"
        memory.failed_attempts[conflict_key] = memory.failed_attempts.get(conflict_key, 0) + 1
        print(
            f"[fast-triangulate-guard] {pad}: 현재 관측과 맞지 않는 sign_xy이므로 "
            f"이번 좌표는 place-ready로 사용하지 않음(conflict {memory.failed_attempts[conflict_key]}/{PAD_TRIANGULATION_CONFLICT_RESET_N})"
        )
        if memory.failed_attempts[conflict_key] >= PAD_TRIANGULATION_CONFLICT_RESET_N:
            memory.pad_observations.pop(pad, None)
            memory.known_locations.pop(pad, None)
            memory.pad_direction_hints.pop(pad, None)
            memory.failed_attempts.pop(conflict_key, None)
            print(f"[fast-triangulate-guard] {pad}: 충돌 반복 → 오래된 pad_observations/rough 좌표 초기화")
        return None
    memory.failed_attempts.pop(f"triangulation_conflict_{pad}", None)

    standoff = PAD_FAST_APPROACH_STANDOFF.get(pad, 0.45)
    approach_xy = _approach_xy_from_sign_xy(current_xy, sign_xy, standoff_m=standoff)
    if approach_xy is None:
        return None

    approach_xy = _x_fence_clamp_xy(float(approach_xy[0]), float(approach_xy[1]))
    dist_to_approach = xy_distance(current_xy, approach_xy)
    if dist_to_approach is not None and dist_to_approach > PAD_APPROACH_MAX_DISTANCE_M:
        print(f"[fast-triangulate] {pad}: approach_xy가 너무 멂({dist_to_approach:.2f}m) → 폐기: {approach_xy}")
        return None

    # Final_v4: VLM은 medium/far인데 approach_xy가 현재 위치와 거의 같으면
    # Menlo go_to는 사실상 움직이지 않고 바로 place 실패 루프에 들어간다.
    current_label = _current_label_from_observations(diverse, current_xy)
    min_delta = float(PAD_SIGN_APPROACH_MIN_DELTA_BY_LABEL.get(current_label, PAD_SIGN_APPROACH_MIN_DELTA_BY_LABEL["unknown"]))
    if dist_to_approach is not None and dist_to_approach < min_delta:
        print(
            f"[fast-triangulate-guard] {pad}: approach_delta={dist_to_approach:.2f}m < {min_delta:.2f}m "
            f"(current_label={current_label}) → 현재 위치 근처 좌표로 판단, place 금지"
        )
        print(
            f"[fast-triangulate-guard] {pad}: 현재 위치 근처 좌표는 go_to/place-ready로 쓰지 않고 "
            "baseline 추가 관측으로 전환"
        )
        return None

    if not _is_safe_destination_candidate(memory, pad, approach_xy, source="fast-triangulated-goto-final-v4"):
        return None

    print(
        f"[fast-triangulate] {pad}: pose={len(diverse)}, cross={metrics['max_cross']:.1f}°, "
        f"resid={metrics['median_residual']:.2f}m, sign_xy={sign_xy}, "
        f"standoff={standoff:.2f}m → approach_xy={approach_xy}"
    )
    return approach_xy


async def text_forward_search(
    ctx: Any,
    memory: AgentMemory,
    expected_pad: str,
    *,
    scan_directions: int = SEARCH_SWEEP_DIRECTIONS,
    forward_steps: int = TEXT_SEARCH_FORWARD_STEPS,
) -> dict[str, Any]:
    """v7 빠른 좌표 탐색.

    폐루프 전진을 하지 않는다. target sign을 발견하면 ray를 저장하고, 좌표가 나오면 즉시 반환한다.
    좌표가 아직 안 나오면 true-strafe로 관측 pose만 빠르게 하나 벌린 뒤 재관측한다.
    """
    expected_pad = expected_pad.upper()
    print(f"[fast-search] start target={expected_pad}, version={AGENT_VERSION}")

    vlm_calls = 0
    consecutive_unavailable = 0
    target_visible_count = 0

    async def current_xy() -> tuple[float, float] | None:
        return robot_xy_from_status(await get_robot_status(ctx))

    async def read_limited(head_yaw: float) -> dict[str, Any] | None:
        nonlocal vlm_calls, consecutive_unavailable
        if vlm_calls >= MAX_VLM_CALLS_PER_SEARCH:
            return None
        head_result = await set_head(ctx, yaw=head_yaw, pitch=TEXT_SEARCH_PITCH)
        if isinstance(head_result, dict) and head_result.get("status") == "failed":
            return None
        await asyncio.sleep(0.16)
        reading = await read_signs_with_vlm(ctx, expected_pad)
        vlm_calls += 1
        if reading is None:
            consecutive_unavailable += 1
        else:
            consecutive_unavailable = 0
        return reading

    async def record_reading(reading: dict[str, Any] | None, *, head_yaw: float, label: str) -> tuple[bool, tuple[float, float] | None]:
        nonlocal target_visible_count
        if reading is None:
            return False, None
        print(
            f"[fast-search] head={head_yaw:+.2f}, call={vlm_calls}/{MAX_VLM_CALLS_PER_SEARCH}, "
            f"visible={reading.get('visible_letters')}, target={expected_pad}, "
            f"pos={reading.get('target_position')}, dist={reading.get('target_distance')}, "
            f"offset={reading.get('target_offset_deg')}"
        )
        if not reading.get("target_visible"):
            return False, None
        target_visible_count += 1
        status = await get_robot_status(ctx)
        obs = _make_sign_observation(status, head_yaw_rad=head_yaw, reading=reading, label=label)
        _remember_pad_observation(memory, expected_pad, obs)
        if obs is not None:
            _remember_pad_direction_hint(memory, expected_pad, status, source=label, reading=reading)

        # v8 primary: VLM이 target sign을 확인한 뒤, 같은 색 blob의 bearing/area로
        # 단일 프레임 접근 좌표를 먼저 만든다. 성공하면 true-strafe/교차각 대기 없이 즉시 go_to.
        blob_xy = await _estimate_pad_approach_xy_from_vlm_blob(
            ctx, memory, expected_pad, head_yaw=head_yaw, reading=reading
        )
        if blob_xy is not None:
            return True, blob_xy

        # fallback: blob이 없거나 VLM 위치와 맞지 않으면 기존 multi-view ray triangulation 사용.
        approach_xy = _estimate_approach_xy_from_memory(memory, expected_pad, await current_xy())
        return True, approach_xy

    await _turn_body_toward_direction_hint(ctx, memory, expected_pad)

    for step in range(1, PAD_TRIANGULATION_SEARCH_STEPS + 1):
        print(f"[fast-search] step={step}/{PAD_TRIANGULATION_SEARCH_STEPS}, stored_obs={len(memory.pad_observations.get(expected_pad, []))}")
        found_this_step = False

        head_list = PAD_FAST_HEAD_YAWS if step == 1 else PAD_FAST_RECHECK_HEAD_YAWS
        for head_yaw in head_list:
            if vlm_calls >= MAX_VLM_CALLS_PER_SEARCH:
                break
            reading = await read_limited(head_yaw)
            if reading is None:
                print(f"[fast-search] head={head_yaw:+.2f}, target={expected_pad}, VLM unavailable {consecutive_unavailable}/2")
                if consecutive_unavailable >= 2:
                    await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
                    return {"status": "continue", "target_xy": await current_xy(), "place_ready": False, "reason": "VLM 일시 실패"}
                continue
            found, approach_xy = await record_reading(reading, head_yaw=head_yaw, label=f"v7_s{step}_h{head_yaw:+.2f}")
            found_this_step = found_this_step or found
            if approach_xy is not None:
                await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
                print(f"[fast-search] {expected_pad} 추정 좌표 확정 → 즉시 go_to 대상 approach_xy={approach_xy}")
                return {"status": "found", "target_xy": approach_xy, "place_ready": False, "reason": "v7_fast_triangulated_goto"}

        if vlm_calls >= MAX_VLM_CALLS_PER_SEARCH:
            break

        if found_this_step:
            # 좌표가 아직 안 나오면 관측 pose만 한 번 벌린다. delivery 전진이 아니라 삼각측량 baseline 확보용이다.
            moved = await _move_to_observation_viewpoint(ctx, memory, expected_pad, attempt=step)
            if not moved:
                print(f"[fast-search] observation baseline 확보 실패 → 다음 cycle 재시도")
                break
            # 이동 직후 기존 memory만으로도 좌표가 나오는지 바로 확인한다.
            maybe_xy = _estimate_approach_xy_from_memory(memory, expected_pad, await current_xy())
            if maybe_xy is not None:
                await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
                print(f"[fast-search] {expected_pad} baseline 이동 후 좌표 확정 → approach_xy={maybe_xy}")
                return {"status": "found", "target_xy": maybe_xy, "place_ready": False, "reason": "v7_fast_after_baseline"}
            continue

        # target이 안 보이면 body heading만 바꿔 빠르게 다시 본다. sign을 못 찾는 경우에만 사용한다.
        print(f"[fast-search] {expected_pad} 미발견 → body scan turn")
        await _turn_body_degrees(ctx, FRONT_LOOP_SCAN_TURN_DEG, reason="v7-fast-target-not-visible-scan")

    await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
    final_xy = _estimate_approach_xy_from_memory(memory, expected_pad, await current_xy())
    if final_xy is not None:
        print(f"[fast-search] loop 종료 시점 좌표 확정 → approach_xy={final_xy}")
        return {"status": "found", "target_xy": final_xy, "place_ready": False, "reason": "v7_fast_after_loop"}

    obs_count = len(memory.pad_observations.get(expected_pad, []))
    if target_visible_count > 0 or obs_count > 0:
        return {
            "status": "continue",
            "target_xy": await current_xy(),
            "place_ready": False,
            "reason": f"{expected_pad} 관측 {obs_count}개, 좌표 미확정 → 다음 cycle 빠른 추가 관측",
        }
    return {"status": "failed", "target_xy": None, "place_ready": False, "reason": f"{expected_pad} sign 미발견"}


print(
    f"[x-fence] line_x={X_FENCE_LINE:.2f}, margin={X_FENCE_MARGIN_M:.2f}, "
    f"forbidden_when={X_FENCE_FORBIDDEN_WHEN}, safe_boundary={_x_fence_safe_boundary():.2f}"
)

# ---------------------------------------------------------------------------
# 학생 TODO: execution
# ---------------------------------------------------------------------------

async def execute_decision(
    ctx: Any,
    decision: AgentDecision,
    observation: Observation,
    memory: AgentMemory,
) -> dict[str, Any]:
    """검증된 결정 하나를 Level 1 robot 행동으로 변환합니다."""

    if decision.next_action == "search_cube":
        searched = await observe_after_search(ctx, memory)
        found = select_visual_detection(searched, decision.target_color) is not None
        return {
            "action": "search_cube", "status": "scanned",
            "found": found, "visible_count": len(searched.detections),
        }

    if decision.next_action == "navigate_to_cube":
        # known_locations 우선: A 좌표가 있으면 vision 추정 없이 바로 이동.
        # bootstrap 단계(A 미확인)에서만 vision 추정을 사용한다.
        target_loc = (decision.target_location or "A").removeprefix("pad_").upper()
        if target_loc in memory.known_locations:
            moved = await go_to_known_location(ctx, memory, target_loc)
            return {"action": "navigate_to_cube", **moved, "source": "known_waypoint"}
        target_xy = estimate_target_xy_from_observation(
            observation, decision.target_color, kind="cube"
        )
        if target_xy is not None:
            result = await go_to_xy(ctx, *target_xy)
            return {
                "action": "navigate_to_cube", "status": "arrived",
                "target_location": target_loc, "target_xy": target_xy,
                "result": result_summary(result), "source": "vision_estimate",
            }
        return {
            "action": "navigate_to_cube", "status": "failed",
            "reason": "A 좌표 없고 시야에도 유효 blob 없습니다.",
            "target_location": target_loc,
        }

    if decision.next_action == "search_pad":
        target_color = decision.target_color
        pad_name = (decision.target_location or DESTINATION_SIGN_RULES.get(target_color or "", "")).upper()
        search_result = await text_forward_search(ctx, memory, pad_name)
        status = search_result.get("status")
        if status == "failed":
            return {
                "action": "search_pad", "status": "failed",
                "target_color": target_color,
                "target_location": pad_name,
                "reason": search_result.get("reason") or f"문자 기반 탐색 실패. {pad_name} sign 미발견.",
            }
        if status == "continue":
            return {
                "action": "search_pad", "status": "continue",
                "target_color": target_color,
                "target_location": pad_name,
                "target_xy": search_result.get("target_xy"),
                "reason": search_result.get("reason") or f"{pad_name} sign은 보였지만 아직 near/place-ready가 아님.",
                "source": "vlm_text_sign_continue",
            }
        target_xy = search_result.get("target_xy")
        if target_xy is not None:
            print(f"[fast-goto] {pad_name} 추정 좌표 계산 완료 → 같은 cycle에서 즉시 go_to 실행: {target_xy}")
            goto_result = await go_to_xy(ctx, float(target_xy[0]), float(target_xy[1]))
            goto_summary = result_summary(goto_result)
            if goto_summary.get("error"):
                return {
                    "action": "search_pad", "status": "failed",
                    "target_color": target_color, "target_location": pad_name, "target_xy": target_xy,
                    "result": goto_summary,
                    "reason": f"{pad_name} 추정 좌표 go_to 실패",
                    "source": "v7_fast_triangulated_go_to_immediate",
                }
            return {
                "action": "search_pad", "status": "found_and_arrived",
                "target_color": target_color, "target_location": pad_name, "target_xy": target_xy,
                "result": goto_summary,
                "place_ready": bool(search_result.get("place_ready", False)),
                "source": "v7_fast_triangulated_go_to_immediate",
            }
        return {
            "action": "search_pad", "status": "failed",
            "target_color": target_color, "target_location": pad_name, "target_xy": target_xy,
            "reason": f"{pad_name} target_xy 없음",
            "source": "v7_fast_triangulated_go_to_immediate",
        }

    if decision.next_action == "navigate_to_pad":
        color = decision.target_color
        pad_name = DESTINATION_SIGN_RULES.get(color, "")
        # holding_cube / delivering_just_found 두 경우 모두 이 분기에 진입하므로
        # known_locations에 반드시 존재한다 (decide에서 확인 후 반환)
        pad_xy = memory.known_locations.get(pad_name)
        if pad_xy is not None:
            result = await go_to_xy(ctx, *pad_xy)
            return {
                "action": "navigate_to_pad", "status": "arrived",
                "target_color": color, "target_location": pad_name, "target_xy": pad_xy,
                "result": result_summary(result), "source": "known_location",
            }
        return {
            "action": "navigate_to_pad", "status": "failed",
            "reason": f"{color} 패드 좌표가 없습니다 (비정상 케이스).",
        }

    if decision.next_action == "pick_cube":
        result = await pick_nearest_cube(ctx)
        return {"action": "pick_cube", "result": result_summary(result)}

    if decision.next_action == "place_cube":
        target_loc = (decision.target_location or "A").removeprefix("pad_").upper()
        drop_at_a = target_loc == "A"
        # place/VLM 확인 직전에는 head를 정면으로 복구한다.
        # 탐색 중 head_yaw가 남아 있으면 표지판이 화면 밖으로 나가거나 place 자세가 흔들릴 수 있다.
        await set_head(ctx, yaw=0.0, pitch=TEXT_SEARCH_PITCH)
        await asyncio.sleep(0.2)

        # Final_v5: 첫 방문 rough 좌표에서는 C/D/B/E 글자가 보이는 것만으로 drop하지 않는다.
        # target sign이 near + center일 때만 실제 place를 수행한다. 아니면 sign 방향으로 go_to 재접근한다.
        if not drop_at_a and target_loc not in memory.confirmed_pads:
            preplace = await evaluate_preplace_with_vlm(ctx, memory, target_loc)
            pre_status = preplace.get("status")
            if pre_status == "mismatch":
                return {
                    "action": "place_cube", "status": "vlm_mismatch",
                    "target_location": target_loc,
                    "reason": preplace.get("reason") or f"VLM이 다른 sign을 확인함 — {target_loc} rough 좌표가 틀렸을 가능성, place 중단",
                }
            if pre_status == "unavailable":
                return {
                    "action": "place_cube", "status": "vlm_unavailable",
                    "target_location": target_loc,
                    "reason": preplace.get("reason") or f"{target_loc} place 전 VLM 확인 불가 — drop 금지",
                }
            if pre_status == "align":
                turn_deg = float(preplace.get("turn_deg") or 0.0)
                ok = await _turn_body_degrees(ctx, turn_deg, reason="preplace-align")
                return {
                    "action": "place_cube", "status": "preplace_refine",
                    "target_location": target_loc,
                    "refine_mode": "align",
                    "turn_deg": turn_deg,
                    "result": {"status": "done" if ok else "failed", "error": None if ok else "align_failed"},
                    "reason": preplace.get("reason") or f"{target_loc} near지만 center 아님 → 짧은 body align 후 재확인",
                }
            if pre_status == "approach":
                args = dict(preplace.get("motion_args") or {})
                if not args:
                    return {
                        "action": "place_cube", "status": "vlm_unavailable",
                        "target_location": target_loc,
                        "reason": f"{target_loc} approach args 없음 — drop 금지",
                    }
                motion_result = await move_velocity(ctx, **args)
                motion_summary = result_summary(motion_result)
                if motion_summary.get("error"):
                    return {
                        "action": "place_cube", "status": "vlm_unavailable",
                        "target_location": target_loc,
                        "result": motion_summary,
                        "reason": f"{target_loc} place 전 short approach 실패 — drop 금지",
                    }
                return {
                    "action": "place_cube", "status": "preplace_refine",
                    "target_location": target_loc,
                    "refine_mode": "approach",
                    "motion_args": args,
                    "result": motion_summary,
                    "reason": preplace.get("reason") or f"{target_loc} medium → 짧은 body-frame 접근 후 재확인",
                }
            if pre_status == "reset_search":
                return {
                    "action": "place_cube", "status": "preplace_reset",
                    "target_location": target_loc,
                    "reason": preplace.get("reason") or f"{target_loc} place-ready 아님 → rough 좌표/관측 초기화 후 재탐색",
                }

        delivered_before = await get_delivered_count(ctx)
        result = await place_nearest_zone(ctx)
        return {
            "action": "place_cube",
            "status": "dropped_at_a" if drop_at_a else "placed",
            "target_location": target_loc,
            "delivered_before": delivered_before,
            "result": result_summary(result),
        }

    if decision.next_action == "skip_target":
        # 물리적 행동 없음 — stage 전이(pending_drop_at_a)는 update_memory가 담당한다
        return {"action": "skip_target", "status": "skipped", "target_color": decision.target_color}

    if decision.next_action == "recover":
        await move_velocity(ctx, vx=-0.15, duration_s=0.8)
        return {"action": "recover", "status": "stepped_back"}

    return {"action": decision.next_action, "status": "no_op"}


# ---------------------------------------------------------------------------
# 학생 TODO: verification
# ---------------------------------------------------------------------------

async def verify_outcome(
    ctx: Any, decision: AgentDecision, action_result: dict[str, Any]
) -> dict[str, Any]:
    """행동 결과를 상태값으로 검증한다.

    v10 핵심:
      - destination place 성공은 held=None이 아니라 delivered_count 증가로 판단한다.
      - held=None인데 delivered_count가 증가하지 않은 경우는 "잘못 내려놓음"으로 보고 패드 확정 금지.
      - A에 버리는 drop은 점수 증가가 없으므로 held 여부로만 처리한다.
    """
    if decision.next_action == "place_cube":
        # place_entity 직후 delivered/held 상태 갱신이 약간 늦게 반영될 수 있어 짧게 안정화한다.
        await asyncio.sleep(0.35)
    held = await get_held_cube_info(ctx)
    sdk_result = action_result.get("result") or {}
    sdk_error = sdk_result.get("error") if isinstance(sdk_result, dict) else None
    sdk_status = sdk_result.get("status") if isinstance(sdk_result, dict) else None
    status = action_result.get("status")
    delivered_after = await get_delivered_count(ctx)

    success = status in {
        "arrived", "scanned", "stepped_back", "found", "found_and_arrived", "continue", "placed", "dropped_at_a", "skipped", "preplace_refine", "preplace_reset",
    } or (sdk_error is None and status is None)
    if status == "arrived" and sdk_status not in {None, "done"}:
        success = False
    if status in {"failed", "vlm_mismatch", "vlm_unavailable"} or sdk_error:
        success = False

    if decision.next_action == "pick_cube":
        success = held is not None

    if decision.next_action == "place_cube":
        # Final_v5: preplace_refine은 실제 place가 아니라 안전 재접근이다. 실패/성공 drop 검증을 하지 않는다.
        if status in {"preplace_refine", "preplace_reset"}:
            success = True
        # VLM mismatch/unavailable로 place 자체를 안 한 경우는 실패지만, memory에서 서로 다르게 처리한다.
        elif status in {"vlm_mismatch", "vlm_unavailable"}:
            success = False
        elif status == "dropped_at_a":
            # 포기한 cube를 A에 버리는 recovery 행동은 점수 증가가 없어도 held가 없어지면 성공으로 본다.
            success = held is None and not sdk_error
        else:
            delivered_before = int(action_result.get("delivered_before", delivered_after))
            delivered_delta = delivered_after - delivered_before
            success = delivered_delta > 0 and held is None and not sdk_error
            if not success:
                print(
                    f"[verify] place destination 실패: delivered_before={delivered_before}, "
                    f"delivered_after={delivered_after}, delta={delivered_delta}, held_after={held}"
                )

    return {
        "success": success,
        "failure_reason": action_result.get("reason") or sdk_error,
        "decision": decision.__dict__,
        "action_result": action_result,
        "delivered_count": delivered_after,
        "held_cube": held,
        "held_color": held["color"] if held else None,
    }

# ---------------------------------------------------------------------------
# 학생 TODO: memory update
# ---------------------------------------------------------------------------

def update_memory(
    memory: AgentMemory,
    observation: Observation,
    decision: AgentDecision,
    verified: dict[str, Any],
) -> None:
    """stage 전환 및 좌표 기록.

    좌표 확정 시점 — "행동으로 증명된 위치만 기억한다":
      - A: 첫 pick 성공 시점의 로봇 좌표
      - 패드: 첫 place 성공 시점의 로봇 좌표 (탐색 시의 blob 추정치를 덮어씀)
    가까운 2색 선택 전략은 사용하지 않는다. 좌표가 저장된 패드는 색상에
    관계없이 모두 운반 대상이며, search_pad 성공 좌표는 place 성공 시점의
    로봇 좌표로 덮어써서 확정한다.
    """
    if "delivered_count" in verified:
        memory.delivered_count = int(verified["delivered_count"])
    memory.held_color = verified.get("held_color")

    action = decision.next_action
    success = verified.get("success", False)
    action_result = verified.get("action_result", {})

    # Bootstrap: A 도달 성공 — 좌표 저장은 하지 않는다 (첫 pick 성공 시 기록)
    if action == "navigate_to_cube" and not memory.bootstrapped_to_a and success:
        memory.bootstrapped_to_a = True
        memory.stage = "at_source"

    # pick 성공: 첫 pick 시 A 좌표 기록, 실패 카운터 초기화
    elif action == "pick_cube" and success:
        memory.stage = "holding_cube"
        memory.failed_attempts.pop("pick", None)
        if "A" not in memory.known_locations:
            a_xy = robot_xy_from_status(observation.robot_status)
            if a_xy is not None:
                memory.known_locations["A"] = a_xy
                print(f"[memory] A 좌표 최초 기록: {a_xy}")

    # pick 실패: 횟수 누적, 한계 초과 시 bootstrap 재시작
    elif action == "pick_cube" and not success:
        count = memory.failed_attempts.get("pick", 0) + 1
        memory.failed_attempts["pick"] = count
        print(f"[recovery] pick 실패 {count}/{MAX_PICK_FAILURES}회")
        if count >= MAX_PICK_FAILURES:
            memory.bootstrapped_to_a = False
            memory.stage = "need_cube"
            memory.failed_attempts.pop("pick", None)
            print("[recovery] 잘못된 위치 → bootstrap 재시작")

    # 패드 탐색 성공: 추정 좌표 저장(첫 접근용) + delivering_just_found로 전환.
    # 이 좌표는 place 성공 시 로봇 좌표로 덮어써진다.
    elif action == "search_pad" and action_result.get("status") in {"found", "found_and_arrived"}:
        target_color = decision.target_color
        pad_name = DESTINATION_SIGN_RULES.get(target_color or "", "")
        target_xy = action_result.get("target_xy")
        if pad_name and target_xy:
            memory.known_locations[pad_name] = (float(target_xy[0]), float(target_xy[1]))
            if action_result.get("status") == "found_and_arrived":
                memory.stage = "at_pad"
                memory.active_pad_name = pad_name
                print(f"[memory] 패드 {pad_name}({target_color}) 추정 좌표 기록 후 즉시 go_to 완료: {target_xy} → 다음 cycle place")
            else:
                memory.stage = "delivering_just_found"
                print(f"[memory] 패드 {pad_name}({target_color}) 삼각측량 approach 좌표 기록: {target_xy} → go_to 배달")

    # 패드 탐색 진행 중: target은 보였지만 center/near가 아니거나 VLM 한계가 왔다.
    # 실패가 아니므로 search 실패 카운트를 올리지 않고 holding_cube로 되돌려 다음 cycle에서 이어서 접근한다.
    elif action == "search_pad" and action_result.get("status") == "continue":
        pad_name = DESTINATION_SIGN_RULES.get(decision.target_color or "", "")
        memory.stage = "holding_cube"
        print(
            f"[front-loop] 패드 {pad_name} 탐색 계속: {action_result.get('reason')} "
            "→ 실패 카운트 증가 없이 다음 cycle에서 이어서 search_pad"
        )

    # 패드 탐색 실패: 실패 횟수 누적 후 LLM 상담 stage로 (재탐색 vs 포기)
    elif action == "search_pad" and action_result.get("status") == "failed":
        pad_name = DESTINATION_SIGN_RULES.get(decision.target_color or "", "")
        if pad_name:
            key = f"search_{pad_name}"
            memory.failed_attempts[key] = memory.failed_attempts.get(key, 0) + 1
        memory.stage = "search_pad_failed"
        print(f"[recovery] 패드 탐색 실패 (누적 {memory.failed_attempts.get(f'search_{pad_name}', 0)}회) → 재탐색/포기 판단")

    # skip: 이 큐브는 포기 — A로 돌아가 drop
    elif action == "skip_target" and success:
        color = decision.target_color
        if color and color not in memory.skipped_colors:
            memory.skipped_colors.append(color)
        memory.stage = "pending_drop_at_a"
        print(f"[recovery] {color} 포기 → A 복귀 후 drop")

    # 패드로 이동 성공 (holding_cube / delivering_just_found 두 경로 공통)
    elif action == "navigate_to_pad" and success:
        memory.stage = "at_pad"
        memory.active_pad_name = decision.target_location

    # Final_v5: place 전 안전 게이트가 아직 near+center가 아니라고 판단해 재접근한 경우.
    # 실제 drop은 하지 않았으므로 place 실패로 세지 않고, 다음 cycle에서 다시 preplace check를 수행한다.
    elif action == "place_cube" and action_result.get("status") == "preplace_refine":
        pad_name = action_result.get("target_location") or (decision.target_location or "").removeprefix("pad_").upper()
        mode = str(action_result.get("refine_mode") or "unknown")
        key = f"preplace_refine_{pad_name}"
        memory.failed_attempts[key] = memory.failed_attempts.get(key, 0) + 1
        if mode == "approach":
            fwd_key = f"preplace_forward_{pad_name}"
            memory.failed_attempts[fwd_key] = memory.failed_attempts.get(fwd_key, 0) + 1
        if memory.failed_attempts[key] >= PAD_PLACE_REFINE_MAX_PER_PAD:
            # 같은 rough 좌표 주변에서 center+near가 안 나오면 더 끌지 않고 오래된 관측까지 초기화한다.
            memory.known_locations.pop(pad_name, None)
            memory.pad_observations.pop(pad_name, None)
            memory.pad_direction_hints.pop(pad_name, None)
            memory.stage = "holding_cube"
            memory.active_pad_name = None
            memory.failed_attempts.pop(key, None)
            memory.failed_attempts.pop(f"preplace_forward_{pad_name}", None)
            print(f"[preplace-refine] {pad_name}: refine {PAD_PLACE_REFINE_MAX_PER_PAD}회 도달 → rough 좌표/관측 초기화 후 재탐색")
        else:
            memory.stage = "at_pad"
            memory.active_pad_name = pad_name
            print(f"[preplace-refine] {pad_name}: drop 없이 {mode} 완료({memory.failed_attempts[key]}/{PAD_PLACE_REFINE_MAX_PER_PAD}) → 다음 cycle place 조건 재확인")

    elif action == "place_cube" and action_result.get("status") == "preplace_reset":
        pad_name = action_result.get("target_location") or (decision.target_location or "").removeprefix("pad_").upper()
        if pad_name:
            memory.known_locations.pop(pad_name, None)
            memory.pad_observations.pop(pad_name, None)
            memory.pad_direction_hints.pop(pad_name, None)
            memory.failed_attempts.pop(f"preplace_refine_{pad_name}", None)
            memory.failed_attempts.pop(f"preplace_forward_{pad_name}", None)
            memory.failed_attempts.pop(f"triangulation_conflict_{pad_name}", None)
        memory.stage = "holding_cube"
        memory.active_pad_name = None
        print(f"[preplace-reset] {pad_name}: {action_result.get('reason')} → drop 없이 관측 초기화 후 재탐색")

    # VLM 확인 불가: Timeout/Cancelled/빈 화면은 "틀린 패드"가 아니므로 좌표를 폐기하지 않는다.
    # 같은 위치에서 짧게 재확인할 수 있도록 at_pad를 유지하고, 반복되면 문자 재탐색으로 돌린다.
    elif action == "place_cube" and action_result.get("status") == "vlm_unavailable":
        pad_name = action_result.get("target_location") or ""
        key = f"vlm_unavailable_{pad_name}"
        memory.failed_attempts[key] = memory.failed_attempts.get(key, 0) + 1
        if memory.failed_attempts[key] >= MAX_VLM_UNAVAILABLE_RETRIES:
            memory.stage = "holding_cube"
            memory.failed_attempts.pop(key, None)
            # 좌표는 유지한다. 다음 decide에서 같은 좌표로 재접근하거나, mismatch가 나면 폐기한다.
            print(f"[recovery] {pad_name} VLM 확인 불가 반복 → 좌표 유지, holding_cube에서 재판단")
        else:
            memory.stage = "at_pad"
            print(f"[recovery] {pad_name} VLM 확인 불가 → 좌표 폐기하지 않고 다음 cycle place 재확인")

    # 첫 방문 VLM 표지판 실제 불일치: A 또는 다른 sign이 보였으므로 rough 좌표가 틀렸을 가능성이 높다.
    # 좌표를 폐기하고 재탐색한다. 반복되면 이 색은 포기한다 (연속성 보호).
    elif action == "place_cube" and action_result.get("status") == "vlm_mismatch":
        pad_name = action_result.get("target_location") or ""
        memory.known_locations.pop(pad_name, None)
        memory.pad_observations.pop(pad_name, None)
        memory.failed_attempts.pop(f"vlm_unavailable_{pad_name}", None)
        key = f"vlm_{pad_name}"
        memory.failed_attempts[key] = memory.failed_attempts.get(key, 0) + 1
        if memory.failed_attempts[key] >= MAX_VLM_MISMATCHES:
            memory.stage = "pending_drop_at_a"
            print(f"[recovery] {pad_name} VLM 실제 불일치 {memory.failed_attempts[key]}회 → 포기, A drop")
        else:
            memory.stage = "holding_cube"
            print(f"[recovery] {pad_name} VLM 실제 불일치 → rough 좌표 폐기, 문자 재탐색")
        memory.active_pad_name = None

    # place 성공: 패드 좌표를 로봇 좌표로 확정한다
    elif action == "place_cube" and success:
        if action_result.get("status") == "dropped_at_a":
            memory.stage = "at_source"
        else:
            pad_name = (decision.target_location or "").removeprefix("pad_").upper()
            # 이 사이클 시작 시점의 robot pose = go_to가 끝난 패드 앞 위치.
            # blob 면적 기반 추정치보다 정밀하므로 waypoint를 이 값으로 덮어쓴다.
            pad_xy = robot_xy_from_status(observation.robot_status)
            if pad_name and pad_xy is not None:
                memory.known_locations[pad_name] = pad_xy
            if pad_name and pad_name not in memory.confirmed_pads:
                memory.confirmed_pads.append(pad_name)
                print(f"[memory] 패드 {pad_name} 확정: {pad_xy} (place 성공 시점 로봇 좌표)")
            memory.failed_attempts.pop(f"search_{pad_name}", None)
            memory.failed_attempts.pop(f"vlm_{pad_name}", None)
            memory.failed_attempts.pop(f"vlm_unavailable_{pad_name}", None)
            memory.stage = "returning_to_source"
        memory.active_pad_name = None
        memory.failed_attempts.pop("place", None)

    # place 실패: destination place는 delivered_count 증가로 검증한다.
    # held가 사라졌지만 delivered가 증가하지 않은 경우는 잘못 내려놓은 것이므로 패드는 절대 확정하지 않는다.
    elif action == "place_cube" and not success:
        bad_pad = memory.active_pad_name or (decision.target_location or "").removeprefix("pad_").upper()
        if bad_pad in memory.known_locations and bad_pad not in memory.confirmed_pads:
            memory.known_locations.pop(bad_pad, None)
            # v8: place 실패는 좌표가 완전히 틀린 경우보다 아직 멀어서 실패한 경우가 많다.
            # 관측 ray/blob 힌트는 유지해서 다음 cycle에서 다시 처음부터 5회 스캔+baseline을 반복하지 않게 한다.
            print(f"[recovery] place 실패 → 미확정 {bad_pad} rough 좌표만 폐기, 관측은 유지")
        count = memory.failed_attempts.get("place", 0) + 1
        memory.failed_attempts["place"] = count
        print(f"[recovery] place 실패 {count}/{MAX_PLACE_FAILURES}회")
        if memory.held_color is None:
            # 이미 cube가 손에서 사라진 경우 재시도할 대상이 없으므로 A로 복귀해 다음 cube를 집는다.
            memory.stage = "returning_to_source"
            memory.active_pad_name = None
            memory.failed_attempts.pop("place", None)
            print("[recovery] cube는 손에서 사라졌지만 점수 증가 없음 → 패드 확정 금지, A 복귀")
        elif count >= MAX_PLACE_FAILURES:
            memory.failed_attempts.pop("place", None)
            memory.stage = "holding_cube"
            memory.active_pad_name = None
            print("[recovery] place 연속 실패 → 문자 기반 재탐색")
        else:
            memory.stage = "holding_cube"
            memory.active_pad_name = None
            print("[recovery] place 실패 → rough 좌표 폐기 후 문자 기반 재탐색")

    # A 복귀 성공: held_color 유무로 분기
    elif action == "navigate_to_cube" and memory.bootstrapped_to_a and success:
        if memory.held_color is not None:
            # 패드 탐색 포기 후 A 복귀: 큐브 아직 보유 → drop 대기
            memory.stage = "drop_at_a_ready"
        else:
            memory.stage = "at_source"

    # recover 처리
    elif action == "recover":
        if memory.stage == "at_source":
            # 정상 경로: pick 연속 실패 → bootstrap 재시작
            memory.bootstrapped_to_a = False
            memory.stage = "need_cube"
        else:
            # 비정상 케이스 — held_color 유무로 복구 방향 결정
            old_stage = memory.stage
            if memory.held_color is not None:
                memory.stage = "holding_cube"
            else:
                memory.stage = "at_source"
            print(f"[recovery] 비정상 recover from {old_stage} → {memory.stage}")

    memory.logs.append({
        "stage": memory.stage,
        "held_color": memory.held_color,
        "delivered_count": memory.delivered_count,
        "decision": decision.next_action,
        "success": success,
        "known_locations": list(memory.known_locations.keys()),
        "confirmed_pads": list(memory.confirmed_pads),
        "pick_failures": memory.failed_attempts.get("pick", 0),
        "llm_consults": memory.llm_consult_count,
    })


# ---------------------------------------------------------------------------
# 실행 루프
# ---------------------------------------------------------------------------

async def run_task01_to_a(ctx: Any, *, max_cycles: int = 12) -> AgentMemory:
    """Task01 전용: A 도달(bootstrapped_to_a)까지만 실행하고 멈춘다. pick은 하지 않는다."""
    memory = AgentMemory()
    memory.known_locations.update(load_configured_waypoints())
    last_result: dict[str, Any] | None = None
    for cycle in range(1, max_cycles + 1):
        print(f"\n[Task01] Cycle {cycle} | stage={memory.stage}")
        observation = await observe_world(ctx, memory)
        decision = await decide_next_action(TASK, observation, memory, last_result)
        print("Decision:", decision)
        if decision.next_action == "stop":
            break
        action_result = await execute_decision(ctx, decision, observation, memory)
        verified = await verify_outcome(ctx, decision, action_result)
        update_memory(memory, observation, decision, verified)
        last_result = verified
        if memory.bootstrapped_to_a:
            print("[Task01] A 도달 — 여기서 중단 (pick부터는 scored run에서)")
            break
    return memory



async def _safe_stop_reason_from_tracker(tracker: CompletionTracker, ctx: Any, *, when: str) -> Any:
    """CompletionTracker 내부 robot_status RPC 취소가 전체 run을 죽이지 않게 감싼다."""
    try:
        return await tracker.stop_reason_from_scene(ctx)
    except asyncio.CancelledError as exc:
        print(f"[completion] {when} stop_reason 확인 취소({exc!r}) → 이번 cycle에서는 계속 진행")
        return None
    except Exception as exc:
        print(f"[completion] {when} stop_reason 확인 실패({exc!r}) → 이번 cycle에서는 계속 진행")
        return None


async def run_agent(
    ctx: Any,
    *,
    max_cycles: int = 20,
    completion: CompletionConfig | None = None,
) -> AgentMemory:
    """얇은 observe-decide-act-verify loop입니다. 이 loop가 아니라 TODO 함수들이 전략을 담습니다."""
    memory = AgentMemory()
    memory.known_locations.update(load_configured_waypoints())
    print(f"[agent-version] {AGENT_VERSION}")
    print(f"[시작] configured waypoints: {memory.known_locations}")
    last_result: dict[str, Any] | None = None
    tracker = CompletionTracker(completion) if completion is not None else None

    for cycle in range(1, max_cycles + 1):
        print(
            f"\n[Level 1] Cycle {cycle} | stage={memory.stage} | "
            f"held={memory.held_color} | delivered={memory.delivered_count}"
        )
        if tracker is not None:
            first_cycle = tracker.started_at is None
            tracker.start_first_cycle()
            if first_cycle:
                tracker.print_start()
            reason = await _safe_stop_reason_from_tracker(tracker, ctx, when="before")
            if reason is not None:
                tracker.mark_ended(reason)
                print(f"Completion target reached before cycle action: {reason}.")
                break

        observation = await observe_world(ctx, memory)
        decision = await decide_next_action(TASK, observation, memory, last_result)
        print("Decision:", decision)

        if decision.next_action == "stop":
            break

        action_result = await execute_decision(ctx, decision, observation, memory)
        verified = await verify_outcome(ctx, decision, action_result)
        update_memory(memory, observation, decision, verified)
        last_result = verified
        if tracker is not None:
            reason = await _safe_stop_reason_from_tracker(tracker, ctx, when="after")
            if reason is not None:
                tracker.mark_ended(reason)
                print(f"Completion target reached after cycle action: {reason}.")
                break

    if tracker is not None:
        try:
            await tracker.print_summary_from_scene(ctx)
        except asyncio.CancelledError as exc:
            print(f"[completion] summary 출력 취소({exc!r}) → memory 반환")
        except Exception as exc:
            print(f"[completion] summary 출력 실패({exc!r}) → memory 반환")
    return memory


async def run(ctx: Any) -> None:
    print(TASK)
    print("Level 1 all-color adaptive-navigation project 실행")
    memory = await run_agent(
        ctx,
        max_cycles=10_000,
        completion=CompletionConfig(level=1, max_elapsed_s=600),
    )
    print("\n실행 완료.")
    print(f"Delivered count: {memory.delivered_count}")
    print(f"LLM consults: {memory.llm_consult_count}")
    print("Logs:")
    for item in memory.logs:
        print(item)


In [ ]:
# Colab/로컬 공통 context setup입니다.
# Colab에서는 왼쪽 열쇠 아이콘(Secrets)에 MENLO_API_KEY, TOKAMAK_API_KEY를 추가하세요.
# 로컬에서는 real/.env 또는 이미 설정된 환경변수를 그대로 사용합니다.
import os

try:
    from google.colab import userdata

    for key in ("MENLO_API_KEY", "TOKAMAK_API_KEY"):
        if not os.environ.get(key):
            value = userdata.get(key)
            if value:
                os.environ[key] = value
except Exception:
    pass

# 주의: 원래 v2 셀의 open_context(cfg, "project")는 menlo_runner에 존재하지 않는 API다.
# 공식 스타터와 동일하게 open_robot_context를 사용한다.
# require_tokamak=True: LLM 상담과 VLM 표지판 확인에 TOKAMAK_API_KEY가 필요하다.
from menlo_runner.config import load_config
from menlo_runner.context import open_robot_context

config = load_config(require_tokamak=True)
ctx = await open_robot_context(config, name_prefix="level-1-all-colors")
print("Viewer:", ctx.viewer_url)


## Task01 — A 이동 단독 검증

10분 scored run 전에 bootstrap(임의 위치 → A 도달)만 짧게 검증합니다. A에 도달하면 pick 없이 즉시 멈춥니다.


In [ ]:
# Task01: A/source 이동 단독 검증 — bootstrap이 끝나면(pick 전에) 자동 중단된다.
memory = await run_task01_to_a(ctx)
print("bootstrapped_to_a:", memory.bootstrapped_to_a)
print("stage:", memory.stage)
print("known_locations:", memory.known_locations)


## Agent 실행

현재 실험에 필요한 TODO를 충분히 구현한 뒤 실행하세요. 이 cell은 기본적으로 10분 scored simulation을 실행합니다.


In [ ]:
memory = await run_agent(
    ctx,
    max_cycles=10_000,
    completion=CompletionConfig(level=1, max_elapsed_s=600),
)
memory


In [ ]:
# 종료할 때 cleanup하세요.
# await ctx.close()
